# EDA — Households, CIFs & Accounts

**Goal.** Describe the whole history of the bank and where it stands today at the three levels where a relationship lives — the **household**, the **customer (CIF)** and the **account** — and the **roles** that tie them together. Who the customers are today, how relationships are built (owners, joint owners, signers, guarantors, the people behind each business), which products they hold, how the base was built year by year, and who left. Every split — **Individual vs. Business** (`IorB`), **Domestic vs. Foreign** (`ForD`), **Select vs. Non-Select** — is run alone and in every combination, and at the level where it makes sense.

**Grain of the file.** One row per **CIF × account × role**. A household groups CIFs; a CIF can be linked to many accounts; an account can carry many CIFs, each with a role. Balances live on the **account**, so they are counted **once, on the primary owner** — never summed over rows (summing rows counts a joint account twice and a business account once per signer).

**Scope.** The file is historical: every DD, CD and LN account ever opened, open or closed. "Today" = open accounts at the snapshot (§6–§11, §14–§15). The history views (§12) and the attrition analysis (§13) use everything.

**The data model**

```
 Household (HouseholdID) ──1:N──  CIF (Customer CIF)  ──N:M──  Account (Product Domain + Account Number)
   composition, size,               IorB, ForD, age,          │      DD / CD / LN, product, Select product,
   relationship start               segment, country          │      branch, officer, status, dates, balance
                                                              │
                                     Role (Relationship Role Code / Description, Ownership Category)
                                     one Primary Owner per account · joint owners, co-borrowers, guarantors,
                                     authorized signers, beneficial owners, attorneys, trustees, beneficiaries…
```

**How to read this notebook**

| § | Question it answers |
|---|---|
| 1–2 | Setup, configuration, load, **mixed-type diagnosis**, explicit column classification |
| 3 | **Data model** — is the file what we think it is? Row grain, CIF → household, account key, **exactly one primary owner per account** |
| 4 | **Data quality** — nulls measured at the right grain, statuses vs. dates, birth dates, segment codes, country, balances, IDs, outliers |
| 5 | **Features** — the four analysis tables (relationship rows, accounts, CIFs, households) and every rule stated once |
| 6 | **The bank today on one page** — the three levels side by side, the granularity funnel, the 8-cell grid IorB × ForD × Select |
| 7 | **Households** — real groups vs. single-CIF fallbacks, size, composition, depth, concentration, the largest relationships |
| 8 | **Customers (CIF)** — every granularity (IorB, ForD, Select and all combinations) × every feature, age, geography |
| 9 | **Roles** — who plays which role on whose account, ownership structures, connected persons, links across households, relationship networks |
| 10 | **The person behind the business** — key-person age for business CIFs, decision-maker age for every customer, household reference age |
| 11 | **Accounts & products** — product catalogue, mix per CIF and per household, Select products, balances, account life, branches, officers, large relationships |
| 12 | **History** — the bank year by year: stock of accounts / customers / households, openings and closings, annual attrition rates, vintages, survival |
| 13 | **Attrition** — the customers and households that left: who, when, how long they stayed, what they had, CIF churn vs. household churn, partial attrition |
| 14 | **Relationships & tests** — correlations, Select uplift with confidence intervals, effect sizes, association between dimensions |
| 15 | **Segment profiles** — final CIF and household fingerprints |
| 16 | **Key findings & exports** — auto-generated numbers and the clean tables for Power BI |
| 17 | **Open questions & data to request** |

**Color legend (brand palette).** <span style="color:#003C71">■</span> Blue `#003C71` = Individual · Domestic · DD · first level of a split &nbsp; <span style="color:#007953">■</span> Green `#007953` = Business · Foreign · LN · Select &nbsp; <span style="color:#5B8BC5">■</span><span style="color:#4FB08E">■</span> lighter tints = second level of a 2-way split &nbsp; <span style="color:#8B95A5">■</span> Gray = other / unclassified / by design &nbsp; <span style="color:#E1261C">■</span> Red `#E1261C` = **reserved** for alerts, negatives and the lost base.

**Conventions.** Medians describe the typical customer (balances are heavy-tailed, so averages mislead). Balance distributions and concentration are shown **among holders** (balance > 0) with the share of holders annotated. Median bars carry **95% bootstrap confidence intervals**; comparisons come with an effect size (Cliff's δ), not just a p-value. Every chart has a table next to it with the exact numbers. Nulls are always measured **at the grain of the column** (a CIF attribute over CIFs, an account attribute over accounts), never over rows.

## Data dictionary

Each column is used at the grain it belongs to. "Name here" is the short name the notebook gives it after loading (the mapping is in the configuration cell, so a renamed column in the extract is fixed in one place).

| Column in the file | Name here | Grain | Role | Rules and notes |
|---|---|---|---|---|
| Household ID (Household Group ID) | `HouseholdID` | Household | ID | Macro-group / household key. **Falls back to the Customer CIF** when the customer has no group, or sits in a temporary or employee group → a one-CIF "household". §3.2 separates **real households** from these fallbacks. |
| Household Group Name | — | Household | Excluded (PII) | Commercial or family group name; falls back to the customer name when the ID falls back to the CIF. Used only to confirm the fallback, then dropped. |
| Household Relationship Start Date | `HHStartFile` | Household | Date | Earliest start across member CIFs: MIN(CIF creation date), falling back to MIN(account opened). Recomputed in §5 and compared. |
| Customer CIF | `CIF` | CIF | ID | Universal customer key. Kept as text. |
| Customer Name | — | CIF | Excluded (PII) | Primary legal name. Dropped after load. |
| Person Type (Individual\|Business) | `IorB` | CIF | Dimension (2) | `Individual` when the core individual flag is Y, else `Business`. Primary split. |
| Residence Origin (Domestic\|Foreign) | `ForD` | CIF | Dimension (2) | `Foreign / International` when the tax-status code marks a non-resident / foreign entity **or** the alien-customer flag is Y, else `Domestic`. Normalized to Domestic / Foreign. |
| CIF Creation Date | `CIFCreated` | CIF | Date | Official onboarding date of the CIF. |
| Birth Date (YYYY-MM-DD) | `BirthDate` | CIF | Date | Parsed from the core Julian format (YYYYDDD). **Individuals: date of birth → `Age`. Business: incorporation date → `BusinessAge`.** Arrives with mixed Python types — diagnosed and parsed in §2.1. |
| Customer Segment Code | `Segment` | CIF | Dimension (code) | `[I\|D\|P][P\|B][tier]` (e.g. IB1, IB4, IP3, DB5) or `Unassigned`. Decoded and checked against IorB / ForD in §4.4. |
| Customer Segment Description | `SegmentDesc` | CIF | Dimension | Human-readable segment (e.g. *Intl Business Select*, *Intl Select*, *Domestic Business*). Says whether the **segment** is a Select segment → `SelectSegment`. |
| Relationship Role Code | `RoleCode` | CIF × account | Dimension | Role of the CIF on the account (`1` = primary, `2` = joint, `3` = signer, letters for guarantor and others). |
| Relationship Role Description | `RoleDesc` → `OwnerRow` | CIF × account | **Critical filter** | Text of the role. **When it says *Primary*, that CIF is the owner of the account.** Balances and account counts are attributed only to that owner — otherwise every joint owner and signer counts the balance again. The other roles (Joint Owner, Authorized Signer, Guarantor…) are grouped into **role families** in §5. |
| Ownership Category | — | CIF × account | Cross-check | `Primary Owner` / `Co-Owner / Authorized / Other`. Used only where the role description is blank, and checked against it in §2.1. |
| Account Number | `AccountNumber` | Account | ID | Unique contract number across DD, CD and LN — combined with the domain into `AccountKey` because numbering is per system (checked in §3.3). |
| Account Name | — | Account | Excluded (PII) | Short title of the account. |
| Product Domain | `Domain` | Account | Dimension (3) | `DD` Demand Deposit · `CD` Certificate of Deposit · `LN` Loan Account. |
| SCC / Product Subtype | `Subtype` | Account | Dimension (high card.) | Service-charge code (DD) or type (CD, LN): fee schedules and product rules. |
| Product Description | `Product` | Account | Dimension (high card.) | Commercial product name from the product parameter catalogs. |
| AccountSegment | `AccountSelect` | Account | Flag | **Strict account-level Select flag:** Select when the product description contains *SELECT*, else Regular. |
| Large Relationship | `LargeRel` | detected | Flag (Y/N) | Not in the dictionary — its grain (CIF, account or household) is detected in §3.5. |
| Branch Code / Branch Name | `BranchCode`, `Branch` | Account | Dimension | Branch where the account was opened or is assigned. |
| Relationship Officer Code / Name | `OfficerCode`, `OfficerInactive` | Account | Dimension | Officer responsible for the account. The name is dropped (PII); only a flag for names marked *RETIRED* (or similar) is kept. |
| Account Status Code / Description | `StatusCode`, `StatusDesc` | Account | Dimension | Core status (Active, Closed / Inactive, Dormant / Restricted, Charged-Off / Loss…). Mapped to four **status groups** in §4.2. |
| Account Date Opened | `Opened` | Account | Date | Date opened / issued / original loan date. |
| Account Date Closed | `Closed` | Account | Date | Closure, maturity redemption or loan payoff date. **Blank = open.** |
| Current Balance ($) | `Balance` | Account | Measure ($) | Ledger balance: available balance for DD, principal for CD, outstanding debt for LN. A single snapshot — **there is no balance history**, so every year-by-year view counts accounts and customers, not money. |
| Country | `Country` | CIF | Dimension (high card.) | Country of the customer. `MISSING DATA` and similar placeholders → missing. Mapped to `Region`. |

## Definitions — stated once, used everywhere

| Term | Rule |
|---|---|
| **Owner** | The CIF whose *Relationship Role Description* says **Primary** on that account. Exactly one per account; §3.4 repairs the exceptions (no primary row → highest-priority co-owner; two primary rows → one kept, the other treated as joint owner). |
| **Open account** | Status group *Active* or *Dormant / restricted* (`OPEN_GROUPS`). *Closed* and *Charged-off* are closed. The close date is used for timing, the status for "open today". |
| **Active customer** (CIF) | Owns at least one open account. **This is the customer base.** |
| **Connected CIF** | Owns no open account but holds a non-owner role (joint owner, signer, guarantor…) on an open account. Part of a relationship, not a customer by balance — and a cross-sell target. |
| **Former customer** | Owned at least one account in the past and has nothing open in any role today. **The lost base.** |
| **Former connected** | Never owned an account and holds nothing open (e.g. a signer of a business that left). |
| **Active household** | At least one member is an active customer. |
| **Real household vs. single CIF** | A real household carries its own group ID; a *single CIF* household is the fallback where the household ID is the customer's own CIF. |
| **Deposits / Loans** | Sum of balances of **open accounts the CIF owns**: DD + CD / LN. `TotalBalance` = deposits + loans (relationship footings). Household = sum over its members. |
| **Select** | Account: product description contains *SELECT*. CIF: `SelectProduct` = owns an open Select account (former customers: ever owned one); `SelectSegment` = segment description says Select. `SELECT_BASIS` picks which one drives the `Select` split; both are kept and cross-checked. Household: any member is Select. |
| **Role family** | Role descriptions grouped by meaning: Primary owner, Joint / co-owner, Co-borrower, Guarantor, Beneficial owner / controller, Authorized signer, Power of attorney / agent, Fiduciary / custodial, Beneficiary, Mailing / tax / interest, Other. |
| **Relationship start / end** | Start = earlier of CIF creation and the first account the CIF was ever linked to. End = the last of its accounts to close (former CIFs only; active ones are *censored* at the snapshot). Lifetime = start → end. |
| **Key person** (business) | The individual linked to the business's own accounts by the highest-priority role (beneficial owner → guarantor → joint owner → co-borrower → signer → attorney…), then by number of accounts, then the eldest; falls back to individuals in the same real household. **Key-person age = the age behind the business.** |
| **Decision-maker age** | Individuals: their own age. Business: key-person age. Lets every customer be read on one age axis. |

## Executive summary — fill in after the run

*Numbers come from §16 (auto-generated key findings) and the observation cells at the end of each section. Re-run the notebook and refresh them whenever the extract changes.*

**The bank in three levels.** … active households (… real groups, … single-CIF), … active customers (… individuals / … businesses) plus … connected CIFs, … open accounts (DD … · CD … · LN …), $… of deposits and $… of loans.

**How relationships are built.** … % of open accounts have a single owner and nobody else; … % carry joint owners, … % signers, … % of loan balances have a guarantor. … % of business accounts name an individual behind them; median key-person age …, vs. … for individual customers. … key persons do not bank with us personally.

**Households vs. customers.** A real household holds a median … CIFs and … product families vs. … for a single customer; … % of customers that look single-product belong to a multi-product household. The top 1% of households hold … % of deposits.

**History.** The base peaked at … customers in …; … accounts were opened and … closed since …. Annual customer attrition … % (…), household attrition … %. Of the customers acquired in …, … % are still here after 5 years.

**Who left.** … former customers (… % of everyone ever onboarded); they stayed a median … years, … % left within a year, … % were single-account relationships, … % had only a loan (run-off) and … % only a CD (maturity). … % of the customers lost inside a real household left a household that is still active.

**Risks and blind spots.** …

# 1. Setup

Imports, the brand palette, and the plotting / statistics helpers used by every section. Nothing here depends on the data.

In [ ]:
import os, re, math, warnings, datetime as dt
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# ---------------- Brand palette ----------------
BLUE, GREEN, RED = '#003C71', '#007953', '#E1261C'
BLUE_T, GREEN_T  = '#5B8BC5', '#4FB08E'            # tints = second level of a 2-way split
BLUE_M           = '#2E5F94'                       # mid blue, third level of a blue family
RED_T            = '#F08A84'                       # tint of red, second level of the "lost" family
GRAY, GRAY_L     = '#8B95A5', '#C9CED6'            # neutral: "Other / unclassified / by design"
INK = dict(primary='#1F2937', secondary='#4B5563', muted='#8B95A5', grid='#E6E8EC', surface='#FFFFFF', card='#F5F7FA')
CMAP_BLUE  = LinearSegmentedColormap.from_list('brand_blue',  ['#F3F6FA', BLUE])
CMAP_GREEN = LinearSegmentedColormap.from_list('brand_green', ['#F1F8F5', GREEN])
CMAP_RED   = LinearSegmentedColormap.from_list('brand_red',   ['#FFFFFF', RED])
CMAP_DIV   = LinearSegmentedColormap.from_list('brand_div',   [RED, '#F2F2F2', BLUE])
BLUES_7    = [CMAP_BLUE(x) for x in np.linspace(0.18, 1.0, 7)]

sns.set_theme(style='white')
mpl.rcParams.update({
    'figure.facecolor': INK['surface'], 'axes.facecolor': INK['surface'], 'savefig.facecolor': INK['surface'],
    'figure.dpi': 105, 'savefig.dpi': 150,
    'font.size': 10, 'axes.titlesize': 10.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': INK['primary'],
    'axes.labelsize': 9, 'axes.labelcolor': INK['secondary'],
    'axes.edgecolor': INK['grid'], 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': INK['grid'], 'grid.linewidth': 0.7,
    'xtick.color': INK['secondary'], 'ytick.color': INK['secondary'], 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5,
    'text.parse_math': False,                      # '$' in money labels is text, not math
    'axes.prop_cycle': mpl.cycler(color=[BLUE, GREEN, BLUE_T, GREEN_T]),
})

# ---------------- Dimensions: fixed order and fixed colors ----------------
FAM = ['DD', 'CD', 'LN']
FAM_LABEL = {'DD': 'Demand deposits (DD)', 'CD': 'Certificates of deposit (CD)', 'LN': 'Loans (LN)'}
CIF_STATUS = ['Active customer', 'Connected (non-owner)', 'Former customer', 'Former connected']
ROLE_FAMILIES = ['Primary owner', 'Joint / co-owner', 'Co-borrower', 'Guarantor', 'Beneficial owner / controller', 'Authorized signer',
                 'Power of attorney / agent', 'Fiduciary / custodial', 'Beneficiary', 'Mailing / tax / interest', 'Other']
STRUCT_ORDER = ['Sole owner', 'Joint / co-borrowers', 'Guaranteed', 'Owner + signers / agents', 'Owner + other roles']
ORDER = {'IorB': ['Individual', 'Business'], 'ForD': ['Domestic', 'Foreign'], 'Select': ['Select', 'Non-Select'],
         'SelectProduct': ['Select', 'Non-Select'], 'SelectSegment': ['Select', 'Non-Select'],
         'ForeignSplit': ['Domestic', 'Dominant country', 'Other foreign'], 'ValueBand': ['Top 1%', 'Next 9%', 'Next 40%', 'Bottom 50%'],
         'Domain': FAM, 'AccountSelect': ['Select', 'Regular'], 'Status': CIF_STATUS,
         'Composition': ['Individuals only', 'Businesses only', 'Mixed'], 'HHForD': ['Domestic', 'Foreign', 'Mixed'],
         'HHSelect': ['Select', 'Non-Select'], 'HHType': ['Real household', 'Single CIF'],
         'SizeBand': ['1', '2', '3', '4–5', '6+'], 'RoleProfile': ['Owner only', 'Owner + other roles', 'Non-owner roles only'],
         'Structure': STRUCT_ORDER, 'StatusGroup': ['Active', 'Dormant / restricted', 'Closed', 'Charged-off', 'Other / unknown']}
COLOR = {'IorB': {'Individual': BLUE, 'Business': GREEN},
         'ForD': {'Domestic': BLUE, 'Foreign': GREEN},
         'Select': {'Select': GREEN, 'Non-Select': GREEN_T},
         'SelectProduct': {'Select': GREEN, 'Non-Select': GREEN_T},
         'SelectSegment': {'Select': GREEN, 'Non-Select': GREEN_T},
         'ForeignSplit': {'Domestic': BLUE, 'Dominant country': GREEN, 'Other foreign': GREEN_T},
         'Domain': {'DD': BLUE, 'CD': BLUE_T, 'LN': GREEN},
         'AccountSelect': {'Select': GREEN, 'Regular': GREEN_T},
         'Status': {'Active customer': BLUE, 'Connected (non-owner)': BLUE_T, 'Former customer': RED, 'Former connected': RED_T},
         'Composition': {'Individuals only': BLUE, 'Businesses only': GREEN, 'Mixed': BLUE_T},
         'HHForD': {'Domestic': BLUE, 'Foreign': GREEN, 'Mixed': GREEN_T},
         'HHSelect': {'Select': GREEN, 'Non-Select': GREEN_T},
         'HHType': {'Real household': BLUE, 'Single CIF': GRAY},
         'SizeBand': dict(zip(['1', '2', '3', '4–5', '6+'], [CMAP_BLUE(x) for x in [0.25, 0.45, 0.6, 0.8, 1.0]])),
         'RoleProfile': {'Owner only': BLUE, 'Owner + other roles': BLUE_T, 'Non-owner roles only': GRAY},
         'Structure': dict(zip(STRUCT_ORDER, [BLUE, GREEN, GREEN_T, BLUE_T, GRAY])),
         'StatusGroup': {'Active': BLUE, 'Dormant / restricted': BLUE_T, 'Closed': GRAY, 'Charged-off': RED, 'Other / unknown': GRAY_L}}
ROLE_COLOR = dict(zip(ROLE_FAMILIES, [BLUE, BLUE_T, '#2E5F94', GREEN, '#0B5D41', GREEN_T, '#7FB8A4', '#A3B4CC', '#D4A373', GRAY_L, GRAY]))
SEP = ' · '

def group_spec(dims):
    """Ordered labels + palette for 1 or 2 dims. Hue = dims[0]; full color / tint = levels of dims[1]."""
    if len(dims) == 1:
        return list(ORDER[dims[0]]), dict(COLOR[dims[0]])
    d1, d2 = dims[0], dims[1]
    labels, pal = [], {}
    for a in ORDER[d1]:
        base = COLOR[d1][a]
        full, tint = (BLUE, BLUE_T) if base == BLUE else (GREEN, GREEN_T) if base == GREEN else (base, GRAY_L)
        for j, b in enumerate(ORDER[d2]):
            lab = a + SEP + b
            labels.append(lab); pal[lab] = full if j == 0 else tint
    return labels, pal

def split_dims(dims):
    """Color encodes up to two dims; a third dim becomes side-by-side panels."""
    return (list(dims[:2]), dims[2]) if len(dims) == 3 else (list(dims), None)

def make_label(d, dims):
    lab = d[dims[0]].astype(str)
    for c in dims[1:]:
        lab = lab + SEP + d[c].astype(str)
    return lab

# ---------------- Formatters ----------------
def money(x, pos=None):
    if x is None or (isinstance(x, float) and np.isnan(x)): return ''
    s = '-' if x < 0 else ''; a = abs(x)
    if a >= 1e9: return f'{s}$' + f'{a/1e9:.1f}'.rstrip('0').rstrip('.') + 'B'
    if a >= 1e6: return f'{s}$' + f'{a/1e6:.1f}'.rstrip('0').rstrip('.') + 'M'
    if a >= 1e3: return f'{s}${a/1e3:.0f}K'
    return f'{s}${a:,.0f}'
MONEY = FuncFormatter(money)
PCT0  = PercentFormatter(100, decimals=0)
def fmt_int(x): return '' if pd.isna(x) else f'{x:,.0f}'
def fmt_pct(x): return '' if pd.isna(x) else f'{x:.1f}%'
def fmt_1(x):   return '' if pd.isna(x) else f'{x:,.1f}'
def fmt_2(x):   return '' if pd.isna(x) else f'{x:,.2f}'
def fmt_p(v):   return '' if pd.isna(v) else ('<0.001' if v < 0.001 else f'{v:.3f}')
def is_(s, v):  return s.eq(v).fillna(False).astype(bool)
def pct(a, b):  return a / b * 100 if b else np.nan

# ---------------- Figure helpers ----------------
def finish(fig, title=None, subtitle=None, name=None):
    """Left-aligned title block in ink colors, tight layout, optional PNG export, show."""
    h = fig.get_figheight()
    if title:
        block = 0.66 if subtitle else 0.42
        fig.tight_layout(rect=[0, 0, 1, 1 - block / h])
        fig.text(0.01, 1 - 0.14 / h, title, ha='left', va='top', fontsize=13, fontweight='bold', color=INK['primary'])
        if subtitle:
            fig.text(0.01, 1 - 0.40 / h, subtitle, ha='left', va='top', fontsize=9.3, color=INK['muted'])
    else:
        fig.tight_layout()
    if globals().get('SAVE_FIGS') and name:
        fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), bbox_inches='tight')
    plt.show(); plt.close(fig)

def style(ax, title=None, xlabel='', ylabel='', money_x=False, money_y=False, pct_x=False, pct_y=False):
    if title is not None: ax.set_title(title)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    if money_x: ax.xaxis.set_major_formatter(MONEY)
    if money_y: ax.yaxis.set_major_formatter(MONEY)
    if pct_x:   ax.xaxis.set_major_formatter(PCT0)
    if pct_y:   ax.yaxis.set_major_formatter(PCT0)
    return ax

def no_data(ax, msg='no data'):
    ax.text(0.5, 0.5, msg, ha='center', va='center', transform=ax.transAxes, color=INK['muted']); ax.set_xticks([]); ax.set_yticks([])
    return ax

def hbar(ax, labels, values, colors, fmt=fmt_int, hatches=None, height=0.62):
    """Horizontal bars with direct value labels. Negative values are drawn as 0."""
    y = np.arange(len(labels)); values = np.asarray(values, dtype=float)
    if len(values) == 0: return no_data(ax)
    bars = ax.barh(y, np.nan_to_num(values).clip(min=0), color=colors, height=height, edgecolor=INK['surface'], linewidth=0.8)
    if hatches:
        for b, hch in zip(bars, hatches): b.set_hatch(hch)
    ax.set_yticks(y); ax.set_yticklabels([str(l) for l in labels]); ax.invert_yaxis()
    ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid'])
    vmax = np.nanmax(values) if np.isfinite(values).any() and np.nanmax(values) > 0 else 1
    for b, v in zip(bars, values):
        if not np.isnan(v):
            ax.text(b.get_width() + vmax * 0.012, b.get_y() + b.get_height() / 2, fmt(v), va='center', ha='left', fontsize=8.3, color=INK['secondary'])
    ax.set_xlim(0, vmax * 1.22)
    return bars

def grouped_bars(ax, ct, order, pal, fmt=None, lo=None, hi=None):
    """ct: index = x categories, columns = groups (subset of order). lo/hi: optional CI bounds (same shape as ct)."""
    cats = list(ct.index); k = max(len([g for g in order if g in ct.columns]), 1); w = 0.8 / k
    x = np.arange(len(cats)); j = 0
    for g in order:
        if g not in ct.columns: continue
        vals = ct[g].values.astype(float); xs = x + (j - (k - 1) / 2) * w
        ax.bar(xs, np.nan_to_num(vals), width=w * 0.9, color=pal[g], label=g, edgecolor=INK['surface'], linewidth=0.5)
        top = vals.copy()
        if lo is not None and hi is not None and g in lo.columns:
            l = lo[g].reindex(ct.index).values.astype(float); h = hi[g].reindex(ct.index).values.astype(float)
            ok = ~np.isnan(l) & ~np.isnan(h) & ~np.isnan(vals)
            if ok.any():
                ax.errorbar(xs[ok], vals[ok], yerr=[np.clip(vals[ok] - l[ok], 0, None), np.clip(h[ok] - vals[ok], 0, None)],
                            fmt='none', ecolor=INK['secondary'], elinewidth=1, capsize=2.5, zorder=3)
                top[ok] = np.maximum(vals[ok], h[ok])
        if fmt:
            for xi, v, t in zip(xs, vals, top):
                if not np.isnan(v) and v > 0: ax.text(xi, t, fmt(v), ha='center', va='bottom', fontsize=7.2, color=INK['secondary'])
        j += 1
    ax.set_xticks(x); ax.set_xticklabels([str(c) for c in cats])
    ax.legend(); return ax

def stacked_100(ax, ct, cols, colors, min_label=7, height=0.62, labels=None):
    """Horizontal 100% stacked bars. ct: rows = bars, columns = stack pieces, values already in % of the row."""
    labels = labels if labels is not None else [str(i) for i in ct.index]
    y = np.arange(len(ct)); left = np.zeros(len(ct))
    for c in cols:
        if c not in ct.columns: continue
        vals = ct[c].fillna(0).values.astype(float); colr = colors[c] if isinstance(colors, dict) else colors
        ax.barh(y, vals, left=left, color=colr, label=str(c), height=height, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= min_label: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(colr))
        left += vals
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid'])
    return ax

def stacked_bars(ax, ct, cols, colors, width=0.8):
    """Vertical stacked bars (counts or $). ct: index = x, columns = pieces."""
    bottom = np.zeros(len(ct))
    for c in cols:
        if c not in ct.columns: continue
        vals = ct[c].fillna(0).values.astype(float)
        ax.bar(ct.index, vals, bottom=bottom, color=colors[c] if isinstance(colors, dict) else colors, label=str(c), width=width, edgecolor=INK['surface'], linewidth=0.4)
        bottom += vals
    return ax

def kpi_cards(kpis, title, subtitle=None, name=None, ncols=4):
    """kpis: list of (label, value, sub-label, accent color)."""
    nrows = math.ceil(len(kpis) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 1.45 * nrows), squeeze=False)
    for ax, it in zip(axes.flat, kpis + [None] * (nrows * ncols - len(kpis))):
        ax.axis('off')
        if it is None: continue
        label, value, sub, color = it
        ax.add_patch(mpl.patches.FancyBboxPatch((0.03, 0.06), 0.94, 0.88, boxstyle='round,pad=0.01,rounding_size=0.05', fc=INK['card'], ec=INK['grid'], transform=ax.transAxes))
        ax.plot([0.055, 0.055], [0.18, 0.82], color=color, lw=4, transform=ax.transAxes, solid_capstyle='round')
        ax.text(0.12, 0.76, label.upper(), fontsize=7.6, color=INK['muted'], transform=ax.transAxes, va='center')
        ax.text(0.12, 0.47, value, fontsize=16, fontweight='bold', color=INK['primary'], transform=ax.transAxes, va='center')
        ax.text(0.12, 0.2, sub, fontsize=7.6, color=INK['secondary'], transform=ax.transAxes, va='center')
    finish(fig, title, subtitle, name=name)

def median_ci(s, n_boot=300, alpha=0.05, seed=1):
    """95% bootstrap percentile interval for the median."""
    a = np.asarray(pd.Series(s).dropna(), dtype=float)
    if len(a) < 10: return (np.nan, np.nan)
    if len(a) > 20000: a = np.random.default_rng(seed).choice(a, 20000, replace=False)
    rng = np.random.default_rng(seed)
    meds = np.median(a[rng.integers(0, len(a), size=(n_boot, len(a)))], axis=1)
    return (float(np.quantile(meds, alpha / 2)), float(np.quantile(meds, 1 - alpha / 2)))

def median_table(d, col, row, colgrp, row_order, col_order, holders=True):
    """Median of `col` for every row × column group (among holders if holders=True) with bootstrap CIs. Returns (median, lo, hi)."""
    dd = d[[col, row, colgrp]].dropna()
    if holders: dd = dd[dd[col] > 0]
    med = pd.DataFrame(np.nan, index=row_order, columns=col_order, dtype=float); lo = med.copy(); hi = med.copy()
    for r in row_order:
        for c in col_order:
            s = dd.loc[is_(dd[row], r) & is_(dd[colgrp], c), col]
            if len(s):
                med.loc[r, c] = s.median(); lo.loc[r, c], hi.loc[r, c] = median_ci(s)
    return med, lo, hi

def lump(s, k=6, other='Other'):
    """Keep the k most frequent levels, fold the rest into `other`."""
    keep = s.value_counts().head(k).index
    return s.where(s.isin(keep) | s.isna(), other)

def text_on(color):
    r, g, b = mpl.colors.to_rgb(color)
    return 'white' if (0.2126 * r + 0.7152 * g + 0.0722 * b) < 0.5 else INK['primary']

def kde_by(ax, d, col, group_col, order, pal, clip=None, bw=1.0):
    """Overlaid density curves per group with a dashed median line."""
    drawn = 0
    for g in order:
        s = d.loc[is_(d[group_col], g), col].dropna()
        if clip is not None: s = s.clip(*clip)
        if len(s) < 5 or s.nunique() < 2: continue
        sns.kdeplot(s, ax=ax, color=pal[g], fill=True, alpha=0.16, linewidth=2, cut=0, bw_adjust=bw, label=f'{g} (n={len(s):,}, median {s.median():,.0f})')
        ax.axvline(s.median(), color=pal[g], lw=1, ls='--', alpha=0.9); drawn += 1
    ax.set_yticks([]); ax.set_ylabel(''); ax.grid(False)
    if drawn:
        ax.set_ylim(0, ax.get_ylim()[1] * 1.45); ax.legend(loc='upper right')
    else: ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes, color=INK['muted'])
    return ax

def box_by(ax, d, col, group_col, order, pal, holders_only=True, annotate=True):
    """Horizontal box plot per group on a log $ axis, among holders (balance > 0). Annotates % holders and n."""
    dd = d[[col, group_col]].dropna()
    pop = dd.groupby(group_col)[col].size()
    if holders_only: dd = dd[dd[col] > 0]
    present = [g for g in order if is_(dd[group_col], g).any()]
    if not present:
        ax.text(0.5, 0.5, 'no holders', ha='center', va='center', transform=ax.transAxes, color=INK['muted']); ax.set_yticks([]); return ax
    dd = dd[dd[group_col].isin(present)].copy(); dd[group_col] = dd[group_col].astype(str)
    sns.boxplot(data=dd, x=col, y=group_col, order=present, hue=group_col, palette={g: pal[g] for g in present}, legend=False,
                showfliers=False, width=0.55, linewidth=1.0, ax=ax, log_scale=True)
    lo, hi = ax.get_xlim(); ax.set_xlim(lo, hi * 60)
    for i, g in enumerate(present):
        s = dd.loc[dd[group_col] == g, col]
        ax.text(s.median(), i - 0.36, money(s.median()), ha='center', va='bottom', fontsize=7.8, color=INK['secondary'])
        if annotate:
            share = len(s) / pop.get(g, np.nan) * 100
            ax.text(0.995, i, f'{share:.0f}% hold · n={len(s):,}', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.8, color=INK['muted'])
    ax.xaxis.set_major_formatter(MONEY); ax.set_ylabel(''); ax.set_xlabel('')
    ax.grid(axis='x', color=INK['grid']); ax.grid(axis='y', visible=False)
    return ax

def styled(tbl, pct_cols=(), money_cols=(), int_cols=(), one_dec=(), two_dec=(), gradient_cols=(), cmap=None):
    fmt = {}
    fmt.update({c: fmt_pct for c in pct_cols if c in tbl.columns})
    fmt.update({c: money for c in money_cols if c in tbl.columns})
    fmt.update({c: fmt_int for c in int_cols if c in tbl.columns})
    fmt.update({c: fmt_1 for c in one_dec if c in tbl.columns})
    fmt.update({c: fmt_2 for c in two_dec if c in tbl.columns})
    st = tbl.style.format(fmt, na_rep='')
    g = [c for c in gradient_cols if c in tbl.columns and pd.to_numeric(tbl[c], errors='coerce').notna().any()]
    if g: st = st.background_gradient(cmap=cmap or CMAP_BLUE, subset=g, vmin=0)
    return st

def heat(ax, t, annot=None, fmt='.0f', cmap=None, vmin=0, vmax=None, suffix='', size=8.5):
    """Annotated heatmap in brand style. `annot` = DataFrame of strings (same shape) to print instead of the values."""
    t = t.astype(float)
    if t.size == 0 or t.isna().all().all(): return no_data(ax)
    sns.heatmap(t, annot=annot.values if annot is not None else True, fmt='' if annot is not None else fmt, cmap=cmap or CMAP_BLUE, cbar=False, ax=ax,
                linewidths=1, linecolor=INK['surface'], annot_kws={'size': size}, vmin=vmin, vmax=vmax)
    if suffix and annot is None:
        for tx in ax.texts: tx.set_text(tx.get_text() + suffix)
    ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False); ax.tick_params(axis='y', labelrotation=0)
    return ax

def cramers_v(a, b):
    try:
        t = pd.crosstab(a.astype(str), b.astype(str))
        t = t.loc[t.sum(1) > 0, t.sum(0) > 0]
        if t.shape[0] < 2 or t.shape[1] < 2: return np.nan
        chi2 = chi2_contingency(t, correction=False)[0]
        return float(np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1))))
    except Exception:
        return np.nan

# ---------------- Concentration ----------------
def lorenz(s):
    s = np.sort(np.asarray(s, dtype=float).clip(min=0))[::-1]
    if len(s) == 0 or s.sum() <= 0: return None, None
    return np.arange(1, len(s) + 1) / len(s) * 100, np.cumsum(s) / s.sum() * 100
def gini(s):
    s = np.sort(np.asarray(s, dtype=float).clip(min=0))
    if len(s) == 0 or s.sum() <= 0: return np.nan
    n = len(s); return float((2 * np.sum(np.arange(1, n + 1) * s) / (n * s.sum())) - (n + 1) / n)
def top_share(s, p):
    s = s[s > 0].sort_values(ascending=False); k = max(int(len(s) * p), 1)
    return s.iloc[:k].sum() / s.sum() * 100 if len(s) and s.sum() > 0 else np.nan
def holders_for(s, target=80):
    s = np.sort(s[s > 0].values)[::-1]
    if len(s) == 0 or s.sum() <= 0: return np.nan
    cum = np.cumsum(s) / s.sum() * 100
    return (np.searchsorted(cum, target) + 1) / len(s) * 100
def value_band(s):
    """Rank by value (largest first) into Top 1% / Next 9% / Next 40% / Bottom 50%."""
    pr = s.rank(ascending=False, method='first') / max(len(s), 1)
    return pd.Series(np.select([pr <= 0.01, pr <= 0.10, pr <= 0.50], ORDER['ValueBand'][:3], ORDER['ValueBand'][3]), index=s.index)

# ---------------- Time: survival and rates ----------------
def kaplan_meier(t, e):
    """Kaplan–Meier survival: t = years observed, e = 1 if the event happened (left), 0 if censored (still here)."""
    t = np.asarray(t, dtype=float); e = np.asarray(e, dtype=bool); ok = ~np.isnan(t); t, e = t[ok], e[ok]
    if len(t) == 0 or e.sum() == 0: return np.array([0.0]), np.array([1.0])
    ts = np.sort(t); times, d = np.unique(t[e], return_counts=True)
    at_risk = len(ts) - np.searchsorted(ts, times, side='left')
    s = np.cumprod(1 - d / at_risk)
    return np.r_[0.0, times], np.r_[1.0, s]
def km_median(x, s):
    below = np.where(s <= 0.5)[0]
    return float(x[below[0]]) if len(below) else np.nan
def km_at(x, s, yrs):
    return float(s[np.searchsorted(x, yrs, side='right') - 1]) * 100

def alive_at(start, end, t):
    """Alive at date t: started on/before t and not ended by t (end NaT = still alive)."""
    return (start <= t) & (end.isna() | (end > t))

def annual_rates(start, end, years, snapshot):
    """Rate for year Y = ended during Y ÷ alive on 1 January Y. end NaT = still alive. The snapshot year is partial and annualized."""
    start = pd.to_datetime(pd.Series(start)); end = pd.to_datetime(pd.Series(end))
    ok = start.notna(); start, end = start[ok], end[ok]
    out = []
    for Y in years:
        y0 = pd.Timestamp(int(Y), 1, 1); y1 = min(pd.Timestamp(int(Y) + 1, 1, 1), snapshot)
        if y1 <= y0: continue
        alive = (start < y0) & (end.isna() | (end >= y0))
        left = alive & end.ge(y0) & end.lt(y1)
        frac = (y1 - y0).days / 365.25
        out.append({'year': int(Y), 'alive 1 Jan': int(alive.sum()), 'ended in year': int(left.sum()),
                    'annual rate %': left.sum() / alive.sum() * 100 / frac if alive.sum() else np.nan})
    return pd.DataFrame(out).set_index('year') if out else pd.DataFrame(columns=['alive 1 Jan', 'ended in year', 'annual rate %'])

# ---------------- Effect sizes ----------------
def cliff(a, b):
    """Mann-Whitney p-value and Cliff's δ (A vs B): P(A > B) − P(A < B)."""
    a = pd.Series(a).dropna(); b = pd.Series(b).dropna()
    if len(a) < 10 or len(b) < 10: return np.nan, np.nan
    u = mannwhitneyu(a, b, alternative='two-sided')
    return float(u.pvalue), float(2 * u.statistic / (len(a) * len(b)) - 1)
def effect_label(d):
    if pd.isna(d): return 'n too small'
    a = abs(d); return 'negligible' if a < 0.147 else 'small' if a < 0.33 else 'medium' if a < 0.474 else 'large'

# ---------------- Text / value rules ----------------
def classify(s, rules, default='Other'):
    """First regex in `rules` [(pattern, label), …] that matches the upper-cased text wins; blank stays blank."""
    up = s.fillna('').astype(str).str.upper().str.strip()
    out = pd.Series([default] * len(s), index=s.index, dtype=object)
    done = pd.Series(False, index=s.index)
    for p, v in rules:
        m = up.str.contains(p, regex=True).fillna(False).astype(bool) & ~done
        out[m] = v; done |= m
    return out.where(up.ne(''), np.nan)

def mix_label(d, prefix, fams=('DD', 'CD', 'LN')):
    """'DD+LN'-style label from count columns `prefix`DD, `prefix`CD, `prefix`LN (> 0 = held)."""
    has = np.column_stack([d[f'{prefix}{f}'].fillna(0).to_numpy() > 0 for f in fams]) if len(d) else np.zeros((0, len(fams)), bool)
    return pd.Series(['+'.join(f for f, h in zip(fams, row) if h) or 'None' for row in has], index=d.index, dtype=object)
MIX_ORDER = ['DD', 'DD+CD', 'DD+LN', 'DD+CD+LN', 'CD', 'CD+LN', 'LN', 'None']

AGE_BANDS  = ['<25', '25–34', '35–44', '45–54', '55–64', '65–74', '75+']
YEAR_BANDS = ['<1', '1–2', '3–5', '6–10', '11–20', '20+']
TIER_BANDS = ['≤ $0', '$0–10K', '$10–50K', '$50–250K', '$250K–1M', '$1M+']
def age_band(s):  return pd.cut(s, [0, 25, 35, 45, 55, 65, 75, np.inf], right=False, labels=AGE_BANDS).astype(object)
def year_band(s): return pd.cut(s, [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
def tier_band(s): return pd.cut(s, [-np.inf, 0, 10e3, 50e3, 250e3, 1e6, np.inf], labels=TIER_BANDS).astype(object)
def count_band(s, top=6):
    return s.map(lambda v: np.nan if pd.isna(v) else (f'{top}+' if v >= top else f'{int(v)}')).astype(object)
COUNT_BANDS = ['0', '1', '2', '3', '4', '5', '6+']

print('Setup OK · pandas', pd.__version__, '· seaborn', sns.__version__)

# 2. Load & configuration

The configuration cell is the only one you normally edit: file, snapshot, the column-name map, and **every business rule** (owner row, Select, open statuses, role families, key-person priority). In Colab the file is uploaded automatically if it is not already in the working directory. `FILE_PATH = 'DEMO'` runs the whole notebook on a synthetic file with the same columns and the same kinds of data problems (no real data — useful to check the notebook end to end).

In [ ]:
# ============================ CONFIGURATION ============================
FILE_PATH     = 'Data EDA - Households CIFs Accounts.xlsx'   # xlsx / xls / csv / parquet · 'DEMO' = synthetic data with the same columns
SHEET         = 0                   # sheet name or index (Excel only)
HEADER        = 0                   # header row (0 = first row)
SNAPSHOT_DATE = None                # e.g. '2026-09-30'. None = latest account-opening date in the file (= extract date)
HISTORY_START = 2000                # first year drawn in the year-by-year views (§12–§13)

# ---- Column names: name used here → name(s) in the file. Case-insensitive; a unique prefix is enough ('Birth Date' finds 'Birth Date (YYYY-MM-DD)') ----
COLS = {
    'HouseholdID':       ['Household ID', 'Household Group ID'],
    'HouseholdName':     ['Household Group Name', 'Household Name'],
    'HHStartFile':       ['Household Relationship Start Date'],
    'CIF':               ['Customer CIF', 'CIF'],
    'CustomerName':      ['Customer Name'],
    'IorB':              ['Person Type'],
    'ForD':              ['Residence Origin'],
    'CIFCreated':        ['CIF Creation Date'],
    'BirthRaw':          ['Birth Date'],
    'Segment':           ['Customer Segment Code'],
    'SegmentDesc':       ['Customer Segment Description'],
    'RoleCode':          ['Relationship Role Code'],
    'RoleDesc':          ['Relationship Role Description'],
    'OwnershipCategory': ['Ownership Category'],
    'AccountNumber':     ['Account Number'],
    'AccountName':       ['Account Name'],
    'DomainRaw':         ['Product Domain'],
    'Subtype':           ['SCC / Product Subtype', 'Product Subtype'],
    'Product':           ['Product Description'],
    'AccountSegment':    ['AccountSegment', 'Account Segment'],
    'LargeRel':          ['Large Relationship'],
    'BranchCode':        ['Branch Code'],
    'Branch':            ['Branch Name'],
    'OfficerCode':       ['Relationship Officer Code'],
    'Officer':           ['Relationship Officer Name'],
    'StatusCode':        ['Account Status Code'],
    'StatusDesc':        ['Account Status Description'],
    'Opened':            ['Account Date Opened'],
    'Closed':            ['Account Date Closed'],
    'Balance':           ['Current Balance'],
    'Country':           ['Country'],
}
REQUIRED = ['HouseholdID', 'CIF', 'IorB', 'ForD', 'AccountNumber', 'DomainRaw', 'Opened', 'Balance']
PII_COLS = ['CustomerName', 'HouseholdName', 'AccountName', 'Officer']       # used only to derive flags in §2.1, then dropped

# ---- Value normalization (first matching regex wins; matching is on the upper-cased text) ----
IORB_RULES   = [(r'^IND|PERSON|NATURAL', 'Individual'), (r'BUS|COMP|CORP|ENTIT|ORGAN|NON.?IND', 'Business')]
FORD_RULES   = [(r'FOREIGN|INTERNATIONAL|NON.?RES|ALIEN|INTL', 'Foreign'), (r'DOMESTIC|RESIDENT|LOCAL|US\b', 'Domestic')]
DOMAIN_RULES = [(r'DEMAND|DDA|CHECK|SAVING|MONEY MARKET|^DD$', 'DD'), (r'CERTIF|TIME DEP|^CD$', 'CD'), (r'LOAN|^LN$|CREDIT|MORTGAGE|LEND', 'LN')]

# ---- Ownership: the owner row carries the balance ----
OWNER_ROLE_PATTERN = r'PRIMARY'         # Relationship Role Description matching this = the CIF that OWNS the account (the rule that decides ownership)
OWNER_CATEGORY     = r'PRIMARY'         # Ownership Category matching this — used only where the role description is blank, and cross-checked
OWNER_ROLE_CODES   = ['1']              # last fallback, where both texts are blank

# ---- Select ----
SELECT_PATTERN = r'SELECT'              # AccountSegment (or product description) / segment description matching this = Select
SELECT_BASIS   = 'product'              # 'product' = CIF owns a Select account · 'segment' = segment description says Select

# ---- Account status → status group (first match wins). Open = still on the books ----
STATUS_RULES = [(r'CHARGE|LOSS|WRITE.?OFF|WRITTEN', 'Charged-off'),
                (r'CLOS|PAID|PAY.?OFF|MATUR|REDEEM|CANCEL|TERMINAT|RENEWED', 'Closed'),
                (r'DORMAN|RESTRICT|INACTIVE|FROZEN|HOLD|BLOCK|ESCHEAT', 'Dormant / restricted'),
                (r'ACTIVE|OPEN|CURRENT|NEW|PENDING|REGULAR|PERFORM', 'Active')]
OPEN_GROUPS  = ['Active', 'Dormant / restricted']

# ---- Role description → role family (first match wins; the owner comes from Ownership Category, not from here) ----
ROLE_RULES = [(r'CO.?BORR|CO.?MAKER|CO.?SIGN|CO.?APPLIC|BORROWER', 'Co-borrower'),
              (r'GUARANT|SURETY|ENDORS|PLEDG', 'Guarantor'),
              (r'BENEFICIAL|CONTROL|UBO|SHAREHOLD|PARTNER|OFFICER|DIRECTOR|PRESIDENT|MEMBER|MANAG', 'Beneficial owner / controller'),
              (r'SIGN|AUTHORI', 'Authorized signer'),
              (r'ATTORNEY|POA|AGENT|REPRESENT|PROXY', 'Power of attorney / agent'),
              (r'TRUST|CUSTOD|GUARDIAN|FIDUC|EXECUT|ADMINIS|CONSERV|UTMA|UGMA|MINOR|ESTATE|REP PAYEE|PAYEE', 'Fiduciary / custodial'),
              (r'BENEFICIAR|POD|ITF|PAYABLE ON|TOD', 'Beneficiary'),
              (r'INTEREST|1099|TAX|MAIL|STATEMENT|ADDRESS|DUPLICATE|NOTICE|CC:', 'Mailing / tax / interest'),
              (r'JOINT|CO.?OWN|SECONDARY|AND.?OR|PRIMARY|OWNER|TENAN', 'Joint / co-owner')]

# ---- The person behind a business (§10): role families that can make an individual the key person, in priority order ----
KEY_PERSON_PRIORITY = ['Beneficial owner / controller', 'Guarantor', 'Joint / co-owner', 'Co-borrower', 'Authorized signer', 'Power of attorney / agent', 'Fiduciary / custodial']
KEY_PERSON_USE_HOUSEHOLD = True         # no role-linked individual → use the individuals of the same real household

# ---- Other knobs ----
OFFICER_INACTIVE_PATTERN = r'RETIRED|INACTIVE|TERMINATED|FORMER|NO LONGER|DO NOT USE'
HOUSE_CODE_SHARE = 3.0                  # an officer code holding more than this % of open accounts is a pooled "house" code
DOMINANT_COUNTRY = None                 # None = the most frequent foreign country gets its own level in ForeignSplit
TOP_COUNTRIES    = 15
TOP_PRODUCTS     = 12

# ---- Outputs ----
SAVE_FIGS    = False                    # True → every figure also saved as PNG in FIG_DIR (deck / Power BI)
FIG_DIR      = 'figures'
EXPORT_CLEAN = True                     # True → accounts / CIFs / households tables written to EXPORT_DIR (Power BI source)
EXPORT_DIR   = 'exports'

if SAVE_FIGS: os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---- DEMO: synthetic extract with the same columns, grain and known data problems as the real one. Every number it produces is fake. ----
def make_demo(n_groups=4000, seed=7):
    rng = np.random.default_rng(seed)
    snap = pd.Timestamp('2026-09-30')
    ROLE = {'1': 'Primary Owner', '2': 'Joint Owner', 'S': 'Secondary Owner', '3': 'Authorized Signer', 'G': 'Guarantor', 'B': 'Beneficial Owner',
            'P': 'Power of Attorney', 'T': 'Trustee / Custodian', 'D': 'Beneficiary (POD)', '4': 'Co-Borrower', 'I': 'Interest Recipient'}
    STATUS = {'Active': 1, 'Closed / Inactive': 2, 'Dormant / Restricted': 3, 'Charged-Off / Loss': 4, 'Paid Off': 5, 'Matured': 6,
              'Restricted - Legal Hold': 7, 'New / Pending': 8}
    PRODUCTS = {'DD': [('11', 'PERSONAL CHECKING', 0, 'I'), ('12', 'SELECT PERSONAL CHECKING', 1, 'I'), ('21', 'BUSINESS CHECKING', 0, 'B'),
                       ('22', 'SELECT BUSINESS CHECKING', 1, 'B'), ('31', 'MONEY MARKET', 0, 'IB'), ('32', 'SELECT MONEY MARKET', 1, 'IB')],
                'CD': [('6', 'CD 12 MONTHS', 0, 'IB'), ('7', 'SELECT CD 6 MONTHS', 1, 'IB'), ('8', 'JUMBO CD', 0, 'IB')],
                'LN': [('84', 'PRIVATE SEC INSURED', 0, 'IB'), ('50', 'CRE TERM LOAN', 0, 'B'), ('51', 'RESIDENTIAL MORTGAGE', 0, 'I'),
                       ('60', 'SBA 7A LOAN', 0, 'B'), ('70', 'CONSUMER LOAN', 0, 'I'), ('75', 'LINE OF CREDIT', 0, 'IB')]}
    DOMAIN_NAME = {'DD': 'Demand Deposit', 'CD': 'Certificate of Deposit', 'LN': 'Loan Account'}
    BRANCHES = [(201, 'Main Branch'), (211, 'Trade Center'), (230, 'International'), (240, 'Corporate Banking'), (250, 'Virtual Branch'), (260, 'Island Branch'), (270, 'North Branch')]
    OFFICERS = [(str(c), f'OFFICER {c}') for c in range(100, 140)] + [('177', 'RETIRED-OFFICER 177'), ('181', 'RETIRED-OFFICER 181'), ('420', 'HOUSE ACCOUNTS'), ('442', 'HOUSE INTERNATIONAL')]
    C_FOR = ['VENEZUELA'] * 12 + ['ARGENTINA'] * 3 + ['PANAMA'] * 2 + ['COLOMBIA'] * 2 + ['SPAIN', 'MEXICO', 'BRITISH VIRGIN ISLANDS', 'DOMINICAN REPUBLIC', 'MISSING DATA']
    C_DOM = ['UNITED STATES'] * 8 + ['PUERTO RICO'] * 2 + ['MISSING DATA']
    cifs, accts, links = {}, [], []
    counter = {'cif': 100000, 'acct': 270000000}

    def new_cif(kind, hh=None, ford=None):
        counter['cif'] += int(rng.integers(1, 7))
        cid = str(counter['cif']) if rng.random() > 0.01 else f'(AA{counter["cif"] % 10000:04d}'
        if ford is None: ford = 'Foreign' if rng.random() < (0.55 if kind == 'Individual' else 0.35) else 'Domestic'
        created = snap - pd.Timedelta(days=int(min(rng.gamma(1.6, 1900), 38 * 365)))
        born = created - pd.Timedelta(days=int((rng.uniform(18, 72) if kind == 'Individual' else rng.exponential(7)) * 365.25))
        country = str(rng.choice(C_FOR if ford == 'Foreign' else C_DOM))
        geo = 'I' if ford == 'Foreign' else ('P' if country == 'PUERTO RICO' else 'D')
        tier = int(rng.integers(1, 5)); sel = tier >= 3 and rng.random() < 0.8
        seg = f'{geo}{"P" if kind == "Individual" else "B"}{tier}' if rng.random() > 0.06 else 'Unassigned'
        if seg == 'Unassigned': desc, sel = 'Unassigned Segment', False
        else: desc = ('Intl ' if geo == 'I' else 'Domestic ') + ('Business' if kind == 'Business' else 'Personal') + (' Select' if sel else '')
        cifs[cid] = dict(hh=hh or cid, kind=kind, ford=ford, created=created, born=born, country=country, seg=seg, desc=desc, select=sel)
        return cid

    LIFE = {'DD': 9.0, 'CD': 2.5, 'LN': 6.0}
    def new_account(owner, dom, others=()):
        c = cifs[owner]
        counter['acct'] += int(rng.integers(1, 40))
        start = c['created'] + pd.Timedelta(days=int(rng.exponential(500)))
        if start >= snap: start = snap - pd.Timedelta(days=int(rng.integers(1, 200)))
        life = rng.exponential(LIFE[dom] * (0.5 if (c['ford'] == 'Foreign' and start.year < 2016) else 1.0) * (1.4 if c['select'] else 1.0))
        end = start + pd.Timedelta(days=int(life * 365.25))
        is_open = end >= snap
        kind = 'I' if c['kind'] == 'Individual' else 'B'
        pool = [p for p in PRODUCTS[dom] if kind in p[3]]
        sel_pool, reg_pool = [p for p in pool if p[2] == 1], [p for p in pool if p[2] == 0]
        p = sel_pool[rng.integers(len(sel_pool))] if (c['select'] and sel_pool and rng.random() < 0.7) else reg_pool[rng.integers(len(reg_pool))]
        if is_open:
            u = rng.random(); status = 'Dormant / Restricted' if u < 0.07 else 'Restricted - Legal Hold' if u < 0.075 else 'New / Pending' if u < 0.08 else 'Active'
            scale = {'DD': 9.3, 'CD': 11.0, 'LN': 11.8}[dom] + (1.3 if kind == 'B' else 0) + (0.9 if c['select'] else 0)
            bal = float(np.round(rng.lognormal(scale, 1.5), 2))
            if dom == 'DD' and rng.random() < 0.02: bal = -float(np.round(rng.lognormal(6, 1), 2))
            closed = snap + pd.Timedelta(days=int(rng.integers(30, 700))) if (dom == 'CD' and rng.random() < 0.25) else pd.NaT   # some open CDs carry their maturity date
        else:
            status = {'DD': ['Closed / Inactive'] * 9 + ['Charged-Off / Loss'], 'CD': ['Matured'] * 5 + ['Closed / Inactive'] * 5,
                      'LN': ['Paid Off'] * 7 + ['Closed / Inactive'] * 2 + ['Charged-Off / Loss']}[dom][rng.integers(10)]
            bal, closed = 0.0, end
        if c['ford'] == 'Foreign': br = BRANCHES[2] if rng.random() < 0.7 else BRANCHES[rng.integers(len(BRANCHES))]
        elif kind == 'B': br = BRANCHES[3] if rng.random() < 0.4 else BRANCHES[rng.integers(len(BRANCHES))]
        else: br = BRANCHES[5] if c['country'] == 'PUERTO RICO' else BRANCHES[rng.choice([0, 1, 4, 6])]
        off = OFFICERS[-1] if (c['ford'] == 'Foreign' and kind == 'I' and rng.random() < 0.5) else OFFICERS[-2] if rng.random() < 0.1 else OFFICERS[rng.integers(len(OFFICERS) - 2)]
        accts.append(dict(num=counter['acct'], dom=dom, sub=p[0], prod=p[1], sel=p[2], bcode=br[0], bname=br[1], ocode=off[0], oname=off[1],
                          status=status, opened=start, closed=closed, bal=bal))
        idx = len(accts) - 1
        links.append((owner, idx, '1'))
        for o, r in dict.fromkeys(others): links.append((o, idx, r))

    def pick_dom(c, kind):
        dom_ = cifs[c]['ford'] == 'Domestic'
        p = {('I', True): [0.70, 0.20, 0.10], ('I', False): [0.86, 0.12, 0.02], ('B', True): [0.66, 0.08, 0.26], ('B', False): [0.88, 0.09, 0.03]}[(kind, dom_)]
        return str(rng.choice(FAM, p=p))

    for g in range(n_groups):
        r = rng.random()
        if r < 0.52:                                   # one individual, no household group
            a = new_cif('Individual')
            for _ in range(1 + rng.poisson(0.5)):
                dom = pick_dom(a, 'I'); others = []
                if dom != 'LN' and rng.random() < 0.18: others.append((new_cif('Individual', ford=cifs[a]['ford']), '2'))
                if dom == 'LN' and rng.random() < 0.3:  others.append((new_cif('Individual', ford=cifs[a]['ford']), '4'))
                if rng.random() < 0.04: others.append((new_cif('Individual'), 'D'))
                if rng.random() < 0.02: others.append((new_cif('Individual'), 'P'))
                new_account(a, dom, others)
        elif r < 0.70:                                 # one business, no group: its signers / guarantors get their own one-CIF "households"
            b = new_cif('Business')
            signers = [new_cif('Individual', ford=cifs[b]['ford']) for _ in range(1 + rng.poisson(0.6))]
            for _ in range(1 + rng.poisson(0.6)):
                dom = pick_dom(b, 'B')
                others = [(s, '3') for s in signers if rng.random() < 0.8]
                if dom == 'LN' and rng.random() < 0.6: others.append((signers[0], 'G'))
                if rng.random() < 0.15: others.append((signers[-1], 'B'))
                new_account(b, dom, others)
            if rng.random() < 0.35: new_account(signers[0], 'DD')                     # the person behind it banks here too
        elif r < 0.88:                                 # family household
            hid = str(2000000 + g); ford = 'Foreign' if rng.random() < 0.5 else 'Domestic'
            members = [new_cif('Individual', hh=hid, ford=ford) for _ in range(2 + rng.poisson(0.5))]
            made = 0
            for i, m in enumerate(members):
                for _ in range(rng.poisson(0.9)):
                    others = []
                    if rng.random() < 0.45: others.append((members[(i + 1) % len(members)], '2'))
                    if rng.random() < 0.05: others.append((members[-1], 'T'))
                    if rng.random() < 0.03: others.append((members[-1], 'I'))
                    new_account(m, pick_dom(m, 'I'), others); made += 1
            if made == 0: new_account(members[0], 'DD', [(members[1], 'S')])
        else:                                          # business group: companies + the people behind them, one household
            hid = str(3000000 + g); ford = 'Foreign' if rng.random() < 0.4 else 'Domestic'
            people = [new_cif('Individual', hh=hid, ford=ford) for _ in range(1 + rng.poisson(0.7))]
            firms = [new_cif('Business', hh=hid, ford=ford) for _ in range(1 + rng.poisson(0.8))]
            for f in firms:
                for _ in range(1 + rng.poisson(0.9)):
                    dom = pick_dom(f, 'B')
                    others = [(p_, '3') for p_ in people if rng.random() < 0.7]
                    if dom == 'LN': others.append((people[0], 'G'))
                    if rng.random() < 0.3: others.append((people[0], 'B'))
                    if len(firms) > 1 and rng.random() < 0.05: others.append((firms[0] if f != firms[0] else firms[-1], '3'))
                    new_account(f, dom, others)
            for p_ in people:
                if rng.random() < 0.5: new_account(p_, 'DD')

    A = pd.DataFrame(accts); L = pd.DataFrame(links, columns=['cif', 'a', 'role'])
    C = pd.DataFrame.from_dict(cifs, orient='index')
    rows = L.join(A, on='a').join(C, on='cif')
    # CIF-level values in the formats the extract really uses (mixed Python types, placeholders, blanks)
    def birth_val(ts, u, kind):
        if u < (0.17 if kind == 'Individual' else 0.25): return None
        if u < 0.52: return ts.strftime('%Y-%m-%d 00:00:00')
        if u < 0.74: return ts
        if u < 0.86: return int(f'{ts.year}{ts.dayofyear:03d}')        # Julian YYYYDDD
        if u < 0.88: return 0
        if u < 0.89: return '1900-01-01'
        return ts.strftime('%Y-%m-%d')
    u = rng.random(len(C))
    birth = {cid: birth_val(b, x, k) for cid, b, x, k in zip(C.index, C['born'], u, C['kind'])}
    created = {cid: (None if x < 0.13 else ts) for cid, ts, x in zip(C.index, C['created'], rng.random(len(C)))}
    hh_start = rows.assign(c=rows['cif'].map(created)).groupby('hh').apply(lambda d: pd.to_datetime(d['c']).min() if pd.to_datetime(d['c']).notna().any() else d['opened'].min())
    own_open_bal = rows[rows['role'].eq('1') & rows['closed'].isna()].groupby('cif')['bal'].sum()
    large = (own_open_bal > 2e6).map({True: 'Y', False: 'N'})
    out = pd.DataFrame({
        'Household ID': rows['hh'], 'Household Group Name': np.where(rows['hh'] == rows['cif'], 'CUSTOMER ' + rows['cif'], 'GROUP ' + rows['hh'].astype(str)),
        'Household Relationship Start Date': rows['hh'].map(hh_start), 'Customer CIF': rows['cif'], 'Customer Name': 'CUSTOMER ' + rows['cif'],
        'Person Type (Individual|Business)': rows['kind'], 'Residence Origin (Domestic|Foreign)': rows['ford'].map({'Domestic': 'Domestic', 'Foreign': 'Foreign / International'}),
        'CIF Creation Date': rows['cif'].map(created), 'Birth Date (YYYY-MM-DD)': rows['cif'].map(birth),
        'Customer Segment Code': rows['seg'], 'Customer Segment Description': rows['desc'],
        'Relationship Role Code': rows['role'], 'Relationship Role Description': rows['role'].map(ROLE),
        'Ownership Category': np.where(rows['role'].eq('1'), 'Primary Owner', 'Co-Owner / Authorized / Other'),
        'Account Number': rows['num'].astype('int64'), 'Account Name': 'ACCOUNT ' + rows['num'].astype(str),
        'Product Domain': rows['dom'].map(DOMAIN_NAME), 'SCC / Product Subtype': rows['sub'], 'Product Description (source fields)': rows['prod'],
        'AccountSegment': np.where(rows['sel'].eq(1), 'Select Account', 'Regular Account'), 'Large Relationship': rows['cif'].map(large).fillna('N'),
        'Branch Code': rows['bcode'].astype('int64'), 'Branch Name': rows['bname'], 'Relationship Officer Code': rows['ocode'], 'Relationship Officer Name': rows['oname'],
        'Account Status Code': rows['status'].map(STATUS).astype('int64'), 'Account Status Description': rows['status'],
        'Account Date Opened': rows['opened'], 'Account Date Closed': rows['closed'], 'Current Balance ($)': rows['bal'], 'Country': rows['country']})
    # ---- Known defects, injected on purpose so every check in §3–§4 has something to find ----
    closed_n = out.loc[out['Account Status Description'].isin(['Closed / Inactive', 'Paid Off', 'Matured']), 'Account Number'].drop_duplicates()
    a1 = closed_n.sample(30, random_state=5); out.loc[out['Account Number'].isin(a1), 'Account Date Closed'] = pd.NaT                 # closed without a close date
    a2 = closed_n.drop(a1.index).sample(30, random_state=6); out.loc[out['Account Number'].isin(a2), 'Current Balance ($)'] = 1234.56  # residual balance
    open_n = out.loc[out['Account Status Description'].eq('Active') & out['Product Domain'].eq('Demand Deposit'), 'Account Number'].drop_duplicates()
    a3 = open_n.sample(20, random_state=7); out.loc[out['Account Number'].isin(a3), 'Account Date Closed'] = snap - pd.Timedelta(days=400)  # open with a past close date
    multi = out.groupby('Account Number')['Customer CIF'].transform('size').gt(1)
    out = out.drop(out[out['Relationship Role Code'].eq('1') & multi].sample(15, random_state=1).index)                                # no primary row
    j = out[out['Relationship Role Code'].eq('2')].sample(10, random_state=2).index
    out.loc[j, ['Relationship Role Code', 'Relationship Role Description', 'Ownership Category']] = ['1', 'Primary Owner', 'Primary Owner']  # two primary rows
    cdn = out.loc[out['Product Domain'].eq('Certificate of Deposit'), 'Account Number'].drop_duplicates().sample(5, random_state=3)
    ddn = out.loc[out['Product Domain'].eq('Demand Deposit'), 'Account Number'].drop_duplicates().sample(5, random_state=4)
    out['Account Number'] = out['Account Number'].replace(dict(zip(cdn.values, ddn.values)))                                         # numbers reused across domains
    out = pd.concat([out, out.sample(25, random_state=seed)], ignore_index=True)                                                        # exact duplicate rows
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
# ---- Colab: upload the file if it is not already present ----
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DEMO = str(FILE_PATH).strip().upper() == 'DEMO'
if IN_COLAB and not DEMO and not os.path.exists(FILE_PATH):
    from google.colab import files
    uploaded = files.upload()
    if uploaded: FILE_PATH = next(iter(uploaded))

def load(path, sheet=0, header=0):
    p = str(path).lower()
    if p.endswith('.csv'):     return pd.read_csv(path, header=header, dtype=str, keep_default_na=True, low_memory=False)
    if p.endswith('.parquet'): return pd.read_parquet(path)
    xls = pd.ExcelFile(path)
    print('Sheets:', xls.sheet_names)
    return pd.read_excel(xls, sheet_name=sheet, header=header)

def resolve_columns(file_cols):
    """Map every name in COLS to the file: exact match first, then a unique case-insensitive prefix."""
    norm = {c: re.sub(r'\s+', ' ', str(c)).strip().lower() for c in file_cols}
    found, used = {}, set()
    for key, names in COLS.items():
        for name in ([names] if isinstance(names, str) else names):
            t = re.sub(r'\s+', ' ', name).strip().lower()
            hit = [c for c in file_cols if norm[c] == t and c not in used] or [c for c in file_cols if norm[c].startswith(t) and c not in used]
            if hit:
                found[key] = hit[0]; used.add(hit[0]); break
    return found

if DEMO:
    raw = make_demo()
    display(Markdown('> **DEMO MODE** — synthetic data. Every number below is fake; it only shows that the notebook runs and what each output looks like.'))
else:
    raw = load(FILE_PATH, SHEET, HEADER)
raw.columns = [str(c).strip() for c in raw.columns]
print(f'Loaded {len(raw):,} rows × {raw.shape[1]} columns')

colmap = resolve_columns(list(raw.columns))
missing_req = [k for k in REQUIRED if k not in colmap]
if missing_req:
    raise KeyError(f'Required columns not found: {missing_req}. Fix their names in COLS. File columns: {list(raw.columns)}')
unknown = [c for c in raw.columns if c not in colmap.values()]
missing_opt = [k for k in COLS if k not in colmap]
if unknown:     print('⚠ Columns in the file that are not in COLS (not analyzed — add them to COLS to use them):', unknown)
if missing_opt: print('⚠ Optional columns not found (created blank, the views that need them will be empty):', missing_opt)
display(pd.DataFrame({'column in the file': pd.Series(colmap)}).rename_axis('name here'))

rel = raw[list(colmap.values())].rename(columns={v: k for k, v in colmap.items()}).copy()
for k in missing_opt: rel[k] = np.nan
rel = rel[list(COLS)]

## 2.1 Mixed types and column classification

Excel hands pandas a column as `object` whenever its cells do not share one Python type — a date column where some cells are real dates, some are text and some are Julian numbers, or an ID column where some codes are numbers and others text. Pandas does not complain; the column simply stops behaving like a date (or an ID) and every later comparison is silently wrong. So the first thing is to **see which columns carry more than one type, and which ones**.

In [ ]:
# ---- 1. Diagnosis: which columns mix Python types? ----
def type_profile(s):
    t = s.dropna().map(lambda v: 'datetime' if isinstance(v, (pd.Timestamp, dt.datetime, dt.date, np.datetime64)) else
                                 'int' if isinstance(v, (int, np.integer)) and not isinstance(v, bool) else
                                 'float' if isinstance(v, (float, np.floating)) else
                                 'text (date-like)' if isinstance(v, str) and re.match(r'^\s*\d{4}-\d{2}-\d{2}', v) else
                                 'text (digits)' if isinstance(v, str) and re.fullmatch(r'\s*-?\d+(\.0+)?\s*', v) else
                                 'text' if isinstance(v, str) else type(v).__name__)
    return t.value_counts()
obj_cols = [c for c in rel.columns if rel[c].dtype == object or pd.api.types.is_string_dtype(rel[c])]   # numeric / datetime dtypes are uniform by construction
prof = {c: type_profile(rel[c]) for c in obj_cols}
types_tbl = pd.DataFrame(prof).T.reindex(rel.columns).fillna(0).astype(int)
types_tbl.insert(0, 'pandas dtype', rel.dtypes.astype(str))
types_tbl['# types'] = (types_tbl.drop(columns='pandas dtype') > 0).sum(axis=1)
mixed_cols = types_tbl.index[types_tbl['# types'] > 1].tolist()
print(f'Columns with more than one Python type among their non-null values: {mixed_cols if mixed_cols else "none"}')
display(types_tbl.loc[mixed_cols] if mixed_cols else types_tbl.head(0))
for c in mixed_cols:
    ex = rel[c].dropna().groupby(rel[c].dropna().map(lambda v: type(v).__name__)).head(2)
    print(f'  {c}: examples → {[repr(v) for v in ex.tolist()[:6]]}')

**Rules applied below (type enforcement).** Nothing is guessed column by column: each column has a declared type.

| Type | Columns | Rule |
|---|---|---|
| ID / code (text) | `HouseholdID`, `CIF`, `AccountNumber`, `RoleCode`, `BranchCode`, `OfficerCode`, `StatusCode`, `Subtype`, `Segment` | Numbers become text without the trailing `.0` (`273130001.0` → `273130001`); blanks and placeholders → missing. |
| Text dimension | names, descriptions, types, country… | Trimmed; placeholders (`MISSING DATA`, `N/A`, `NULL`, `-`, …) → missing; country upper-cased. |
| Date | `BirthDate`, `CIFCreated`, `HHStartFile`, `Opened`, `Closed` | Real dates kept; text dates parsed; 7-digit numbers read as **Julian YYYYDDD**; 8-digit numbers as YYYYMMDD; `0` and sentinel dates (1900-01-01, 1899-12-30, 1753-01-01, 9999-…) → missing (counted). |
| Money | `Balance` | `$` and thousands separators removed, then numeric. |
| Value maps | `IorB`, `ForD`, `Domain`, `OwnerRow` (role description contains *Primary*) | Normalized with the rules in the configuration cell; the raw → normalized table is printed so nothing is mapped silently. |

In [ ]:
PLACEHOLDERS = {'MISSING DATA', 'MISSING', 'N/A', 'NA', 'N.A.', 'NULL', 'NONE', 'NAN', 'NAT', 'UNKNOWN', 'TBD', '-', '--', '?', '.', ''}
SENTINEL_DATES = {pd.Timestamp('1900-01-01'), pd.Timestamp('1899-12-30'), pd.Timestamp('1899-12-31'), pd.Timestamp('1753-01-01')}

def to_text(s):
    def f(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return np.nan
        if isinstance(v, (float, np.floating)) and float(v).is_integer(): v = int(v)
        v = str(v).strip()
        v = re.sub(r'^(-?\d+)\.0+$', r'\1', v)
        return np.nan if v.upper() in PLACEHOLDERS else v
    return s.map(f).astype(object)

def clean_dim(s, upper=False):
    def f(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return np.nan
        v = re.sub(r'\s+', ' ', str(v)).strip()
        if v.upper() in PLACEHOLDERS: return np.nan
        return v.upper() if upper else v
    return s.map(f).astype(object)

def parse_dates_mixed(s):
    """Dates that arrive as real dates, text, Julian YYYYDDD numbers or YYYYMMDD numbers → Timestamp. Returns (dates, n_sentinels)."""
    if pd.api.types.is_datetime64_any_dtype(s):                       # already a clean date column: nothing to parse
        out = pd.to_datetime(s, utc=True).dt.tz_localize(None).astype('datetime64[ns]')
        sentinel = out.isin(SENTINEL_DATES) | (out.dt.year < 1800) | (out.dt.year > 2200)
        return out.mask(sentinel), int(sentinel.sum())
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    nn = s[s.notna()].astype(object)
    is_dt = nn.map(lambda v: isinstance(v, (pd.Timestamp, dt.datetime, dt.date, np.datetime64))).astype(bool)
    if is_dt.any():
        out.loc[is_dt[is_dt].index] = pd.to_datetime(nn[is_dt], errors='coerce', utc=True).dt.tz_localize(None).values
    txt = pd.Series([re.sub(r'\.0+$', '', str(v).strip()) for v in nn[~is_dt]], index=nn[~is_dt].index, dtype=object)
    if txt.empty:
        sentinel = out.isin(SENTINEL_DATES) | (out.dt.year < 1800) | (out.dt.year > 2200)
        return out.mask(sentinel), int(sentinel.sum())
    digits = txt.str.fullmatch(r'\d+').fillna(False).astype(bool)
    num = txt[digits]
    n_numeric_placeholder = int((~num.str.len().isin([7, 8])).sum())     # 0, 1, 99… are placeholders, not dates
    jul = num[num.str.len() == 7]
    if len(jul):
        doy = pd.to_numeric(jul.str[4:], errors='coerce')
        d = pd.to_datetime(jul.str[:4], format='%Y', errors='coerce') + pd.to_timedelta(doy - 1, unit='D')
        out.loc[jul.index] = d.where(doy.between(1, 366)).values
    ymd = num[num.str.len() == 8]
    if len(ymd): out.loc[ymd.index] = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce').values
    other = txt[~digits]
    if len(other):
        try: parsed = pd.to_datetime(other, errors='coerce', format='mixed')
        except (TypeError, ValueError): parsed = pd.to_datetime(other, errors='coerce')
        out.loc[other.index] = pd.to_datetime(parsed, utc=True).dt.tz_localize(None).values
    sentinel = out.isin(SENTINEL_DATES) | (out.dt.year < 1800) | (out.dt.year > 2200)
    return out.mask(sentinel), int(sentinel.sum()) + n_numeric_placeholder

# ---- IDs and codes ----
ID_COLS = ['HouseholdID', 'CIF', 'AccountNumber', 'RoleCode', 'BranchCode', 'OfficerCode', 'StatusCode', 'Subtype', 'Segment']
for c in ID_COLS: rel[c] = to_text(rel[c])
# ---- Text dimensions ----
for c in ['IorB', 'ForD', 'SegmentDesc', 'RoleDesc', 'OwnershipCategory', 'DomainRaw', 'Product', 'AccountSegment', 'LargeRel', 'Branch', 'StatusDesc',
          'HouseholdName', 'CustomerName', 'AccountName', 'Officer']:
    rel[c] = clean_dim(rel[c])
rel['Country'] = clean_dim(rel['Country'], upper=True)
rel['LargeRel'] = rel['LargeRel'].map(lambda v: v.upper()[:1] if isinstance(v, str) else v)
# ---- Dates ----
date_report = {}
for src, dst in [('BirthRaw', 'BirthDate'), ('CIFCreated', 'CIFCreated'), ('HHStartFile', 'HHStartFile'), ('Opened', 'Opened'), ('Closed', 'Closed')]:
    before = rel[src].notna().sum()
    rel[dst], n_sent = parse_dates_mixed(rel[src])
    date_report[dst] = {'non-blank in file': int(before), 'parsed': int(rel[dst].notna().sum()), 'sentinel / impossible → blank': n_sent,
                        'unparseable → blank': int(before - rel[dst].notna().sum() - n_sent)}
rel = rel.drop(columns=['BirthRaw'])
print('Date parsing (rows):'); display(pd.DataFrame(date_report).T)
# ---- Money ----
b = rel['Balance']
if not pd.api.types.is_numeric_dtype(b):
    b = b.astype(str).str.replace(r'[,$\s]', '', regex=True).str.replace(r'^\((.*)\)$', r'-\1', regex=True)
rel['Balance'] = pd.to_numeric(b, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)

# ---- Value maps (raw → normalized, printed so nothing is mapped silently) ----
rel['IorB_raw'], rel['ForD_raw'] = rel['IorB'], rel['ForD']
rel['IorB']   = classify(rel['IorB'], IORB_RULES, default=np.nan)
rel['ForD']   = classify(rel['ForD'], FORD_RULES, default=np.nan)
rel['Domain'] = classify(rel['DomainRaw'], DOMAIN_RULES, default='Other')
# Owner row = the role description says PRIMARY. Ownership Category, then the role code, only fill blanks.
role_prim = rel['RoleDesc'].fillna('').astype(str).str.upper().str.contains(OWNER_ROLE_PATTERN, regex=True)
cat_prim = rel['OwnershipCategory'].fillna('').astype(str).str.upper().str.contains(OWNER_CATEGORY, regex=True)
rel['OwnerRow'] = np.where(rel['RoleDesc'].notna(), role_prim, np.where(rel['OwnershipCategory'].notna(), cat_prim, rel['RoleCode'].isin(OWNER_ROLE_CODES)))
both = rel['RoleDesc'].notna() & rel['OwnershipCategory'].notna()
if both.any():
    print(f'Owner rule = role description contains "{OWNER_ROLE_PATTERN}". Agreement with Ownership Category: {(role_prim[both] == cat_prim[both]).mean():.2%} of rows')
    dis = rel[both & (role_prim != cat_prim)]
    if len(dis):
        print(f'⚠ {len(dis):,} rows where they disagree (the role description wins):')
        display(dis.groupby(['RoleCode', 'RoleDesc', 'OwnershipCategory']).size().rename('rows').to_frame())
rel['OwnerRowSource'] = rel['RoleDesc'].fillna('(blank)')
for raw_c, new_c in [('IorB_raw', 'IorB'), ('ForD_raw', 'ForD'), ('DomainRaw', 'Domain'), ('OwnerRowSource', 'OwnerRow')]:
    t = rel.groupby([rel[raw_c].fillna('(blank)'), rel[new_c].astype(object).fillna('(unmapped)')]).size().rename('rows').reset_index()
    t.columns = ['value in file', 'mapped to', 'rows']
    print(f'\n{new_c}:'); display(t)
    if t['mapped to'].eq('(unmapped)').any() and new_c != 'OwnerRow':
        print(f'⚠ {new_c}: values not mapped → extend the rules in the configuration cell')
if rel['Domain'].eq('Other').any():
    print(f'⚠ {rel["Domain"].eq("Other").sum():,} rows with a product domain that is not DD / CD / LN — kept as "Other", excluded from DD / CD / LN views')

# ---- Flags derived from PII, then PII dropped ----
rel['OfficerInactive'] = rel['Officer'].fillna('').str.upper().str.contains(OFFICER_INACTIVE_PATTERN, regex=True)
rel['HHNameIsCustName'] = rel['HouseholdName'].fillna('').str.upper().eq(rel['CustomerName'].fillna('').str.upper()) & rel['HouseholdName'].notna()
rel = rel.drop(columns=PII_COLS + ['IorB_raw', 'ForD_raw', 'DomainRaw', 'OwnershipCategory', 'OwnerRowSource'])
rel['AccountKey'] = rel['Domain'].astype(str) + '-' + rel['AccountNumber'].astype(str)

# ---- Column classification (explicit, at the grain each column belongs to) ----
GRAIN = {'HouseholdID': 'Household', 'HHStartFile': 'Household',
         'CIF': 'CIF', 'IorB': 'CIF', 'ForD': 'CIF', 'CIFCreated': 'CIF', 'BirthDate': 'CIF', 'Segment': 'CIF', 'SegmentDesc': 'CIF', 'Country': 'CIF', 'HHNameIsCustName': 'CIF',
         'RoleCode': 'CIF × account', 'RoleDesc': 'CIF × account', 'OwnerRow': 'CIF × account',
         'AccountKey': 'Account', 'AccountNumber': 'Account', 'Domain': 'Account', 'Subtype': 'Account', 'Product': 'Account', 'AccountSegment': 'Account',
         'BranchCode': 'Account', 'Branch': 'Account', 'OfficerCode': 'Account', 'OfficerInactive': 'Account', 'StatusCode': 'Account', 'StatusDesc': 'Account',
         'Opened': 'Account', 'Closed': 'Account', 'Balance': 'Account', 'LargeRel': 'detected in §3.5'}
ROLE_OF = {'HouseholdID': 'id', 'CIF': 'id', 'AccountKey': 'id', 'AccountNumber': 'id', 'Balance': 'measure · balance ($)', 'OwnerRow': 'filter (owner row)',
           'OfficerInactive': 'flag (from officer name)', 'HHNameIsCustName': 'flag (from names)', 'LargeRel': 'flag'}
for c in ['HHStartFile', 'CIFCreated', 'BirthDate', 'Opened', 'Closed']: ROLE_OF[c] = 'date'
classification = pd.DataFrame({
    'grain': pd.Series(GRAIN), 'role': pd.Series({c: ROLE_OF.get(c, 'dimension') for c in rel.columns}),
    'dtype': rel.dtypes.astype(str), 'nulls % (rows)': (rel.isna().mean() * 100).round(1), 'distinct': rel.nunique(),
    'example': pd.Series({c: rel[c].dropna().iloc[0] if rel[c].notna().any() else None for c in rel.columns}),
}).loc[[c for c in rel.columns]]
display(classification)

# 3. Data model — is the file what we think it is?

Four questions decide whether any number later can be trusted:

1. **Row grain.** Is a row really one CIF × account × role? (exact duplicates, repeated keys, rows without a key)
2. **CIF → household.** Does every CIF belong to exactly one household, and which households are real groups vs. one-CIF fallbacks?
3. **Account key.** Is the account number unique across DD / CD / LN, and are the account attributes the same on every row of an account?
4. **Ownership.** Does every account have **exactly one primary owner**? This is the rule that keeps balances from being double-counted, so every exception is sized in dollars and repaired explicitly.

Nothing is analyzed in this section; it only fixes the grain so that §4 onward reads one value per entity.

## 3.1 Row grain

In [ ]:
n_rows_file = len(rel)
null_key = rel['CIF'].isna() | rel['AccountNumber'].isna()
print(f'Rows in the file: {n_rows_file:,}')
print(f'Rows without CIF or account number (dropped): {null_key.sum():,}')
rel = rel[~null_key]
n_exact = int(rel.duplicated().sum())
rel = rel.drop_duplicates().reset_index(drop=True)
print(f'Exact duplicate rows (dropped): {n_exact:,}')
KEY = ['CIF', 'AccountKey', 'RoleCode']
dupk = rel.duplicated(KEY, keep=False)
print(f'Rows sharing CIF × account × role but differing elsewhere: {dupk.sum():,} (first kept)')
if dupk.any():
    diff_cols = [c for c in rel.columns if c not in KEY and rel.loc[dupk].groupby(KEY)[c].nunique(dropna=False).gt(1).any()]
    print('  columns that differ inside those duplicates:', diff_cols)
    display(rel.loc[dupk].sort_values(KEY).head(6))
rel = rel.drop_duplicates(KEY).reset_index(drop=True)
multi_role = rel.groupby(['CIF', 'AccountKey'])['RoleCode'].nunique()
print(f'CIF × account pairs with more than one role (legitimate, e.g. owner and signer): {(multi_role > 1).sum():,} of {len(multi_role):,}')
print(f'\nGrain confirmed: {len(rel):,} rows = one per CIF × account × role')

## 3.2 CIF → household, and real households vs. single-CIF fallbacks

The household ID **falls back to the customer's own CIF** when the customer has no group (unassigned, temporary or employee groups). Those "households" have one member by construction and say nothing about relationships; the real groups are the ones with their own ID. Every household-level view separates the two (`HHType`).

In [ ]:
blank_hh = rel['HouseholdID'].isna()
if blank_hh.any():
    print(f'⚠ {rel.loc[blank_hh, "CIF"].nunique():,} CIFs with a blank household ID → their own CIF is used (the dictionary fallback rule)')
    rel.loc[blank_hh, 'HouseholdID'] = rel.loc[blank_hh, 'CIF']
hh_per_cif = rel.groupby('CIF')['HouseholdID'].nunique()
print(f'CIFs in more than one household: {(hh_per_cif > 1).sum():,} of {len(hh_per_cif):,}' + ('  → the most frequent household is kept' if (hh_per_cif > 1).any() else ''))
if (hh_per_cif > 1).any():
    display(rel[rel['CIF'].isin(hh_per_cif[hh_per_cif > 1].index[:3])][['CIF', 'HouseholdID', 'AccountKey', 'RoleCode']].sort_values('CIF'))
cif_hh = (rel.groupby(['CIF', 'HouseholdID']).size().reset_index(name='n').sort_values(['CIF', 'n'], ascending=[True, False])
             .drop_duplicates('CIF').set_index('CIF')['HouseholdID'])
rel['HouseholdID'] = rel['CIF'].map(cif_hh)

mem = cif_hh.reset_index().groupby('HouseholdID')['CIF'].agg(['size', 'first'])
hh_fallback = (mem['size'] == 1) & (mem.index.to_series() == mem['first'])
HHTYPE = pd.Series(np.where(hh_fallback, 'Single CIF', 'Real household'), index=mem.index)
cif_set = set(cif_hh.index)
odd = (~hh_fallback) & mem.index.to_series().isin(cif_set)
name_match = rel.drop_duplicates('CIF').assign(HHType=lambda d: d['HouseholdID'].map(HHTYPE)).groupby('HHType')['HHNameIsCustName'].mean() * 100

hh_tbl = pd.DataFrame({'Households': HHTYPE.value_counts(), 'CIFs': mem.groupby(HHTYPE)['size'].sum()}).reindex(ORDER['HHType'])
hh_tbl['% of households'] = hh_tbl['Households'] / hh_tbl['Households'].sum() * 100
hh_tbl['% of CIFs'] = hh_tbl['CIFs'] / hh_tbl['CIFs'].sum() * 100
hh_tbl['CIFs per household'] = hh_tbl['CIFs'] / hh_tbl['Households']
hh_tbl['% where group name = customer name'] = name_match.reindex(hh_tbl.index)
display(styled(hh_tbl, int_cols=['Households', 'CIFs'], pct_cols=['% of households', '% of CIFs', '% where group name = customer name'], two_dec=['CIFs per household']))
real_size = mem.loc[~hh_fallback, 'size']
print(f'Real households with a single member: {(real_size == 1).sum():,} · with 2: {(real_size == 2).sum():,} · 3–5: {real_size.between(3, 5).sum():,} · 6+: {(real_size >= 6).sum():,} · largest: {real_size.max() if len(real_size) else 0:,} CIFs')
print(f'Real household IDs that coincide with the CIF of a different customer: {odd.sum():,}' + ('  ⚠ ID spaces overlap' if odd.any() else ''))

## 3.3 Account key and account-level consistency

Account numbers come from three systems (DD, CD, LN). The key is **domain + number** (`AccountKey`). Every account-level attribute must be identical on every row of the account (one row per CIF on it); a conflict means the extract joined something at the wrong grain.

In [ ]:
num_dom = rel.groupby('AccountNumber')['Domain'].nunique()
print(f'Account numbers used in more than one product domain: {(num_dom > 1).sum():,}' + ('  → AccountKey = domain + number keeps them apart' if (num_dom > 1).any() else ''))
if (num_dom > 1).any():
    display(rel[rel['AccountNumber'].isin(num_dom[num_dom > 1].index[:3])].drop_duplicates('AccountKey')[['AccountNumber', 'Domain', 'Product', 'Opened', 'StatusDesc']])

ACCT_ATTRS = ['Domain', 'Subtype', 'Product', 'AccountSegment', 'BranchCode', 'Branch', 'OfficerCode', 'OfficerInactive', 'StatusCode', 'StatusDesc', 'Opened', 'Closed', 'Balance', 'LargeRel']
CIF_ATTRS  = ['HouseholdID', 'IorB', 'ForD', 'CIFCreated', 'BirthDate', 'Segment', 'SegmentDesc', 'Country', 'HHNameIsCustName', 'LargeRel']
n_acc, n_cif = rel['AccountKey'].nunique(), rel['CIF'].nunique()
def conflicts(key, cols, n):
    g = rel.groupby(key)
    return pd.DataFrame({'entities with > 1 value': {c: int((g[c].nunique() > 1).sum()) for c in cols}}).assign(**{'% of entities': lambda t: t['entities with > 1 value'] / n * 100})
cons = pd.concat({'Account attribute (per AccountKey)': conflicts('AccountKey', ACCT_ATTRS, n_acc), 'CIF attribute (per CIF)': conflicts('CIF', CIF_ATTRS, n_cif)})
display(styled(cons, int_cols=['entities with > 1 value'], pct_cols=['% of entities']))
print('Rule: the first non-blank value is kept for each entity; conflicts above are the size of what that rule decides.')

## 3.4 Ownership — exactly one primary owner per account

A *Relationship Role Description* that says **Primary** marks the CIF that **owns** the account. Balances and account counts go to that CIF only. Two kinds of exception are repaired, and both are sized in dollars:

- **No primary row** → the account would vanish from every balance. The CIF with the highest-priority role on it becomes the owner (joint owner first, then co-borrower, guarantor…), flagged `Imputed`.
- **Several primary rows** → the balance would be counted twice. One is kept (the earliest-created CIF); the others are treated as joint owners.

In [ ]:
rel['RoleFamilyRaw'] = classify(rel['RoleDesc'], ROLE_RULES, default='Other').fillna('Other')
acct_bal = rel.groupby('AccountKey')['Balance'].first()
n_own = rel[rel['OwnerRow']].groupby('AccountKey')['CIF'].nunique().reindex(acct_bal.index, fill_value=0)
own_band = n_own.clip(upper=3).map({0: '0 — no primary row', 1: '1 — one owner (OK)', 2: '2 primary rows', 3: '3+ primary rows'})
o_tbl = pd.DataFrame({'Accounts': own_band.value_counts(), 'Balance $': acct_bal.groupby(own_band).sum(), 'Abs. balance $': acct_bal.abs().groupby(own_band).sum()})
o_tbl = o_tbl.reindex([b for b in ['1 — one owner (OK)', '0 — no primary row', '2 primary rows', '3+ primary rows'] if b in o_tbl.index])
o_tbl['% of accounts'] = o_tbl['Accounts'] / o_tbl['Accounts'].sum() * 100
o_tbl['% of abs. balance'] = o_tbl['Abs. balance $'] / max(o_tbl['Abs. balance $'].sum(), 1) * 100
display(styled(o_tbl, int_cols=['Accounts'], money_cols=['Balance $', 'Abs. balance $'], pct_cols=['% of accounts', '% of abs. balance']))

# ---- Repair: pick one owner per account ----
created_first = rel.groupby('CIF')['CIFCreated'].min()
prio = {f: i for i, f in enumerate(ROLE_FAMILIES)}
cand = rel.assign(_own=rel['OwnerRow'].astype(int), _prio=rel['RoleFamilyRaw'].map(prio).fillna(99), _created=rel['CIF'].map(created_first))
cand = cand.sort_values(['AccountKey', '_own', '_prio', '_created', 'CIF'], ascending=[True, False, True, True, True], na_position='last')
chosen = cand.drop_duplicates('AccountKey')
OWNER = chosen.set_index('AccountKey')['CIF']
OWNER_SOURCE = pd.Series(np.where(n_own.reindex(OWNER.index) == 1, 'Primary row', np.where(n_own.reindex(OWNER.index) > 1, 'Several primary rows → one kept', 'No primary row → imputed')), index=OWNER.index)

rel['RoleFamily'] = rel['RoleFamilyRaw']
rel.loc[rel['OwnerRow'], 'RoleFamily'] = 'Joint / co-owner'          # primary rows that are not the chosen owner act as co-owners
rel.loc[chosen.index, 'RoleFamily'] = 'Primary owner'
rel['OwnerCIF'] = rel['AccountKey'].map(OWNER)
rel['IsOwner'] = rel['CIF'].eq(rel['OwnerCIF'])
imp = OWNER_SOURCE.eq('No primary row → imputed')
if imp.any():
    print('Imputed owners came from these roles:'); display(chosen.set_index('AccountKey').loc[imp[imp].index, 'RoleFamilyRaw'].value_counts().to_frame('accounts'))
print(f'Owner source: {OWNER_SOURCE.value_counts().to_dict()}')
print(f'Check: {rel[rel["RoleFamily"].eq("Primary owner")].groupby("AccountKey").size().max()} owner row per account at most · accounts without owner: {OWNER.isna().sum()}')

## 3.5 Base tables and the grain of `Large Relationship`

The base tables take one value per entity (first non-blank, conflicts sized in §3.3): **accounts** (one row per `AccountKey`, with its owner) and **CIFs** (one row per `CIF`). `Large Relationship` is not in the dictionary, so its grain is measured: the share of accounts, CIFs and households where it takes a single value. It is used at the coarsest level where it is constant.

In [ ]:
acct = rel.groupby('AccountKey', sort=False)[[c for c in ACCT_ATTRS if c != 'LargeRel'] + ['AccountNumber']].first()
acct['OwnerCIF'] = OWNER.reindex(acct.index)
acct['OwnerSource'] = OWNER_SOURCE.reindex(acct.index)
cif = rel.groupby('CIF', sort=False)[[c for c in CIF_ATTRS if c != 'LargeRel']].first()

lr = rel.dropna(subset=['LargeRel'])
if len(lr):
    def one_value_share(key, sub):
        """Share of entities with a single value, among entities that have at least 2 sub-entities (otherwise constancy is trivial)."""
        g = lr.groupby(key).agg(v=('LargeRel', 'nunique'), n=(sub, 'nunique'))
        g = g[g['n'] >= 2]
        return (g['v'] <= 1).mean() * 100 if len(g) else np.nan
    lr_grain = pd.Series({'Household': one_value_share('HouseholdID', 'CIF'), 'CIF': one_value_share('CIF', 'AccountKey'), 'Account': one_value_share('AccountKey', 'CIF')})
    display(lr_grain.to_frame('% with one value (entities with 2+ members / accounts / CIFs)').style.format(fmt_pct))
    lr_grain = lr_grain.fillna(100)
    LARGE_GRAIN = next((lvl for lvl in ['Household', 'CIF', 'Account'] if lr_grain[lvl] >= 99), 'Account')
    key = {'Household': 'HouseholdID', 'CIF': 'CIF', 'Account': 'AccountKey'}[LARGE_GRAIN]
    flag = lr.assign(_y=lr['LargeRel'].eq('Y')).groupby(key)['_y'].max()
    if LARGE_GRAIN == 'Account': acct['LargeRel'] = flag.reindex(acct.index).fillna(False)
    else:
        cif['LargeRel'] = (cif.index.to_series().map(flag) if LARGE_GRAIN == 'CIF' else cif['HouseholdID'].map(flag)).fillna(False).astype(bool)
    print(f'Large Relationship is constant at the {LARGE_GRAIN} level → used there. Y = {flag.mean():.1%} of {LARGE_GRAIN.lower()}s')
else:
    LARGE_GRAIN = None; print('Large Relationship not available')

levels = pd.DataFrame({
    'Entities': [cif['HouseholdID'].nunique(), (HHTYPE == 'Real household').sum(), len(cif), len(acct), len(rel)],
}, index=['Households (all)', '  of which real groups', 'CIFs', 'Accounts (DD + CD + LN, ever)', 'Relationship rows (CIF × account × role)'])
display(styled(levels, int_cols=['Entities']))
for f in FAM: print(f'  {FAM_LABEL[f]}: {acct["Domain"].eq(f).sum():,} accounts ever')

# 4. Data quality

What is missing, inconsistent or impossible — measured **at the grain of each column** (a birth date over CIFs, a close date over accounts). Nothing is deleted here; §5 applies the rules that come out of this section.

## 4.1 Nulls at the right grain

In [ ]:
GRAIN_KEY = {'Household': None, 'CIF': None, 'Account': None}
def null_rate(col, grain):
    if grain == 'Household': s = rel.drop_duplicates('HouseholdID')[col] if col in rel.columns else pd.Series(dtype=float)
    elif grain == 'CIF':     s = cif[col] if col in cif.columns else rel.drop_duplicates('CIF')[col]
    elif grain == 'Account': s = acct[col] if col in acct.columns else rel.drop_duplicates('AccountKey')[col]
    else:                    s = rel[col]
    return s.isna().mean() * 100, len(s), int(s.isna().sum())
q_cols = [('HHStartFile', 'Household'), ('IorB', 'CIF'), ('ForD', 'CIF'), ('CIFCreated', 'CIF'), ('BirthDate', 'CIF'), ('Segment', 'CIF'), ('SegmentDesc', 'CIF'), ('Country', 'CIF'),
          ('RoleCode', 'Row'), ('RoleDesc', 'Row'), ('Subtype', 'Account'), ('Product', 'Account'), ('AccountSegment', 'Account'), ('BranchCode', 'Account'), ('Branch', 'Account'),
          ('OfficerCode', 'Account'), ('StatusCode', 'Account'), ('StatusDesc', 'Account'), ('Opened', 'Account'), ('Closed', 'Account'), ('Balance', 'Account')]
rows = []
for c, g in q_cols:
    p_, n_, k_ = null_rate(c, g)
    rows.append({'column': c, 'grain': g, 'entities': n_, 'blank': k_, '% blank': p_})
quality = pd.DataFrame(rows).set_index('column')
BY_DESIGN = {'Closed': 'blank = account still open (read §4.2 instead)'}
quality['note'] = [BY_DESIGN.get(c, '') for c in quality.index]
for c, lab in [('BirthDate', 'birth / incorporation date'), ('CIFCreated', 'CIF creation date')]:
    for g in ORDER['IorB']:
        m = is_(cif['IorB'], g)
        quality.loc[f'{c} · {g}', ['grain', 'entities', 'blank', '% blank', 'note']] = ['CIF', int(m.sum()), int(cif.loc[m, c].isna().sum()), cif.loc[m, c].isna().mean() * 100, lab]
quality['flag'] = np.where(quality['note'].str.startswith('blank ='), 'by design', np.where(quality['% blank'] > 20, 'HIGH', np.where(quality['% blank'] > 5, 'medium', '')))
display(styled(quality, int_cols=['entities', 'blank'], pct_cols=['% blank']))

nz = quality[quality['% blank'] > 0].sort_values('% blank', ascending=False)
if len(nz):
    fig, ax = plt.subplots(figsize=(9, 0.6 + 0.36 * len(nz)))
    cols_ = [GRAY if f == 'by design' else (RED if v > 20 else BLUE) for f, v in zip(nz['flag'], nz['% blank'])]
    hbar(ax, [f'{i}  ({g})' for i, g in zip(nz.index, nz['grain'])], nz['% blank'].values, cols_, fmt=fmt_pct)
    style(ax, xlabel='% of entities at the column grain'); ax.xaxis.set_major_formatter(PCT0)
    finish(fig, 'Missing values per column, at the grain of the column', 'Gray = blank by design · red = more than 20% missing', name='04_nulls')

## 4.2 Account statuses and dates

The status code / description is mapped to four **status groups** with `STATUS_RULES`; the table shows every code so the mapping can be checked by eye. Then the status is tested against the dates: an open account should have no close date in the past, a closed account should have one. **Snapshot** = latest account-opening date in the file (≈ extract date) unless `SNAPSHOT_DATE` is set — close dates cannot be used for this because open CDs can carry their maturity date in the close field.

In [ ]:
SNAPSHOT = pd.Timestamp(SNAPSHOT_DATE) if SNAPSHOT_DATE else min(acct['Opened'].max(), pd.Timestamp.today().normalize())
print('Snapshot date:', SNAPSHOT.date(), '(set SNAPSHOT_DATE in §2 to override)')

acct['StatusGroup'] = classify(acct['StatusDesc'], STATUS_RULES, default='Other / unknown').fillna('Other / unknown')
acct['IsOpen'] = np.where(acct['StatusGroup'].isin(OPEN_GROUPS), True,
                 np.where(acct['StatusGroup'].isin(['Closed', 'Charged-off']), False, acct['Closed'].isna() | (acct['Closed'] > SNAPSHOT))).astype(bool)
st = (acct.assign(_n=1, _cd=acct['Closed'].notna(), _bal=acct['Balance'].fillna(0))
          .groupby([acct['StatusCode'].fillna('(blank)'), acct['StatusDesc'].fillna('(blank)'), 'StatusGroup'])
          .agg(Accounts=('_n', 'sum'), WithCloseDate=('_cd', 'mean'), Balance=('_bal', 'sum')))
st = st.join(pd.crosstab([acct['StatusCode'].fillna('(blank)'), acct['StatusDesc'].fillna('(blank)'), acct['StatusGroup']], acct['Domain']))
st['Open?'] = [g in OPEN_GROUPS for g in st.index.get_level_values('StatusGroup')]
st['WithCloseDate'] *= 100
st = st.rename(columns={'WithCloseDate': '% with close date', 'Balance': 'Balance $'})
display(styled(st, int_cols=['Accounts'] + [f for f in FAM if f in st.columns], pct_cols=['% with close date'], money_cols=['Balance $']))
if acct['StatusGroup'].eq('Other / unknown').any():
    print(f'⚠ {acct["StatusGroup"].eq("Other / unknown").sum():,} accounts with a status not covered by STATUS_RULES — open/closed decided by the close date. Extend STATUS_RULES.')

rows = []
def add(name, mask, note='', money_col=None):
    mask = pd.Series(mask, index=acct.index).fillna(False).astype(bool)
    r = {'check (accounts)': name, 'accounts': int(mask.sum()), '% of accounts': mask.mean() * 100, 'note': note}
    for f in FAM: r[f] = int((mask & acct['Domain'].eq(f)).sum())
    r['balance $'] = acct.loc[mask, 'Balance'].sum() if money_col else np.nan
    rows.append(r)
o, c_ = acct['IsOpen'], ~acct['IsOpen']
add('Open status · close date in the past', o & acct['Closed'].le(SNAPSHOT), 'status wins for "open today"; dates used for timing')
add('Open status · close date in the future', o & acct['Closed'].gt(SNAPSHOT), 'likely a maturity date (CD / LN) carried in the close field')
add('Closed status · no close date', c_ & acct['Closed'].isna(), 'closing year unknown → left out of the year-by-year views', money_col=True)
add('Closed status · close date after the snapshot', c_ & acct['Closed'].gt(SNAPSHOT))
add('Close date before open date', acct['Closed'] < acct['Opened'], 'life set to missing')
add('Opened after the snapshot', acct['Opened'] > SNAPSHOT)
add('Opened before 1950', acct['Opened'].dt.year < 1950, 'legacy / conversion dates?')
add('Open date missing', acct['Opened'].isna())
add('Closed account with a non-zero balance', c_ & acct['Balance'].fillna(0).ne(0), 'closure not completed or stale balance', money_col=True)
add('Open account with a zero balance', o & acct['Balance'].fillna(0).eq(0), 'normal for some DDAs; odd for CDs and loans')
add('DD balance negative (overdraft)', acct['Domain'].eq('DD') & acct['Balance'].lt(0), 'kept: real exposure', money_col=True)
add('CD balance negative', acct['Domain'].eq('CD') & acct['Balance'].lt(0), money_col=True)
add('LN balance negative (credit balance)', acct['Domain'].eq('LN') & acct['Balance'].lt(0), money_col=True)
add('Owner imputed (no primary row)', acct['OwnerSource'].eq('No primary row → imputed'), '§3.4', money_col=True)
add('Several primary rows (one kept)', acct['OwnerSource'].eq('Several primary rows → one kept'), '§3.4', money_col=True)
add('Officer marked retired / inactive', acct['OfficerInactive'] & o, 'open accounts only — unmanaged relationships', money_col=True)
acct_checks = pd.DataFrame(rows).set_index('check (accounts)')
display(styled(acct_checks, int_cols=['accounts'] + FAM, pct_cols=['% of accounts'], money_cols=['balance $']))
for c in ['Opened', 'Closed']:
    s = acct[c]; print(f'{c}: {s.min().date() if s.notna().any() else "-"} → {s.max().date() if s.notna().any() else "-"} · blank {s.isna().sum():,} · after snapshot {(s > SNAPSHOT).sum():,}')

## 4.3 Birth dates — age of people, age of companies

`BirthDate` means two different things: the **date of birth** of an individual and the **incorporation date** of a business. They are checked separately. Ages are computed at the snapshot here; former customers are re-aged at the date they left in §13.

In [ ]:
ind_c, biz_c = is_(cif['IorB'], 'Individual'), is_(cif['IorB'], 'Business')
age_raw = (SNAPSHOT - cif['BirthDate']).dt.days / 365.25
rows = []
def addc(name, mask, note=''):
    mask = pd.Series(mask, index=cif.index).fillna(False).astype(bool)
    rows.append({'check (CIFs)': name, 'CIFs': int(mask.sum()), '% of all CIFs': mask.mean() * 100,
                 '% of their type': mask.sum() / max((ind_c if name.startswith('Ind') else biz_c if name.startswith('Bus') else pd.Series(True, index=cif.index)).sum(), 1) * 100, 'note': note})
addc('Individuals · birth date missing', ind_c & cif['BirthDate'].isna())
addc('Individuals · under 18 (minor / custodial)', ind_c & age_raw.lt(18))
addc('Individuals · over 105 (implausible → blank)', ind_c & age_raw.gt(105), 'set to missing in §5')
addc('Individuals · born in the future', ind_c & age_raw.lt(0), 'set to missing in §5')
addc('Individuals · born after their CIF was created', ind_c & (cif['BirthDate'] > cif['CIFCreated']))
addc('Individuals · birth date = CIF creation date', ind_c & cif['BirthDate'].eq(cif['CIFCreated']), 'placeholder?')
addc('Business · incorporation date missing', biz_c & cif['BirthDate'].isna(), 'key-person age (§10) fills the gap')
addc('Business · incorporated in the future', biz_c & age_raw.lt(0), 'set to missing in §5')
addc('Business · incorporated > 150 years ago', biz_c & age_raw.gt(150), 'set to missing in §5')
addc('Business · incorporated after its CIF was created', biz_c & (cif['BirthDate'] > cif['CIFCreated']), 'possible for a new entity; check')
addc('Business · incorporation < 1 year', biz_c & age_raw.between(0, 1), 'plausible — new companies')
addc('CIF creation date missing', cif['CIFCreated'].isna(), 'relationship start falls back to the first account')
addc('CIF created after the snapshot', cif['CIFCreated'] > SNAPSHOT)
addc('IorB missing', cif['IorB'].isna()); addc('ForD missing', cif['ForD'].isna())
cif_checks = pd.DataFrame(rows).set_index('check (CIFs)')
display(styled(cif_checks, int_cols=['CIFs'], pct_cols=['% of all CIFs', '% of their type']))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.4))
for ax, m, ttl, colr, clip in [(axes[0], ind_c, 'Individuals · years since birth', BLUE, (0, 110)), (axes[1], biz_c, 'Business · years since incorporation', GREEN, (0, 80))]:
    s = age_raw[m].dropna()
    if len(s): sns.histplot(x=s.clip(*clip), bins=55, color=colr, edgecolor=INK['surface'], linewidth=0.3, ax=ax)
    style(ax, title=f'{ttl}  (n={len(s):,})', xlabel='years at the snapshot', ylabel='CIFs')
finish(fig, 'Raw ages before cleaning', 'Spikes at a single value usually mean a default date in the source', name='04_ages_raw')

## 4.4 Segment codes and Select — consistency

`Segment` looks like `[geo][type][tier]`: geo **I** = International, **D** = Domestic, **P** = Puerto Rico; type **P** = personal (Individual), **B** = Business; tier = 1…n. If the decoding is right the letters agree with `ForD` and `IorB`. The segment description says whether the **segment** is a Select segment; the account says whether the **product** is a Select product. They are two different definitions of Select and are cross-checked in §5.

In [ ]:
seg = cif['Segment'].astype(object)
parts = seg.str.extract(r'^([A-Z])([PB])(\d+)$')
cif['Segment_Geo']  = parts[0].map({'I': 'International', 'D': 'Domestic', 'P': 'Puerto Rico'})
cif['Segment_Type'] = parts[1].map({'P': 'Individual', 'B': 'Business'})
cif['Segment_Tier'] = pd.to_numeric(parts[2], errors='coerce')
decodable = cif['Segment_Type'].notna()
print(f'Segment decodable: {decodable.mean():.1%} of CIFs · Unassigned: {is_(seg, "Unassigned").sum():,} · other undecodable: {(~decodable & ~is_(seg, "Unassigned")).sum():,}')
undec = seg[~decodable].value_counts().head(10)
if len(undec): print('Undecodable segment values:', undec.to_dict())
exp_ford = cif['Segment_Geo'].map({'International': 'Foreign', 'Domestic': 'Domestic', 'Puerto Rico': 'Domestic'})
cif['Segment_vs_IorB_ok'] = cif['Segment_Type'].eq(cif['IorB']).where(decodable)
cif['Segment_vs_ForD_ok'] = exp_ford.eq(cif['ForD']).where(decodable)
print(f'Segment type vs IorB agree: {cif["Segment_vs_IorB_ok"].dropna().mean():.1%}   ·   Segment geo vs ForD agree: {cif["Segment_vs_ForD_ok"].dropna().mean():.1%}  (decodable CIFs only)')
print('\nSegment type × IorB'); display(pd.crosstab(cif['Segment_Type'].fillna('(undecodable)'), cif['IorB'].astype(str), margins=True))
print('Segment geo × ForD');    display(pd.crosstab(cif['Segment_Geo'].fillna('(undecodable)'), cif['ForD'].astype(str), margins=True))

cif['SelectSegment'] = np.where(cif['SegmentDesc'].fillna('').str.upper().str.contains(SELECT_PATTERN, regex=True), 'Select', 'Non-Select')
seg_cat = (cif.groupby([cif['Segment'].fillna('(blank)'), cif['SegmentDesc'].fillna('(blank)')])
              .agg(CIFs=('IorB', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100),
                   Select=('SelectSegment', lambda s: is_(s, 'Select').mean() * 100)).sort_values('CIFs', ascending=False))
seg_cat = seg_cat.rename(columns={'Individuals': '% Individual', 'Foreign': '% Foreign', 'Select': 'Select segment?'})
display(styled(seg_cat, int_cols=['CIFs'], pct_cols=['% Individual', '% Foreign', 'Select segment?'], gradient_cols=['CIFs']))
multi_desc = cif.groupby('Segment')['SegmentDesc'].nunique()
if (multi_desc > 1).any(): print(f'⚠ Segment codes with more than one description: {multi_desc[multi_desc > 1].to_dict()}')

acct_sel_vals = acct['AccountSegment'].fillna('(blank)').value_counts()
print('\nAccountSegment values:', acct_sel_vals.to_dict())
prod_sel = acct['Product'].fillna('').str.upper().str.contains(SELECT_PATTERN, regex=True)
seg_sel = acct['AccountSegment'].fillna('').str.upper().str.contains(SELECT_PATTERN, regex=True)
print(f'AccountSegment says Select: {seg_sel.sum():,} accounts · product description contains "{SELECT_PATTERN}": {prod_sel.sum():,} · agree: {(seg_sel == prod_sel).mean():.1%}')

## 4.5 Country vs. ForD, and ID hygiene

Domestic should mean United States / Puerto Rico (and US territories). Country disagreements are small but they sit in the country-risk view. IDs with unusual characters (parentheses, spaces, letters in numeric keys) break joins with other extracts, so they are counted.

In [ ]:
DOMESTIC_COUNTRIES = {'UNITED STATES', 'UNITED STATES OF AMERICA', 'USA', 'US', 'U.S.', 'U.S.A.', 'PUERTO RICO', 'PR',
                      'VIRGIN ISLANDS (U.S.)', 'U.S. VIRGIN ISLANDS', 'US VIRGIN ISLANDS', 'VIRGIN ISLANDS, U.S.', 'GUAM'}
ctry_kind = np.where(cif['Country'].isna(), '(missing)', np.where(cif['Country'].isin(DOMESTIC_COUNTRIES), 'US / PR', 'Other country'))
cif['Country_vs_ForD_ok'] = np.where(cif['Country'].isna(), np.nan,
                            np.where(is_(cif['ForD'], 'Domestic'), cif['Country'].isin(DOMESTIC_COUNTRIES), ~cif['Country'].isin(DOMESTIC_COUNTRIES)))
print('Country group × ForD (CIFs)'); display(pd.crosstab(pd.Series(ctry_kind, index=cif.index, name='Country group'), cif['ForD'].astype(str), margins=True))
bad = cif[pd.Series(cif['Country_vs_ForD_ok'], index=cif.index).eq(0)]
print(f'CIFs whose country disagrees with ForD: {len(bad):,}')
if len(bad): display(bad.groupby(['ForD', 'Country']).size().sort_values(ascending=False).head(10).to_frame('CIFs'))

id_rows = []
for name, s in [('CIF', cif.index.to_series()), ('HouseholdID', cif['HouseholdID'].drop_duplicates()), ('AccountNumber', acct['AccountNumber'])]:
    s = s.astype(str)
    id_rows.append({'ID': name, 'distinct': s.nunique(), 'non-alphanumeric chars': int(s.str.contains(r'[^A-Za-z0-9]').sum()),
                    'letters in it': int(s.str.contains(r'[A-Za-z]').sum()), 'min length': s.str.len().min(), 'max length': s.str.len().max(),
                    'examples of odd values': ', '.join(s[s.str.contains(r'[^A-Za-z0-9]')].head(4).tolist())})
display(pd.DataFrame(id_rows).set_index('ID'))

## 4.6 Outliers worth knowing by name

Extreme accounts and CIFs drive totals. They are listed so the reader can decide whether they are real (a corporate deposit, a fiduciary CIF on hundreds of accounts) or errors. IDs only — no names.

In [ ]:
open_a = acct[acct['IsOpen']]
for f in FAM:
    t = open_a[open_a['Domain'].eq(f)].nlargest(5, 'Balance')
    if len(t):
        print(f'\nTop 5 open {f} accounts by balance ({money(t["Balance"].sum())} = {t["Balance"].sum() / max(open_a.loc[open_a["Domain"].eq(f), "Balance"].clip(lower=0).sum(), 1):.1%} of open {f}):')
        display(t[['AccountNumber', 'OwnerCIF', 'Product', 'Branch', 'Opened', 'StatusDesc', 'Balance']].assign(OwnerIorB=t['OwnerCIF'].map(cif['IorB']), OwnerForD=t['OwnerCIF'].map(cif['ForD'])))
links_per_cif = rel.groupby('CIF').agg(accounts_any_role=('AccountKey', 'nunique'), as_owner=('IsOwner', 'sum'))
print('\nCIFs linked to the most accounts (any role):')
display(links_per_cif.nlargest(8, 'accounts_any_role').join(cif[['IorB', 'ForD', 'HouseholdID']]))
cifs_per_acct = rel.groupby('AccountKey')['CIF'].nunique()
print('Accounts with the most CIFs on them:')
display(cifs_per_acct.nlargest(5).to_frame('CIFs').join(acct[['Domain', 'Product', 'IsOpen', 'Balance']]))
hh_size_all = cif.groupby('HouseholdID').size()
print('Largest households (members):'); display(hh_size_all.nlargest(5).to_frame('CIFs'))

**Observations §3–§4** *(fill in after the run)*

- **Grain:** … rows, … exact duplicates, … repeated CIF × account × role keys; … CIFs in more than one household.
- **Households:** … % of households are single-CIF fallbacks holding … % of CIFs; real groups average … CIFs.
- **Ownership:** … accounts without a primary row ($… repaired), … with several ($… that would have been double-counted).
- **Statuses and dates:** … open accounts with a past close date, … closed accounts without a close date (excluded from the timing views), … closed accounts still carrying $… of balance.
- **Ages:** … % of individuals without a birth date, … % of businesses without an incorporation date; … implausible values blanked.
- **Segments and Select:** segment code agrees with IorB … % and ForD … %; … `Unassigned`. AccountSegment and product description agree … %.
- **Country / IDs:** … country–ForD conflicts; … IDs with odd characters.

# 5. Feature engineering — the four analysis tables

Everything later reads one of four tables, each at its own grain. Every derived field is created here, with its rule stated once.

| Table | Grain | What it adds |
|---|---|---|
| `rel` | CIF × account × role | `RoleFamily`, `IsOwner` (the chosen owner row), `OwnerCIF` |
| `acct` | Account | `StatusGroup`, `IsOpen`, `EndDate` (close date of closed accounts), `LifeYears`, `AccountSelect`, `Deposits` / `Loans` (open balance), roles present on it (`Has: …`), ownership `Structure`, and the **owner's** IorB / ForD / Select / household |
| `cif` | CIF | `Age` / `BusinessAge`, owned accounts ever / open by family, `Deposits`, `Loans`, `TotalBalance`, `ProductFamilies`, `ProductMix`, `SelectProduct` / `SelectSegment` / `Select`, non-owner roles (`RoleAcctsOpen`, `LinkedBusinesses`, `RoleProfile`), `Status`, `RelationshipStart`, `ChurnDate`, `LifetimeYears`, `ExitProfile`, `ExitType`, bands, `Region`, `ForeignSplit`, `ValueBand` |
| `hh` | Household | members by type and status, `Composition`, `HHForD`, `HHSelect`, `SizeBand`, `HHType`, balances and products summed over members, `Status`, start / end / lifetime, `ValueBand`, lead member |

| Feature | Rule |
|---|---|
| `EndDate` (account) | Close date of a closed account. Closed with no close date (or a close date after the snapshot) → **unknown**: the account counts as closed today but is left out of the year-by-year views. |
| `Deposits` / `Loans` / `Overdraft` | Balance of open accounts: DD + CD (positive balances only) / LN (positive only). An overdrawn DDA is credit exposure, kept apart in `Overdraft`, never netted against deposits. Closed accounts contribute 0 even if they carry a residual balance (§4.2). |
| `Structure` (account) | `Sole owner` (nobody else on it) → `Joint / co-borrowers` → `Guaranteed` → `Owner + signers / agents` (signer, attorney, fiduciary, beneficial owner) → `Owner + other roles` (beneficiary, mailing…) — first that applies. |
| `Status` (CIF) | Active customer / Connected (non-owner) / Former customer / Former connected — see Definitions. |
| `RoleProfile` (CIF) | `Owner only` · `Owner + other roles` (owns accounts and also signs / guarantees others) · `Non-owner roles only`. |
| `LinkedBusinesses` | For a CIF: number of distinct **businesses** on whose open accounts it holds a non-owner role. An individual with links is usually an owner or executive of those businesses. |
| `ExitProfile` / `ExitType` | What a former customer ever owned (`DD only`, `CD only`, `LN only`, `DD+CD`, `Deposits + LN`) and how it ended: `Involuntary (charge-off)` if an owned account was charged off in the 12 months before the customer left (an old charge-off followed by years of normal banking is not why they left), `Loan run-off` (LN only), `CD run-off` (CD only), else `Deposit relationship lost`. |
| `Composition` (household) | Over all members: `Individuals only` / `Businesses only` / `Mixed`. |
| `HHForD` | `Domestic` / `Foreign` if all members agree, else `Mixed`. |
| `Status` (household) | The best member status: `Active` (≥ 1 active customer) → `Connected only` → `Former` → `Former connected`. |
| `ValueBand` | Rank by `TotalBalance` among **active** customers (or active households): Top 1% / Next 9% / Next 40% / Bottom 50%. |
| `ForeignSplit` | `Domestic` / the dominant foreign country / `Other foreign` — when one country is most of the foreign book, "foreign" averages describe that country. |

## 5.1 Accounts

In [ ]:
acct['EndDate'] = acct['Closed'].where(~acct['IsOpen'])
acct.loc[acct['EndDate'] > SNAPSHOT, 'EndDate'] = pd.NaT
acct['EndUnknown'] = ~acct['IsOpen'] & acct['EndDate'].isna()
acct['LifeYears'] = ((acct['EndDate'].fillna(SNAPSHOT) - acct['Opened']).dt.days / 365.25).where(~acct['EndUnknown'] & acct['Opened'].notna())
acct.loc[acct['LifeYears'] < 0, 'LifeYears'] = np.nan
acct['OpenYear'] = acct['Opened'].dt.year
acct['CloseYear'] = acct['EndDate'].dt.year
sel_src = acct['AccountSegment'].where(acct['AccountSegment'].notna(), acct['Product']).fillna('')
acct['AccountSelect'] = np.where(sel_src.str.upper().str.contains(SELECT_PATTERN, regex=True), 'Select', 'Regular')
acct['OpenBalance'] = acct['Balance'].fillna(0).where(acct['IsOpen'], 0.0)
acct['Deposits'] = acct['OpenBalance'].clip(lower=0).where(acct['Domain'].isin(['DD', 'CD']), 0.0)   # an overdrawn DDA is credit exposure, not a negative deposit
acct['Overdraft'] = (-acct['OpenBalance'].clip(upper=0)).where(acct['Domain'].eq('DD'), 0.0)
acct['Loans'] = acct['OpenBalance'].clip(lower=0).where(acct['Domain'].eq('LN'), 0.0)             # a negative loan balance is a credit balance owed to the customer
acct['Dormant'] = acct['IsOpen'] & acct['StatusGroup'].eq('Dormant / restricted')
acct['ChargedOff'] = acct['StatusGroup'].eq('Charged-off')
acct['LifeBand'] = year_band(acct['LifeYears'])

# Who else is on each account (rows of CIFs other than the owner)
oth = rel[~rel['IsOwner']].join(acct[['IsOpen', 'EndDate', 'OpenBalance']], on='AccountKey')
fam_ct = pd.crosstab(oth['AccountKey'], oth['RoleFamily']).gt(0) if len(oth) else pd.DataFrame(index=acct.index)
acct['OtherCIFs'] = oth.groupby('AccountKey')['CIF'].nunique().reindex(acct.index).fillna(0).astype(int)
acct['CIFsOnAccount'] = acct['OtherCIFs'] + 1
for fam in ROLE_FAMILIES[1:]:
    acct[f'Has: {fam}'] = fam_ct[fam].reindex(acct.index).fillna(False).astype(bool) if fam in fam_ct.columns else False
def has(f): return acct[f'Has: {f}']
acct['Structure'] = pd.Series(np.select(
    [acct['OtherCIFs'].eq(0), has('Joint / co-owner') | has('Co-borrower'), has('Guarantor'),
     has('Authorized signer') | has('Power of attorney / agent') | has('Fiduciary / custodial') | has('Beneficial owner / controller')],
    STRUCT_ORDER[:4], STRUCT_ORDER[4]), index=acct.index, dtype=object)
print(f'Accounts: {len(acct):,} · open {acct["IsOpen"].sum():,} · closed {(~acct["IsOpen"]).sum():,} (close date unknown for {acct["EndUnknown"].sum():,})')

## 5.2 CIFs — what each customer is, owns, and how it is linked

In [ ]:
ind_c, biz_c = is_(cif['IorB'], 'Individual'), is_(cif['IorB'], 'Business')
age_raw = (SNAPSHOT - cif['BirthDate']).dt.days / 365.25
cif['Age'] = age_raw.where(ind_c & age_raw.between(0, 105))
cif['BusinessAge'] = age_raw.where(biz_c & age_raw.between(0, 150))
cif['HHType'] = cif['HouseholdID'].map(HHTYPE)

# ---- Country → Region; dominant foreign country ----
REGION = {
 'North America': ['UNITED STATES', 'UNITED STATES OF AMERICA', 'USA', 'US', 'PUERTO RICO', 'CANADA', 'VIRGIN ISLANDS (U.S.)', 'U.S. VIRGIN ISLANDS', 'VIRGIN ISLANDS, U.S.', 'GUAM'],
 'Mexico & Central America': ['MEXICO', 'GUATEMALA', 'EL SALVADOR', 'HONDURAS', 'NICARAGUA', 'COSTA RICA', 'PANAMA', 'BELIZE'],
 'South America': ['VENEZUELA', 'COLOMBIA', 'ECUADOR', 'PERU', 'BOLIVIA', 'CHILE', 'ARGENTINA', 'URUGUAY', 'PARAGUAY', 'BRAZIL', 'GUYANA', 'SURINAME', 'FRENCH GUIANA',
                   'VENEZUELA, BOLIVARIAN REPUBLIC OF', 'BOLIVARIAN REPUBLIC OF VENEZUELA', 'VENEZUELA (BOLIVARIAN REPUBLIC OF)', 'BOLIVIA, PLURINATIONAL STATE OF'],
 'Caribbean': ['DOMINICAN REPUBLIC', 'CUBA', 'HAITI', 'JAMAICA', 'BAHAMAS', 'BAHAMAS, THE', 'THE BAHAMAS', 'TRINIDAD AND TOBAGO', 'BARBADOS', 'CAYMAN ISLANDS',
               'BRITISH VIRGIN ISLANDS', 'VIRGIN ISLANDS, BRITISH', 'ARUBA', 'CURACAO', 'CURAÇAO', 'SINT MAARTEN', 'SAINT MARTIN', 'BERMUDA', 'TURKS AND CAICOS ISLANDS',
               'TURKS & CAICOS', 'TURKS AND CAICOS', 'ANTIGUA AND BARBUDA', 'ANTIGUA', 'SAINT KITTS AND NEVIS', 'ST. KITTS AND NEVIS', 'ST KITTS AND NEVIS', 'SAINT LUCIA',
               'ST. LUCIA', 'ST LUCIA', 'SAINT VINCENT AND THE GRENADINES', 'ST. VINCENT AND THE GRENADINES', 'GRENADA', 'DOMINICA', 'ANGUILLA', 'MONTSERRAT', 'BONAIRE',
               'NETHERLANDS ANTILLES', 'SAINT BARTHELEMY', 'ST. BARTHELEMY', 'MARTINIQUE', 'GUADELOUPE'],
 'Europe': ['SPAIN', 'ITALY', 'FRANCE', 'GERMANY', 'UNITED KINGDOM', 'PORTUGAL', 'SWITZERLAND', 'NETHERLANDS', 'BELGIUM', 'IRELAND', 'SWEDEN', 'NORWAY', 'DENMARK',
            'FINLAND', 'AUSTRIA', 'GREECE', 'POLAND', 'LUXEMBOURG', 'MONACO', 'ANDORRA', 'MALTA', 'CYPRUS', 'CZECH REPUBLIC', 'CZECHIA', 'CZECH REP.', 'HUNGARY', 'ROMANIA',
            'UKRAINE', 'RUSSIA', 'RUSSIAN FEDERATION', 'GIBRALTAR', 'ISLE OF MAN', 'JERSEY', 'GUERNSEY', 'LIECHTENSTEIN', 'CROATIA', 'SLOVAKIA', 'SLOVENIA', 'BULGARIA',
            'SERBIA', 'ESTONIA', 'LATVIA', 'LITHUANIA', 'ICELAND', 'HOLLAND', 'GREAT BRITAIN', 'ENGLAND', 'SCOTLAND', 'UK', 'U.K.', 'SAN MARINO', 'VATICAN CITY', 'ALBANIA',
            'MONTENEGRO', 'BOSNIA AND HERZEGOVINA', 'NORTH MACEDONIA', 'MOLDOVA', 'BELARUS', 'GEORGIA', 'ARMENIA', 'AZERBAIJAN'],
 'Middle East & Africa': ['ISRAEL', 'UNITED ARAB EMIRATES', 'SAUDI ARABIA', 'QATAR', 'KUWAIT', 'BAHRAIN', 'TURKEY', 'TÜRKIYE', 'LEBANON', 'JORDAN', 'EGYPT', 'MOROCCO',
                          'SOUTH AFRICA', 'NIGERIA', 'KENYA', 'GHANA', 'ANGOLA', 'TUNISIA', 'ALGERIA', 'IRAN', 'IRAQ', 'SYRIA', 'OMAN', 'YEMEN', 'ETHIOPIA', 'TANZANIA',
                          'UGANDA', 'SENEGAL', 'IVORY COAST', "COTE D'IVOIRE", 'CAMEROON', 'CONGO', 'ZIMBABWE', 'ZAMBIA', 'MOZAMBIQUE', 'BOTSWANA', 'NAMIBIA', 'MAURITIUS',
                          'SEYCHELLES', 'LIBYA', 'SUDAN', 'CABO VERDE', 'CAPE VERDE', 'EQUATORIAL GUINEA'],
 'Asia & Oceania': ['CHINA', 'HONG KONG', 'HONG KONG SAR', 'HONG KONG SAR CHINA', 'JAPAN', 'INDIA', 'SINGAPORE', 'SOUTH KOREA', 'KOREA, REPUBLIC OF', 'TAIWAN', 'PHILIPPINES',
                    'THAILAND', 'VIETNAM', 'MALAYSIA', 'INDONESIA', 'PAKISTAN', 'AUSTRALIA', 'NEW ZEALAND', 'MACAU', 'MACAO', 'KAZAKHSTAN', 'MARSHALL ISLANDS', 'VANUATU', 'SAMOA',
                    'COOK ISLANDS', 'FIJI', 'BANGLADESH', 'SRI LANKA', 'NEPAL'],
}
country_to_region = {c: r for r, lst in REGION.items() for c in lst}
def region_of(c):
    if not isinstance(c, str): return '(missing)'
    return country_to_region.get(c.strip().upper(), 'Other / unmapped')
cif['Region'] = cif['Country'].map(region_of)
REGION_ORDER = [r for r in list(REGION) + ['Other / unmapped', '(missing)'] if r in set(cif['Region'])]
unmapped = cif.loc[cif['Region'].eq('Other / unmapped'), 'Country'].value_counts()
if len(unmapped): print(f'⚠ {len(unmapped)} countries not in the region map (add them to REGION): {unmapped.head(20).to_dict()}')
fr_ctry = cif.loc[is_(cif['ForD'], 'Foreign'), 'Country'].value_counts()
DOM_COUNTRY = DOMINANT_COUNTRY or (fr_ctry.index[0] if len(fr_ctry) else 'NONE')
DOM_LABEL = str(DOM_COUNTRY).title()
cif['ForeignSplit'] = pd.Series(np.select([is_(cif['ForD'], 'Domestic'), is_(cif['ForD'], 'Foreign') & is_(cif['Country'], DOM_COUNTRY), is_(cif['ForD'], 'Foreign')],
                                          ['Domestic', DOM_LABEL, 'Other foreign'], 'n/a'), index=cif.index, dtype=object).replace('n/a', np.nan)
ORDER['ForeignSplit'] = ['Domestic', DOM_LABEL, 'Other foreign']; COLOR['ForeignSplit'] = {'Domestic': BLUE, DOM_LABEL: GREEN, 'Other foreign': GREEN_T}
print(f'Dominant foreign country: {DOM_LABEL} ({fr_ctry.iloc[0] / max(fr_ctry.sum(), 1):.0%} of foreign CIFs with a country)' if len(fr_ctry) else 'No foreign CIFs')

# ---- What each CIF owns (owner = the chosen primary owner of the account) ----
own = acct.dropna(subset=['OwnerCIF'])
def per_cif(s, fill=0): return s.reindex(cif.index).fillna(fill)
g = own.groupby('OwnerCIF')
cif['OwnedEver'] = per_cif(g.size()).astype(int)
cif['OwnedOpen'] = per_cif(g['IsOpen'].sum()).astype(int)
for f in FAM:
    m = own['Domain'].eq(f)
    cif[f'Ever_{f}'] = per_cif(own[m].groupby('OwnerCIF').size()).astype(int)
    cif[f'Open_{f}'] = per_cif(own[m & own['IsOpen']].groupby('OwnerCIF').size()).astype(int)
    cif[f'{f} balance'] = per_cif(own[m].groupby('OwnerCIF')['OpenBalance'].sum())
    cif[f'Has_{f}'] = cif[f'Open_{f}'].gt(0)
    cif[f'Had_{f}'] = cif[f'Ever_{f}'].gt(0)
cif['Deposits'] = per_cif(g['Deposits'].sum())
cif['Loans'] = per_cif(g['Loans'].sum())
cif['Overdraft'] = per_cif(g['Overdraft'].sum())
cif['TotalBalance'] = cif['Deposits'] + cif['Loans']
cif['SelectOpen'] = per_cif(own[own['IsOpen'] & own['AccountSelect'].eq('Select')].groupby('OwnerCIF').size()).astype(int)
cif['SelectEver'] = per_cif(own[own['AccountSelect'].eq('Select')].groupby('OwnerCIF').size()).astype(int)
cif['DormantOpen'] = per_cif(own[own['Dormant']].groupby('OwnerCIF').size()).astype(int)
cif['ChargedOffEver'] = per_cif(own[own['ChargedOff']].groupby('OwnerCIF').size()).astype(int)
cif['FirstOwnedOpened'] = g['Opened'].min().reindex(cif.index)
cif['LastOwnedClosed'] = own[~own['IsOpen']].groupby('OwnerCIF')['EndDate'].max().reindex(cif.index)
cif['MonthsSinceLastClose'] = (SNAPSHOT - cif['LastOwnedClosed']).dt.days / 30.44
cif['ProductFamilies'] = cif[[f'Has_{f}' for f in FAM]].sum(axis=1).astype(int)
cif['FamiliesEver'] = cif[[f'Had_{f}' for f in FAM]].sum(axis=1).astype(int)
cif['ProductMix'] = mix_label(cif, 'Open_')
cif['MixEver'] = mix_label(cif, 'Ever_')
cif['OnlyDormant'] = cif['OwnedOpen'].gt(0) & cif['DormantOpen'].eq(cif['OwnedOpen'])

# ---- Roles on accounts the CIF does not own ----
oth['OwnerIorB'] = oth['OwnerCIF'].map(cif['IorB'])
cif['RoleAccts'] = per_cif(oth.groupby('CIF')['AccountKey'].nunique()).astype(int)
cif['RoleAcctsOpen'] = per_cif(oth[oth['IsOpen']].groupby('CIF')['AccountKey'].nunique()).astype(int)
cif['LinkedOwners'] = per_cif(oth[oth['IsOpen']].groupby('CIF')['OwnerCIF'].nunique()).astype(int)
cif['LinkedBusinesses'] = per_cif(oth[oth['IsOpen'] & is_(oth['OwnerIorB'], 'Business')].groupby('CIF')['OwnerCIF'].nunique()).astype(int)
prio = {f: i for i, f in enumerate(ROLE_FAMILIES)}
mr = (oth.assign(_p=oth['RoleFamily'].map(prio).fillna(99), _closed=(~oth['IsOpen']).astype(int))
         .sort_values(['CIF', '_closed', '_p']).drop_duplicates('CIF').set_index('CIF')['RoleFamily'])
cif['MainOtherRole'] = mr.reindex(cif.index)
cif['RoleProfile'] = pd.Series(np.select([cif['OwnedEver'].gt(0) & cif['RoleAccts'].gt(0), cif['OwnedEver'].gt(0), cif['RoleAccts'].gt(0)],
                                         ['Owner + other roles', 'Owner only', 'Non-owner roles only'], 'None'), index=cif.index, dtype=object)

# ---- Status, relationship start / end, lifetime ----
cif['Status'] = pd.Series(np.select([cif['OwnedOpen'].gt(0), cif['RoleAcctsOpen'].gt(0), cif['OwnedEver'].gt(0)], CIF_STATUS[:3], CIF_STATUS[3]), index=cif.index, dtype=object)
link = rel[['CIF', 'AccountKey']].drop_duplicates().join(acct[['Opened', 'EndDate', 'IsOpen', 'EndUnknown']], on='AccountKey')
cif['FirstLinkOpened'] = link.groupby('CIF')['Opened'].min().reindex(cif.index)
cif['LastLinkEnd'] = link[~link['IsOpen']].groupby('CIF')['EndDate'].max().reindex(cif.index)
cif['AnyEndUnknown'] = link.groupby('CIF')['EndUnknown'].any().reindex(cif.index).fillna(False).astype(bool)
cif['RelationshipStart'] = cif[['CIFCreated', 'FirstLinkOpened']].min(axis=1)
former_any = cif['Status'].isin(['Former customer', 'Former connected'])
cif['ChurnDate'] = cif['LastLinkEnd'].where(former_any)
cif['ChurnDateUncertain'] = former_any & cif['AnyEndUnknown']
cif['EndDate'] = cif['ChurnDate']
end_for_life = cif['ChurnDate'].where(former_any, SNAPSHOT)
cif['LifetimeYears'] = (pd.to_datetime(end_for_life) - cif['RelationshipStart']).dt.days / 365.25
cif.loc[cif['LifetimeYears'] < 0, 'LifetimeYears'] = np.nan
cif['CohortYear'] = cif['RelationshipStart'].dt.year
cif['YearsSinceChurn'] = (SNAPSHOT - cif['ChurnDate']).dt.days / 365.25
cif['MonthsSinceChurn'] = (SNAPSHOT - cif['ChurnDate']).dt.days / 30.44
age_end = (pd.to_datetime(end_for_life) - cif['BirthDate']).dt.days / 365.25
cif['AgeAtEnd'] = age_end.where(ind_c & age_end.between(0, 110))
cif['BusinessAgeAtEnd'] = age_end.where(biz_c & age_end.between(0, 150))
cif['Churned'] = cif['Status'].eq('Former customer').astype(int)

# ---- How former customers ended ----
lastc = own[~own['IsOpen'] & own['EndDate'].notna()].sort_values('EndDate').drop_duplicates('OwnerCIF', keep='last').set_index('OwnerCIF')
cif['LastClosedDomain'] = lastc['Domain'].reindex(cif.index)
cif['LastClosedGroup'] = lastc['StatusGroup'].reindex(cif.index)
dd_, cd_, ln_ = cif['Had_DD'], cif['Had_CD'], cif['Had_LN']
EXIT_ORDER = ['DD only', 'CD only', 'LN only', 'DD+CD', 'Deposits + LN', 'No DD/CD/LN']
EXIT_COLOR = dict(zip(EXIT_ORDER, [BLUE, BLUE_T, GREEN, BLUE_M, GREEN_T, GRAY_L]))
cif['ExitProfile'] = pd.Series(np.select([ln_ & (dd_ | cd_), ln_, dd_ & cd_, dd_, cd_], ['Deposits + LN', 'LN only', 'DD+CD', 'DD only', 'CD only'], 'No DD/CD/LN'), index=cif.index, dtype=object)
EXIT_TYPES = ['Deposit relationship lost', 'Loan run-off', 'CD run-off', 'Involuntary (charge-off)']
EXIT_TYPE_COLOR = dict(zip(EXIT_TYPES, [RED, GREEN, BLUE_T, GRAY]))
co_ = own[own['ChargedOff']].assign(_churn=lambda d: d['OwnerCIF'].map(cif['LastLinkEnd']))
co_near = co_[co_['EndDate'].isna() | (co_['EndDate'] >= co_['_churn'] - pd.Timedelta(days=365))].groupby('OwnerCIF').size()
cif['ChargedOffAtExit'] = per_cif(co_near).gt(0) & cif['Status'].eq('Former customer')
cif['ExitType'] = pd.Series(np.select([cif['ChargedOffAtExit'], cif['ExitProfile'].eq('LN only'), cif['ExitProfile'].eq('CD only')],
                                      ['Involuntary (charge-off)', 'Loan run-off', 'CD run-off'], 'Deposit relationship lost'),
                            index=cif.index, dtype=object).where(cif['Status'].eq('Former customer'))

# ---- Select ----
cif['SelectProduct'] = np.where(np.where(cif['OwnedOpen'] > 0, cif['SelectOpen'], cif['SelectEver']) > 0, 'Select', 'Non-Select')
cif['Select'] = cif['SelectProduct'] if SELECT_BASIS == 'product' else cif['SelectSegment']

# ---- Bands and group labels ----
cif['AgeBand'] = age_band(cif['Age'])
cif['AgeAtEndBand'] = age_band(cif['AgeAtEnd'])
cif['BusinessAgeBand'] = year_band(cif['BusinessAge'])
cif['BusinessAgeAtEndBand'] = year_band(cif['BusinessAgeAtEnd'])
cif['TenureBand'] = year_band(cif['LifetimeYears'])
cif['DepositTier'] = tier_band(cif['Deposits'])
cif['OwnedEverBand'] = count_band(cif['OwnedEver'])
cif['OwnedOpenBand'] = count_band(cif['OwnedOpen'])
cif['Group2'] = make_label(cif, ['IorB', 'ForD'])
cif['Group3'] = make_label(cif, ['IorB', 'ForD', 'Select'])
GROUP2_ORDER = group_spec(['IorB', 'ForD'])[0]
GROUP3_ORDER = [a + SEP + b + SEP + c for a in ORDER['IorB'] for b in ORDER['ForD'] for c in ORDER['Select']]
act_c = cif['Status'].eq('Active customer')
cif['ValueBand'] = value_band(cif.loc[act_c, 'TotalBalance']).reindex(cif.index)
cif['TopDecile'] = cif['ValueBand'].isin(['Top 1%', 'Next 9%'])
for c_ in ['Deposits', 'Loans', 'TotalBalance']: cif[f'log_{c_}'] = np.log1p(cif[c_].clip(lower=0))
cif['HHMembers'] = cif['HouseholdID'].map(cif.groupby('HouseholdID').size())

# ---- Owner attributes on every account ----
for c_ in ['IorB', 'ForD', 'Select', 'SelectSegment', 'Group2', 'Group3', 'HouseholdID', 'HHType', 'Status', 'Segment', 'Region', 'ForeignSplit']:
    acct[c_] = acct['OwnerCIF'].map(cif[c_])
print(f'CIFs: {len(cif):,} · ' + ' · '.join(f'{k} {v:,}' for k, v in cif['Status'].value_counts().reindex(CIF_STATUS).fillna(0).astype(int).items()))

## 5.3 Households

In [ ]:
HH_STATUS = ['Active', 'Connected only', 'Former', 'Former connected']
COLOR['HHStatus'] = dict(zip(HH_STATUS, [BLUE, BLUE_T, RED, RED_T])); ORDER['HHStatus'] = HH_STATUS
c_ = cif.assign(_ind=ind_c.astype(int), _biz=biz_c.astype(int), _dom=is_(cif['ForD'], 'Domestic').astype(int), _for=is_(cif['ForD'], 'Foreign').astype(int),
                _act=act_c.astype(int), _conn=cif['Status'].eq('Connected (non-owner)').astype(int), _former=cif['Status'].eq('Former customer').astype(int),
                _owner=cif['OwnedEver'].gt(0).astype(int), _sel=(is_(cif['Select'], 'Select') & cif['OwnedEver'].gt(0)).astype(int),
                _rank=cif['Status'].map({s: i for i, s in enumerate(CIF_STATUS)}), _large=(cif['LargeRel'] if 'LargeRel' in cif.columns else pd.Series(False, index=cif.index)).fillna(False).astype(int))
g = c_.groupby('HouseholdID')
hh = pd.DataFrame({'Members': g.size(), 'Individuals': g['_ind'].sum(), 'Businesses': g['_biz'].sum(), 'DomesticMembers': g['_dom'].sum(), 'ForeignMembers': g['_for'].sum(),
                   'ActiveMembers': g['_act'].sum(), 'ConnectedMembers': g['_conn'].sum(), 'FormerMembers': g['_former'].sum(), 'Owners': g['_owner'].sum(),
                   'SelectMembers': g['_sel'].sum(), 'StatusRank': g['_rank'].min(), 'LargeRelAny': g['_large'].max(),
                   'Deposits': g['Deposits'].sum(), 'Loans': g['Loans'].sum(), 'OwnedOpen': g['OwnedOpen'].sum(), 'OwnedEver': g['OwnedEver'].sum(),
                   'RelationshipStart': g['RelationshipStart'].min(), 'LastMemberEnd': g['ChurnDate'].max()})
for f in FAM:
    hh[f'Open_{f}'] = g[f'Open_{f}'].sum(); hh[f'Ever_{f}'] = g[f'Ever_{f}'].sum(); hh[f'{f} balance'] = g[f'{f} balance'].sum()
hh['SelectOpen'] = g['SelectOpen'].sum()
if LARGE_GRAIN == 'Account':
    hh['LargeRelAny'] = acct[acct['IsOpen'] & acct['LargeRel'].fillna(False).astype(bool)].groupby('HouseholdID').size().reindex(hh.index).fillna(0).gt(0).astype(int)
hh['TotalBalance'] = hh['Deposits'] + hh['Loans']
hh['Status'] = hh['StatusRank'].map(dict(enumerate(HH_STATUS)))
hh['HHType'] = HHTYPE.reindex(hh.index)
hh['Composition'] = pd.Series(np.select([hh['Individuals'].gt(0) & hh['Businesses'].eq(0), hh['Businesses'].gt(0) & hh['Individuals'].eq(0), hh['Individuals'].gt(0) & hh['Businesses'].gt(0)],
                                        ORDER['Composition'], 'n/a'), index=hh.index).replace('n/a', np.nan)
hh['HHForD'] = pd.Series(np.select([hh['DomesticMembers'].gt(0) & hh['ForeignMembers'].eq(0), hh['ForeignMembers'].gt(0) & hh['DomesticMembers'].eq(0),
                                    hh['ForeignMembers'].gt(0) & hh['DomesticMembers'].gt(0)], ORDER['HHForD'], 'n/a'), index=hh.index).replace('n/a', np.nan)
hh['HHSelect'] = np.where(hh['SelectMembers'] > 0, 'Select', 'Non-Select')
hh['SizeBand'] = pd.cut(hh['Members'], [0, 1, 2, 3, 5, np.inf], labels=ORDER['SizeBand']).astype(object)
for f in FAM:
    hh[f'Has_{f}'] = hh[f'Open_{f}'].gt(0); hh[f'Had_{f}'] = hh[f'Ever_{f}'].gt(0)
hh['ProductFamilies'] = hh[[f'Has_{f}' for f in FAM]].sum(axis=1).astype(int)
hh['FamiliesEver'] = hh[[f'Had_{f}' for f in FAM]].sum(axis=1).astype(int)
hh['ProductMix'] = mix_label(hh, 'Open_')
hh['HHStartFile'] = rel.groupby('HouseholdID')['HHStartFile'].min().reindex(hh.index)
hh_former = hh['Status'].isin(['Former', 'Former connected'])
hh['EndDate'] = hh['LastMemberEnd'].where(hh_former)
hh['LifetimeYears'] = (pd.to_datetime(hh['EndDate'].where(hh_former, SNAPSHOT)) - hh['RelationshipStart']).dt.days / 365.25
hh.loc[hh['LifetimeYears'] < 0, 'LifetimeYears'] = np.nan
hh['CohortYear'] = hh['RelationshipStart'].dt.year
hh['Churned'] = hh['Status'].eq('Former').astype(int)
act_h = hh['Status'].eq('Active')
hh['ValueBand'] = value_band(hh.loc[act_h, 'TotalBalance']).reindex(hh.index)
hh['LeadCIF'] = cif.reset_index().sort_values(['HouseholdID', 'TotalBalance', 'OwnedEver'], ascending=[True, False, False]).drop_duplicates('HouseholdID').set_index('HouseholdID')['CIF'].reindex(hh.index)
hh['LeadIorB'] = hh['LeadCIF'].map(cif['IorB']); hh['LeadForD'] = hh['LeadCIF'].map(cif['ForD'])
hh['HHGroup'] = make_label(hh, ['Composition', 'HHForD'])
cif['HHStatus'] = cif['HouseholdID'].map(hh['Status'])
cif['HHComposition'] = cif['HouseholdID'].map(hh['Composition'])
acct['Composition'] = acct['HouseholdID'].map(hh['Composition'])

# File's household start vs recomputed start
d_start = (hh['HHStartFile'] - hh['RelationshipStart']).dt.days
print(f'Households: {len(hh):,} · ' + ' · '.join(f'{k} {v:,}' for k, v in hh['Status'].value_counts().reindex(HH_STATUS).fillna(0).astype(int).items()))
print(f'Household start in the file vs recomputed (min of member CIF creation / first account): same day {d_start.eq(0).mean():.1%} · '
      f'file later by > 1 year {d_start.gt(365).mean():.1%} · file earlier by > 1 year {d_start.lt(-365).mean():.1%} · file blank {hh["HHStartFile"].isna().mean():.1%}')

## 5.4 Select — product definition vs. segment definition

Two definitions of a Select customer exist in the file. `SELECT_BASIS` picks the one that drives every `Select` split; this table shows how much they disagree, overall and by IorB × ForD.

In [ ]:
a_ = cif[act_c]
print(f'\nSelect among active customers — by product: {is_(a_["SelectProduct"], "Select").mean():.1%} · by segment: {is_(a_["SelectSegment"], "Select").mean():.1%} · '
      f'agree: {(a_["SelectProduct"] == a_["SelectSegment"]).mean():.1%} · Cramér\'s V {cramers_v(a_["SelectProduct"], a_["SelectSegment"]):.2f} · SELECT_BASIS = "{SELECT_BASIS}"')
display(pd.crosstab(a_['SelectProduct'].rename('Select by product'), a_['SelectSegment'].rename('Select by segment'), margins=True))
print('By IorB × ForD (% of active customers):')
display(a_.groupby('Group2').agg(**{'by product': ('SelectProduct', lambda s: is_(s, 'Select').mean() * 100), 'by segment': ('SelectSegment', lambda s: is_(s, 'Select').mean() * 100),
                                   'agree': ('SelectProduct', lambda s: (s == a_.loc[s.index, 'SelectSegment']).mean() * 100)}).reindex(GROUP2_ORDER).style.format(fmt_pct))

In [ ]:
# ---- Labels used by the charts ----
HEADLINE_BAL = ['Deposits', 'Loans', 'TotalBalance']
PRODUCT_BAL  = ['DD balance', 'CD balance', 'LN balance']
LABEL = {'DD balance': 'DDA balance', 'CD balance': 'CD balance', 'LN balance': 'Loan balance', 'Deposits': 'Deposits (DD + CD)', 'Loans': 'Loans (LN)',
         'TotalBalance': 'Total balance (deposits + loans)', 'LifetimeYears': 'Relationship years', 'Age': 'Age (Individuals)',
         'BusinessAge': 'Business age (years since incorporation)', 'OwnedOpen': 'Open accounts owned', 'ProductFamilies': 'Product families held',
         'KP_Age': 'Key-person age (Business)', 'DecisionAge': 'Decision-maker age'}
print(f'\nTables ready · rel {rel.shape} · acct {acct.shape} · cif {cif.shape} · hh {hh.shape}')

## 5.5 The person behind each business, and each household's reference person

A business CIF carries the **incorporation** date, not the age of a person. The people behind it are the individuals with a role on **the business's own accounts**. The key person is chosen with an explicit rule (analyzed in §10):

1. Only individuals with a role in `KEY_PERSON_PRIORITY` on accounts the business owns (beneficial owner / controller → guarantor → joint owner → co-borrower → authorized signer → attorney → fiduciary). Beneficiaries and mailing roles never qualify.
2. Links on **open** accounts first; then the role priority; then a person with a known birth date; then the one on more of the business's accounts; then the eldest.
3. No role-linked individual → individuals of the **same real household** (`KEY_PERSON_USE_HOUSEHOLD`), flagged as `Household` source.

`DecisionAge` = own age for individuals, key-person age for businesses. The **household reference person** is the individual member holding the most money (ties → the eldest); a household with only businesses takes the key person of its largest business.

In [ ]:
indiv_set, bus_set = set(cif.index[ind_c]), set(cif.index[biz_c])
kprio = {f: i for i, f in enumerate(KEY_PERSON_PRIORITY)}
kp_src = oth[oth['OwnerCIF'].isin(bus_set) & oth['CIF'].isin(indiv_set) & oth['RoleFamily'].isin(KEY_PERSON_PRIORITY)]
pairs = (kp_src.assign(_p=kp_src['RoleFamily'].map(kprio), _open=kp_src['IsOpen'].astype(int))
               .groupby(['OwnerCIF', 'CIF']).agg(prio=('_p', 'min'), accounts=('AccountKey', 'nunique'), open_accounts=('_open', 'sum')).reset_index())
pairs['role'] = pairs['prio'].map(dict(enumerate(KEY_PERSON_PRIORITY))); pairs['source'] = 'Account role'
if KEY_PERSON_USE_HOUSEHOLD:
    real_m = cif['HHType'].eq('Real household')
    hb = cif.loc[biz_c & real_m, ['HouseholdID']].rename_axis('OwnerCIF').reset_index()
    hi = cif.loc[ind_c & real_m, ['HouseholdID']].rename_axis('CIF').reset_index()
    hp = hb.merge(hi, on='HouseholdID')[['OwnerCIF', 'CIF']]
    hp = hp.merge(pairs[['OwnerCIF', 'CIF']], how='left', indicator=True)
    hp = hp[hp['_merge'].eq('left_only')].drop(columns='_merge')
    pairs = pd.concat([pairs, hp.assign(prio=len(KEY_PERSON_PRIORITY), accounts=0, open_accounts=0, role='Same household', source='Household')], ignore_index=True)
pairs['Age'] = pairs['CIF'].map(cif['Age'])
best = (pairs.assign(_closed_only=pairs['open_accounts'].eq(0).astype(int), _noage=pairs['Age'].isna().astype(int))
             .sort_values(['OwnerCIF', '_closed_only', 'prio', '_noage', 'open_accounts', 'accounts', 'Age'], ascending=[True, True, True, True, False, False, False])
             .drop_duplicates('OwnerCIF').set_index('OwnerCIF'))
cif['KP_CIF'] = best['CIF'].reindex(cif.index)
cif['KP_Role'] = best['role'].reindex(cif.index)
cif['KP_Source'] = best['source'].reindex(cif.index)
cif['KP_Age'] = best['Age'].reindex(cif.index)
lp = pairs.groupby('OwnerCIF').agg(LinkedPersons=('CIF', 'nunique'), LinkedAgeMin=('Age', 'min'), LinkedAgeMedian=('Age', 'median'), LinkedAgeMax=('Age', 'max'))
for c_ in lp.columns: cif[c_] = lp[c_].reindex(cif.index)
cif.loc[biz_c, 'LinkedPersons'] = cif.loc[biz_c, 'LinkedPersons'].fillna(0)
cif['KP_Status'] = cif['KP_CIF'].map(cif['Status'])
cif['KP_IsCustomer'] = cif['KP_Status'].eq('Active customer').where(cif['KP_CIF'].notna())
KP_LABELS = KEY_PERSON_PRIORITY + ['Same household only', 'No individual linked']
cif['KP_Label'] = pd.Series(np.where(cif['KP_Source'].eq('Account role'), cif['KP_Role'], np.where(cif['KP_Source'].eq('Household'), 'Same household only', 'No individual linked')),
                            index=cif.index, dtype=object).where(biz_c)
cif['KP_AgeBand'] = age_band(cif['KP_Age'])
cif['KP_AgeAtEnd'] = (cif['KP_Age'] - cif['YearsSinceChurn'].fillna(0)).where(cif['KP_Age'].notna())
cif['DecisionAge'] = cif['Age'].where(ind_c, cif['KP_Age'].where(biz_c))
cif['DecisionAgeAtEnd'] = cif['AgeAtEnd'].where(ind_c, cif['KP_AgeAtEnd'].where(biz_c))
cif['DecisionAgeBand'] = age_band(cif['DecisionAge'])
cif['DecisionAgeAtEndBand'] = age_band(cif['DecisionAgeAtEnd'])

# ---- Household reference person ----
ref_i = cif[ind_c & cif['Age'].notna()].reset_index().sort_values(['HouseholdID', 'TotalBalance', 'Age'], ascending=[True, False, False]).drop_duplicates('HouseholdID').set_index('HouseholdID')
ref_b = cif[biz_c & cif['KP_Age'].notna()].reset_index().sort_values(['HouseholdID', 'TotalBalance'], ascending=[True, False]).drop_duplicates('HouseholdID').set_index('HouseholdID')
hh['RefAge'] = ref_i['Age'].reindex(hh.index)
hh['RefAgeSource'] = pd.Series(np.where(hh['RefAge'].notna(), 'Individual member', None), index=hh.index, dtype=object)
fill = hh['RefAge'].isna() & ref_b['KP_Age'].reindex(hh.index).notna()
hh.loc[fill, 'RefAge'] = ref_b['KP_Age'].reindex(hh.index)[fill]
hh.loc[fill, 'RefAgeSource'] = 'Key person of a member business'
hh['RefAgeBand'] = age_band(hh['RefAge'])

ab = cif[biz_c & act_c]
print(f'Active businesses: {len(ab):,} · with a role-linked individual: {ab["KP_Source"].eq("Account role").mean():.1%} · household only: {ab["KP_Source"].eq("Household").mean():.1%} · '
      f'none: {ab["KP_Source"].isna().mean():.1%} · key-person age known: {ab["KP_Age"].notna().mean():.1%}')
print(f'Decision-maker age known for {cif.loc[act_c, "DecisionAge"].notna().mean():.1%} of active customers (own age {cif.loc[act_c & ind_c, "Age"].notna().mean():.1%} of individuals)')
print(f'Household reference age known for {hh.loc[act_h, "RefAge"].notna().mean():.1%} of active households · sources: {hh.loc[act_h, "RefAgeSource"].value_counts().to_dict()}')

## 5.6 Analysis helpers

The profile tables (CIF level and household level) and the standard figures (demographics, product depth, balances) are defined once and reused for **every** granularity, so segments are always compared on identical axes.

In [ ]:
def _index_for(dims):
    return pd.Index(ORDER[dims[0]], name=dims[0]) if len(dims) == 1 else pd.MultiIndex.from_product([ORDER[c] for c in dims], names=dims)

def profile_table(d, dims):
    """CIF level. One row per segment of `dims`: size, ages, tenure, product depth, roles, deposits and loans."""
    g = d.groupby(dims, dropna=False)
    t = pd.DataFrame({'CIFs': g.size()})
    t['% CIFs'] = t['CIFs'] / t['CIFs'].sum() * 100
    t['Median age (Ind.)'] = g['Age'].median()
    t['Median business age'] = g['BusinessAge'].median()
    t['Median key-person age (Bus.)'] = g['KP_Age'].median()
    t['Median tenure (y)'] = g['LifetimeYears'].median()
    t['Avg open accts'] = g['OwnedOpen'].mean()
    t['Avg families'] = g['ProductFamilies'].mean()
    for f in FAM: t[f'% has {f}'] = g[f'Has_{f}'].mean() * 100
    if 'Select' not in dims: t['% Select'] = g['Select'].apply(lambda s: is_(s, 'Select').mean() * 100)
    if 'ForD' not in dims:   t['% Foreign'] = g['ForD'].apply(lambda s: is_(s, 'Foreign').mean() * 100)
    t['% in real household'] = g['HHType'].apply(lambda s: is_(s, 'Real household').mean() * 100)
    t['% also in other roles'] = g['RoleAcctsOpen'].apply(lambda s: s.gt(0).mean() * 100)
    t['Deposits $'] = g['Deposits'].sum(); t['% Deposits'] = t['Deposits $'] / max(t['Deposits $'].sum(), 1e-9) * 100
    t['Deposit holders'] = g['Deposits'].apply(lambda s: int((s > 0).sum()))
    t['Median deposits (holders)'] = g['Deposits'].apply(lambda s: s[s > 0].median())
    t['Loans $'] = g['Loans'].sum(); t['% Loans'] = t['Loans $'] / t['Loans $'].sum() * 100 if t['Loans $'].sum() else np.nan
    t['Loan holders'] = g['Loans'].apply(lambda s: int((s > 0).sum()))
    t['Median loans (holders)'] = g['Loans'].apply(lambda s: s[s > 0].median())
    return t.reindex(_index_for(dims)).dropna(subset=['CIFs'])

PCT_COLS   = ['% CIFs', '% HHs', '% Select', '% Foreign', '% Deposits', '% Loans', '% has DD', '% has CD', '% has LN', '% in real household', '% also in other roles',
              '% real household', '% with a business', '% with an individual']
MONEY_COLS = ['Deposits $', 'Median deposits (holders)', 'Loans $', 'Median loans (holders)']
ONE_DEC    = ['Median age (Ind.)', 'Median business age', 'Median key-person age (Bus.)', 'Median tenure (y)', 'Avg open accts', 'Avg families', 'Avg members',
              'Median reference age', 'Median relationship (y)']
def show_profile(t): display(styled(t, pct_cols=PCT_COLS, money_cols=MONEY_COLS, int_cols=['CIFs', 'HHs', 'Deposit holders', 'Loan holders'], one_dec=ONE_DEC,
                                    gradient_cols=['% CIFs', '% HHs', '% Deposits', '% Loans']))

def hh_profile_table(d, dims):
    """Household level. One row per segment of `dims`."""
    g = d.groupby(dims, dropna=False)
    t = pd.DataFrame({'HHs': g.size()})
    t['% HHs'] = t['HHs'] / t['HHs'].sum() * 100
    t['Avg members'] = g['Members'].mean()
    if 'HHType' not in dims: t['% real household'] = g['HHType'].apply(lambda s: is_(s, 'Real household').mean() * 100)
    t['% with a business'] = g['Businesses'].apply(lambda s: s.gt(0).mean() * 100)
    t['Median reference age'] = g['RefAge'].median()
    t['Median relationship (y)'] = g['LifetimeYears'].median()
    t['Avg open accts'] = g['OwnedOpen'].mean()
    t['Avg families'] = g['ProductFamilies'].mean()
    for f in FAM: t[f'% has {f}'] = g[f'Has_{f}'].mean() * 100
    if 'HHSelect' not in dims: t['% Select'] = g['HHSelect'].apply(lambda s: is_(s, 'Select').mean() * 100)
    t['Deposits $'] = g['Deposits'].sum(); t['% Deposits'] = t['Deposits $'] / max(t['Deposits $'].sum(), 1e-9) * 100
    t['Median deposits (holders)'] = g['Deposits'].apply(lambda s: s[s > 0].median())
    t['Loans $'] = g['Loans'].sum(); t['% Loans'] = t['Loans $'] / t['Loans $'].sum() * 100 if t['Loans $'].sum() else np.nan
    t['Median loans (holders)'] = g['Loans'].apply(lambda s: s[s > 0].median())
    return t.reindex(_index_for(dims)).dropna(subset=['HHs'])

def fig_demographics(d, dims, name=None):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    panels = [('Age', 'Age — Individuals', (0, 100), is_(d['IorB'], 'Individual')),
              ('BusinessAge', 'Business age (since incorporation)', (0, 60), is_(d['IorB'], 'Business')),
              ('KP_Age', 'Key-person age — Business', (0, 100), is_(d['IorB'], 'Business')),
              ('LifetimeYears', 'Relationship years — all', (0, 45), pd.Series(True, index=d.index))]
    nrows, ncols = (len(panels), len(facets)) if facet else (1, len(panels))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols if not facet else 6.5 * ncols, 3.3 * nrows), squeeze=False)
    for i, (col, ttl, clip, mask) in enumerate(panels):
        for j, fv in enumerate(facets):
            ax = axes[i, j] if facet else axes[0, i]
            m = mask & (is_(d[facet], fv) if fv is not None else True)
            kde_by(ax, d[m], col, '_grp', order, pal, clip=clip)
            style(ax, title=ttl + (f'  ·  {fv}' if fv is not None else ''), xlabel='years')
    finish(fig, f'Demographics & tenure by {" × ".join(dims)}', 'Density curves; dashed line = median. Key-person age = the individual behind each business (§10).', name=name)

def fig_counts(d, dims, name=None):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    panels = [('Open accounts owned per CIF', d['OwnedOpenBand'], COUNT_BANDS), ('Product families held (DD / CD / LN)', d['ProductFamilies'].astype(str), ['0', '1', '2', '3']),
              ('Product mix', d['ProductMix'], MIX_ORDER)]
    nrows, ncols = (len(panels), len(facets)) if facet else (1, len(panels))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.3 * ncols if not facet else 7 * ncols, 3.3 * nrows), squeeze=False)
    for i, (ttl, series, cat_order) in enumerate(panels):
        for j, fv in enumerate(facets):
            ax = axes[i, j] if facet else axes[0, i]
            m = is_(d[facet], fv) if fv is not None else pd.Series(True, index=d.index)
            if not m.any(): no_data(ax); continue
            ct = pd.crosstab(series[m], d.loc[m, '_grp'], normalize='columns') * 100
            ct = ct.reindex([c for c in cat_order if c in ct.index])
            grouped_bars(ax, ct, order, pal)
            style(ax, title=ttl + (f'  ·  {fv}' if fv is not None else ''), ylabel='% of CIFs in group', pct_y=True)
            if ttl == 'Product mix': ax.tick_params(axis='x', labelrotation=30)
    finish(fig, f'Product depth by {" × ".join(dims)}', 'Share of CIFs in each group · accounts the CIF owns (primary owner) and that are open', name=name)

def fig_balances(d, dims, cols, name=None, title='Balances', unit='CIFs'):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    nrows, ncols = len(facets), len(cols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 0.9 + 0.55 * len(order) * nrows + 1.6 * nrows), squeeze=False)
    for j, fv in enumerate(facets):
        for i, col in enumerate(cols):
            ax = axes[j, i]
            m = is_(d[facet], fv) if fv is not None else pd.Series(True, index=d.index)
            box_by(ax, d[m], col, '_grp', order, pal)
            style(ax, title=LABEL.get(col, col) + (f'  ·  {fv}' if fv is not None else ''))
            if i > 0: ax.set_yticklabels([])
    finish(fig, f'{title} by {" × ".join(dims)}', f'Among holders (balance > 0), log scale, no outliers drawn. Label = median; right = share of the {unit} in the group that hold it.', name=name)

def composition_fig(frames, dims, title, name=None, measures=(('CIFs', None), ('Deposits', 'Deposits'), ('Loans', 'Loans'))):
    """One panel per dim: share of count vs share of deposits vs share of loans (each bar sums to 100%)."""
    fig, axes = plt.subplots(1, len(dims), figsize=(5.2 * len(dims), 3.3), squeeze=False)
    for ax, dim in zip(axes[0], dims):
        order, pal = group_spec([dim])
        comp = pd.DataFrame({lab: (frames.groupby(dim).size() if col is None else frames.groupby(dim)[col].sum()) for lab, col in measures}).reindex(order).fillna(0).clip(lower=0)
        comp = (comp / comp.sum(axis=0).replace(0, np.nan) * 100).T
        stacked_100(ax, comp, order, pal)
        style(ax, title=f'By {dim}'); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=len(order))
    finish(fig, title, 'Each bar sums to 100%', name=name)
print('Helpers ready')

# 6. The bank today — three levels on one page

Households, customers and accounts are three different counts of the same bank. The KPIs put them side by side; the funnel shows how many entities exist at each level **ever** vs. **today**, and how they nest (CIFs per household, accounts per customer, CIFs per account).

In [ ]:
A = acct[acct['IsOpen']]; C = cif[act_c]; H = hh[act_h]
ind_a, biz_a = is_(C['IorB'], 'Individual'), is_(C['IorB'], 'Business')
real_H = is_(H['HHType'], 'Real household')
kpis = [
    ('Active households', f'{len(H):,}', f'{real_H.sum():,} real groups · {len(hh):,} ever', BLUE),
    ('Active customers (CIF)', f'{len(C):,}', f'{ind_a.sum():,} individuals · {biz_a.sum():,} businesses', BLUE),
    ('Connected CIFs', f'{cif["Status"].eq("Connected (non-owner)").sum():,}', 'roles on open accounts, own nothing', BLUE_T),
    ('Open accounts', f'{len(A):,}', ' · '.join(f'{f} {A["Domain"].eq(f).sum():,}' for f in FAM), BLUE),
    ('Deposits (DD + CD)', money(A['Deposits'].sum()), f'{C["Deposits"].gt(0).mean():.0%} of customers hold deposits', BLUE),
    ('Loans (LN)', money(A['Loans'].sum()), f'{C["Loans"].gt(0).mean():.0%} of customers borrow', GREEN),
    ('Individuals', f'{ind_a.mean():.0%}', f'{is_(C["ForD"], "Foreign")[ind_a].mean():.0%} of them foreign', BLUE),
    ('Foreign customers', f'{is_(C["ForD"], "Foreign").mean():.0%}', f'{DOM_LABEL}: {is_(C["ForeignSplit"], DOM_LABEL).sum() / max(is_(C["ForD"], "Foreign").sum(), 1):.0%} of them', GREEN),
    ('Select customers', f'{is_(C["Select"], "Select").mean():.0%}', f'by {SELECT_BASIS} · other definition {is_(C["SelectSegment" if SELECT_BASIS == "product" else "SelectProduct"], "Select").mean():.0%}', GREEN),
    ('Open accounts per customer', f'{C["OwnedOpen"].mean():.2f}', f'{C["ProductFamilies"].le(1).mean():.0%} hold one product family at most', BLUE),
    ('CIFs per real household', f'{H.loc[real_H, "Members"].mean():.1f}' if real_H.any() else '-', f'{is_(H["Composition"], "Mixed").sum():,} mix individuals and businesses', BLUE),
    ('Lost base', f'{cif["Status"].eq("Former customer").sum():,}', f'former customers · {hh["Status"].eq("Former").sum():,} former households', RED),
]
kpi_cards(kpis, 'The bank today at a glance', f'Snapshot {SNAPSHOT.date()} · households, customers (CIF) and accounts', name='06_kpis')

## 6.1 The granularity funnel — ever vs. today, and how the levels nest

In [ ]:
fun = pd.DataFrame([
    ('Households', 'All households in the file', len(hh)), ('Households', 'with an owner ever', int(hh['Owners'].gt(0).sum())), ('Households', 'Active today', int(act_h.sum())),
    ('Households', '  of which real groups', int((act_h & is_(hh['HHType'], 'Real household')).sum())),
    ('CIFs', 'All CIFs in the file', len(cif)), ('CIFs', 'Owned an account ever', int(cif['OwnedEver'].gt(0).sum())), ('CIFs', 'Active customers', int(act_c.sum())),
    ('CIFs', 'Connected (non-owner)', int(cif['Status'].eq('Connected (non-owner)').sum())), ('CIFs', 'Former customers', int(cif['Status'].eq('Former customer').sum())),
    ('CIFs', 'Former connected', int(cif['Status'].eq('Former connected').sum())),
    ('Accounts', 'Accounts ever (DD + CD + LN)', len(acct)), ('Accounts', 'Open today', int(acct['IsOpen'].sum())), ('Accounts', '  of which dormant / restricted', int(acct['Dormant'].sum())),
    ('Accounts', 'Closed', int((~acct['IsOpen']).sum())), ('Rows', 'CIF × account × role rows', len(rel)), ('Rows', 'on open accounts', int(rel['AccountKey'].map(acct['IsOpen']).sum())),
], columns=['Level', 'Measure', 'Count']).set_index(['Level', 'Measure'])
fun['% of level total'] = fun['Count'] / fun.groupby(level=0)['Count'].transform('first') * 100
display(styled(fun, int_cols=['Count'], pct_cols=['% of level total']))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
mb = count_band(H['Members'])
ct = pd.crosstab(mb, H['HHType'], normalize='columns').reindex(index=[b for b in COUNT_BANDS if b in set(mb)], columns=ORDER['HHType']) * 100
grouped_bars(axes[0], ct, ORDER['HHType'], COLOR['HHType'], fmt=lambda v: f'{v:.0f}%'); style(axes[0], title='CIFs per active household', ylabel='% of households', pct_y=True)
ct = pd.crosstab(C['OwnedOpenBand'], C['IorB'], normalize='columns').reindex(index=[b for b in COUNT_BANDS if b in set(C['OwnedOpenBand'])], columns=ORDER['IorB']) * 100
grouped_bars(axes[1], ct, ORDER['IorB'], COLOR['IorB'], fmt=lambda v: f'{v:.0f}%'); style(axes[1], title='Open accounts owned per active customer', ylabel='% of customers', pct_y=True)
cb = count_band(A['CIFsOnAccount'])
ct = pd.crosstab(cb, A['Domain'], normalize='columns').reindex(index=[b for b in COUNT_BANDS if b in set(cb)], columns=FAM) * 100
grouped_bars(axes[2], ct, FAM, COLOR['Domain'], fmt=lambda v: f'{v:.0f}%'); style(axes[2], title='CIFs on each open account (owner + others)', ylabel='% of accounts', pct_y=True)
finish(fig, 'How the three levels nest', 'Households → customers → accounts, today', name='06_nesting')

st_ct = pd.crosstab(cif['Status'], cif['IorB']).reindex(index=CIF_STATUS, columns=ORDER['IorB']).fillna(0).astype(int)
st_ct['All'] = st_ct.sum(axis=1); st_ct['% of CIFs'] = st_ct['All'] / st_ct['All'].sum() * 100
print('Every CIF in the file by status:'); display(styled(st_ct, int_cols=ORDER['IorB'] + ['All'], pct_cols=['% of CIFs']))

## 6.2 The 8-cell grid: IorB × ForD × Select (active customers)

Counts say who the customers are; deposits and loans say where the money is. The two rarely agree — that gap is the story.

In [ ]:
grid = profile_table(C, ['IorB', 'ForD', 'Select'])
show_profile(grid)
labels = [SEP.join(ix) for ix in grid.index]
order4, pal4 = group_spec(['IorB', 'ForD'])
colors = [pal4[SEP.join(ix[:2])] for ix in grid.index]
hatches = ['' if ix[2] == 'Select' else '///' for ix in grid.index]
fig, axes = plt.subplots(1, 3, figsize=(15, 0.8 + 0.5 * len(grid)), sharey=True)
for ax, col, ttl in zip(axes, ['% CIFs', '% Deposits', '% Loans'], ['Share of customers', 'Share of deposits', 'Share of loans']):
    hbar(ax, labels, grid[col].fillna(0).values, colors, fmt=fmt_pct, hatches=hatches)
    style(ax, title=ttl); ax.xaxis.set_major_formatter(PCT0)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, 'Who the customers are vs. where the money is', 'Color = IorB × ForD · hatched = Non-Select. Each panel sums to 100%.', name='06_grid')
composition_fig(C, ['IorB', 'ForD', 'Select'], 'Composition: customers vs. money', name='06_composition')

## 6.3 The same money, read at three levels

Deposits and loans by product domain and by who owns them (IorB × ForD of the owner), and by the kind of household the owner belongs to.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.6), gridspec_kw={'width_ratios': [1, 1.1, 1.1]})
bal_dom = A[A['OpenBalance'] > 0].pivot_table(index='Domain', columns='Group2', values='OpenBalance', aggfunc='sum').reindex(index=FAM, columns=order4).fillna(0)
stacked_bars(axes[0], bal_dom, order4, pal4, width=0.6); axes[0].yaxis.set_major_formatter(MONEY); axes[0].legend(fontsize=7.5)
for i, v in enumerate(bal_dom.sum(axis=1)): axes[0].text(i, v, money(v), ha='center', va='bottom', fontsize=8.5, color=INK['secondary'])
style(axes[0], title='Open balance by domain and owner (IorB × ForD)')
stacked_100(axes[1], bal_dom.div(bal_dom.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100, order4, pal4); style(axes[1], title='Share of each domain by owner')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=7.5)
comp_h = pd.DataFrame({'Households': H.groupby('Composition').size(), 'Deposits': H.groupby('Composition')['Deposits'].sum(), 'Loans': H.groupby('Composition')['Loans'].sum()}).reindex(ORDER['Composition']).fillna(0).clip(lower=0)
comp_h = (comp_h / comp_h.sum(axis=0).replace(0, np.nan) * 100).T
stacked_100(axes[2], comp_h, ORDER['Composition'], COLOR['Composition']); style(axes[2], title='Active households by composition'); axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3)
finish(fig, 'Where the money sits', f'Open accounts only · owner = primary owner · {money(A["Deposits"].sum())} deposits, {money(A["Loans"].sum())} loans', name='06_money_levels')
dom_tbl = A.groupby('Domain').agg(Accounts=('OpenBalance', 'size'), Balance=('OpenBalance', 'sum'), Median=('OpenBalance', lambda s: s[s > 0].median()),
                                   Owners=('OwnerCIF', 'nunique'), Select=('AccountSelect', lambda s: is_(s, 'Select').mean() * 100),
                                   Business=('IorB', lambda s: is_(s, 'Business').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100)).reindex(FAM)
dom_tbl = dom_tbl.rename(columns={'Balance': 'Balance $', 'Median': 'Median balance (> 0)', 'Owners': 'Owner CIFs', 'Select': '% Select products', 'Business': '% business-owned', 'Foreign': '% foreign-owned'})
display(styled(dom_tbl, int_cols=['Accounts', 'Owner CIFs'], money_cols=['Balance $', 'Median balance (> 0)'], pct_cols=['% Select products', '% business-owned', '% foreign-owned']))

**Observations §6** *(fill in after the run)*

- **Three levels:** … active households (… real groups), … active customers, … open accounts; … connected CIFs that own nothing.
- **Nesting:** a real household holds … CIFs; a customer owns … open accounts (… % just one); … % of open accounts carry more than one CIF.
- **Customers vs. money:** … % of customers are businesses and hold … % of deposits / … % of loans. Largest cell by count: …; by money: ….

# 7. Households — the relationship as the bank groups it

A household is the bank's own grouping of related CIFs (a family, an economic group, a company and its owners). Most "households" in the file are single-CIF fallbacks; the real groups are fewer but they are where relationships are multi-person and multi-product. Everything here is on **active households** unless stated; balances are the sum of what the members own.

## 7.1 Real groups vs. single-CIF fallbacks

In [ ]:
show_profile(hh_profile_table(H, ['HHType']))
st_hh = pd.crosstab(hh['HHType'], hh['Status']).reindex(index=ORDER['HHType'], columns=HH_STATUS).fillna(0).astype(int)
st_hh['All'] = st_hh.sum(axis=1)
print('All households by type and status:'); display(st_hh)

fig, axes = plt.subplots(1, 2, figsize=(15, 2.8), gridspec_kw={'width_ratios': [1, 1.15]})
pct_st = st_hh[HH_STATUS].div(st_hh['All'].replace(0, np.nan), axis=0) * 100
stacked_100(axes[0], pct_st, HH_STATUS, COLOR['HHStatus']); style(axes[0], title='Status of every household, by type'); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=4)
comp = pd.DataFrame({'Households': H.groupby('HHType').size(), 'Customers': H.groupby('HHType')['ActiveMembers'].sum(),
                     'Deposits': H.groupby('HHType')['Deposits'].sum(), 'Loans': H.groupby('HHType')['Loans'].sum()}).reindex(ORDER['HHType']).fillna(0).clip(lower=0)
stacked_100(axes[1], (comp / comp.sum(axis=0).replace(0, np.nan) * 100).T, ORDER['HHType'], COLOR['HHType']); style(axes[1], title='Active: share of households vs. customers vs. money')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
finish(fig, 'Real households vs. single-CIF fallbacks', 'A fallback "household" is one CIF whose household ID is its own CIF', name='07_hhtype')

## 7.2 Size — how many CIFs, and what size buys

Real households only (fallbacks have one member by construction). Median deposits come with 95% bootstrap intervals.

In [ ]:
R = H[is_(H['HHType'], 'Real household')]
if len(R):
    show_profile(hh_profile_table(R, ['SizeBand']))
    sizes = [s for s in ORDER['SizeBand'] if s in set(R['SizeBand'])]
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
    hbar(axes[0], sizes, R['SizeBand'].value_counts().reindex(sizes).values, [COLOR['SizeBand'][s] for s in sizes], fmt=lambda v: f'{v:,.0f} ({v / len(R):.0%})')
    style(axes[0], title='Real active households by number of CIFs', xlabel='households')
    med = R.assign(_all='All')
    m_, lo_, hi_ = median_table(med, 'Deposits', 'SizeBand', '_all', sizes, ['All'], holders=True)
    grouped_bars(axes[1], m_, ['All'], {'All': BLUE}, fmt=money, lo=lo_, hi=hi_); style(axes[1], title='Median household deposits (holders) · 95% CI', money_y=True); axes[1].get_legend().remove()
    axes[1].set_ylim(0, np.nanmax(hi_.values) * 1.2 if np.isfinite(np.nanmax(hi_.values)) else 1)
    fam = R.groupby('SizeBand')[['ProductFamilies']].mean().reindex(sizes)
    for f in FAM: fam[f'% {f}'] = R.groupby('SizeBand')[f'Has_{f}'].mean().reindex(sizes) * 100
    grouped_bars(axes[2], fam[[f'% {f}' for f in FAM]].rename(columns=lambda c: c[2:]), FAM, COLOR['Domain'], fmt=lambda v: f'{v:.0f}%')
    style(axes[2], title='Share of households holding each family', pct_y=True); axes[2].set_ylim(0, 115)
    finish(fig, 'Household size', 'Real active households · size = number of CIFs in the group (any status)', name='07_size')
else:
    print('No real households in the file')

## 7.3 Composition — individuals, businesses or both; domestic, foreign or both

`Mixed` households are where a family and its companies sit together: they are the closest thing the file has to an economic group. `HHForD = Mixed` means members on both sides of the residence line.

In [ ]:
for dims in [['Composition'], ['HHForD'], ['HHSelect'], ['Composition', 'HHForD'], ['HHType', 'Composition']]:
    display(Markdown(f'#### {" × ".join(dims)}'))
    show_profile(hh_profile_table(H, dims))
composition_fig(H, ['Composition', 'HHForD', 'HHSelect'], 'Active households: count vs. money', name='07_composition',
                measures=(('Households', None), ('Deposits', 'Deposits'), ('Loans', 'Loans')))
mixed = H[is_(H['Composition'], 'Mixed')]
if len(mixed):
    print(f'Mixed households: {len(mixed):,} · median {mixed["Individuals"].median():.0f} individuals + {mixed["Businesses"].median():.0f} businesses · '
          f'{money(mixed["Deposits"].sum())} deposits ({mixed["Deposits"].sum() / max(H["Deposits"].sum(), 1):.1%}) · {money(mixed["Loans"].sum())} loans ({mixed["Loans"].sum() / max(H["Loans"].sum(), 1):.1%})')
    mm = cif[cif['HouseholdID'].isin(mixed.index) & ind_c]
    print(f'  individuals in them: {len(mm):,} · {mm["Status"].eq("Active customer").mean():.0%} own an open account themselves · {mm["LinkedBusinesses"].gt(0).mean():.0%} hold a role on a business account')

## 7.4 Household depth vs. customer depth

A customer who owns one DDA can belong to a household that holds DDAs, CDs and loans. Reading depth at the CIF level understates the relationship; the gap is measured here on active customers who belong to **real** households with 2+ members.

In [ ]:
inm = C[is_(C['HHType'], 'Real household') & C['HHMembers'].ge(2)].copy()
if len(inm):
    inm['HHFamilies'] = inm['HouseholdID'].map(hh['ProductFamilies'])
    ct = pd.crosstab(inm['ProductFamilies'].rename('Families held by the customer'), inm['HHFamilies'].rename('Families held by the household'))
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={'width_ratios': [1, 1.2]})
    heat(axes[0], ct, annot=ct.map(fmt_int), cmap=CMAP_BLUE); style(axes[0], title='Active customers in multi-CIF real households', xlabel='families held by the household', ylabel='families held by the customer')
    rows_ = []
    for g_ in ORDER['IorB']:
        m = is_(inm['IorB'], g_)
        rows_.append({'IorB': g_, 'Customer level': inm.loc[m, 'ProductFamilies'].le(1).mean() * 100, 'Household level': inm.loc[m, 'HHFamilies'].le(1).mean() * 100})
    t = pd.DataFrame(rows_).set_index('IorB')
    grouped_bars(axes[1], t, ['Customer level', 'Household level'], {'Customer level': BLUE, 'Household level': GREEN}, fmt=lambda v: f'{v:.0f}%')
    style(axes[1], title='Share with at most one product family', pct_y=True); axes[1].set_ylim(0, 110)
    hidden = (inm['ProductFamilies'].le(1) & inm['HHFamilies'].ge(2)).mean() * 100
    finish(fig, 'Depth hidden in the household', f'{hidden:.0f}% of these customers look single-product but their household holds 2+ families', name='07_depth')
    print(f'Customers in multi-CIF real households: {len(inm):,} · single-family at CIF level {inm["ProductFamilies"].le(1).mean():.1%} vs. at household level {inm["HHFamilies"].le(1).mean():.1%}')
else:
    print('No active customers in multi-CIF real households')
mixes = pd.DataFrame({'Customers (CIF)': C['ProductMix'].value_counts(normalize=True) * 100, 'Households': H['ProductMix'].value_counts(normalize=True) * 100}).reindex(MIX_ORDER).fillna(0)
fig, ax = plt.subplots(figsize=(11, 3.2))
grouped_bars(ax, mixes, ['Customers (CIF)', 'Households'], {'Customers (CIF)': BLUE, 'Households': GREEN}, fmt=lambda v: f'{v:.0f}%')
style(ax, title='Product mix: customers vs. households (active)', pct_y=True)
finish(fig, 'Product mix at two levels', 'Families of open owned accounts', name='07_mix_levels')

## 7.5 Concentration — households vs. customers, and the largest relationships

Lorenz curves among holders at both levels: if households are more concentrated than customers, the top relationships are groups of CIFs that look smaller one by one. Then the value pyramid at household level and the 20 largest active households (IDs only).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, col in zip(axes, ['Deposits', 'Loans']):
    for lab, s, colr, ls in [('Customers (CIF)', C[col], BLUE, '-'), ('Households', H[col], GREEN, '-'), ('Real households only', R[col] if len(R) else pd.Series(dtype=float), GREEN_T, '--')]:
        s = s[s > 0]
        x, y = lorenz(s)
        if x is not None: ax.plot(x, y, color=colr, lw=2, ls=ls, label=f'{lab} (n={len(s):,} · Gini {gini(s):.2f} · {holders_for(s, 80):.1f}% hold 80%)')
    ax.axhline(80, color=INK['grid'], lw=1); ax.set_xlim(0, 100); ax.set_ylim(0, 101); ax.xaxis.set_major_formatter(PCT0); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=LABEL[col], xlabel='% of holders (largest first)', ylabel=f'cumulative % of {col.lower()}'); ax.legend(loc='lower right', fontsize=7.8)
finish(fig, 'Concentration among holders — customers vs. households', name='07_lorenz')

VB = ORDER['ValueBand']; vb_colors = dict(zip(VB, [CMAP_BLUE(x) for x in [1.0, 0.72, 0.42, 0.16]]))
vbh = H.groupby('ValueBand').agg(HHs=('Members', 'size'), Members=('Members', 'mean'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), Total=('TotalBalance', 'sum'),
                                 Entry=('TotalBalance', 'min'), Median=('TotalBalance', 'median'), Real=('HHType', lambda s: is_(s, 'Real household').mean() * 100),
                                 Mixed=('Composition', lambda s: is_(s, 'Mixed').mean() * 100), Families=('ProductFamilies', 'mean')).reindex(VB)
for c_ in ['HHs', 'Deposits', 'Loans', 'Total']: vbh[f'% {c_}'] = vbh[c_] / max(vbh[c_].sum(), 1e-9) * 100
vbh = vbh.rename(columns={'Members': 'Avg members', 'Entry': 'Entry threshold', 'Median': 'Median total balance', 'Real': '% real household', 'Mixed': '% mixed', 'Families': 'Avg families'})
display(styled(vbh, int_cols=['HHs'], money_cols=['Deposits', 'Loans', 'Total', 'Entry threshold', 'Median total balance'],
               pct_cols=['% HHs', '% Deposits', '% Loans', '% Total', '% real household', '% mixed'], one_dec=['Avg members', 'Avg families']))
fig, ax = plt.subplots(figsize=(11, 2.8))
stacked_100(ax, vbh[['% HHs', '% Deposits', '% Loans']].rename(columns=lambda c: c[2:]).T, VB, vb_colors); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=4)
finish(fig, 'Household value pyramid', 'Active households ranked by total balance (deposits + loans of all members)', name='07_pyramid')

top_h = H.nlargest(20, 'TotalBalance')[['HHType', 'Members', 'Individuals', 'Businesses', 'Composition', 'HHForD', 'HHSelect', 'OwnedOpen', 'ProductMix', 'Deposits', 'Loans', 'TotalBalance', 'RefAge']]
print(f'Top 20 active households = {top_h["TotalBalance"].sum() / max(H["TotalBalance"].clip(lower=0).sum(), 1):.1%} of total balance:')
display(styled(top_h, money_cols=['Deposits', 'Loans', 'TotalBalance'], int_cols=['Members', 'Individuals', 'Businesses', 'OwnedOpen'], one_dec=['RefAge']))

**Observations §7** *(fill in after the run)*

- **Real vs. fallback:** … % of active households are real groups; they hold … % of customers and … % of deposits / … % of loans.
- **Size:** real households hold … CIFs (median); deposits rise from … (2 CIFs) to … (6+).
- **Composition:** … mixed households (family + companies) hold … % of deposits and … % of loans; … % of the individuals in them bank with us personally.
- **Depth:** … % of customers in multi-CIF households look single-product but their household holds 2+ families.
- **Concentration:** top 1% of households hold … % of total balance (customers: … %).

# 8. Customers (CIF) — every granularity × every feature

Active customers only (CIFs that own at least one open account). Seven granularities run through the same profile table and the same figures:

`IorB` · `ForD` · `Select` · `IorB × ForD` · `IorB × Select` · `ForD × Select` · `IorB × ForD × Select`

For each: **(1)** profile table, **(2)** demographics & tenure (with the key-person age of businesses), **(3)** product depth, **(4)** headline balances, **(5)** product balances. Color encodes up to two dimensions; the third becomes side-by-side panels.

In [ ]:
GRANULARITIES = [['IorB'], ['ForD'], ['Select'], ['IorB', 'ForD'], ['IorB', 'Select'], ['ForD', 'Select'], ['IorB', 'ForD', 'Select']]
for dims in GRANULARITIES:
    tag = ' × '.join(dims); slug = '08_' + '_'.join(dims)
    display(Markdown(f'### {tag}'))
    show_profile(profile_table(C, dims))
    fig_demographics(C, dims, name=f'{slug}_demographics')
    fig_counts(C, dims, name=f'{slug}_products')
    fig_balances(C, dims, HEADLINE_BAL, name=f'{slug}_balances', title='Headline balances')
    fig_balances(C, dims, PRODUCT_BAL, name=f'{slug}_product_balances', title='Product balances')

## 8.1 Age of individual customers

Bands make the comparison across segments easy; medians are on every curve. Business age and the age of the person behind each business are in §10.

In [ ]:
d_ind = C[is_(C['IorB'], 'Individual')]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, dims in zip(axes, [['ForD'], ['Select'], ['ForD', 'Select']]):
    order, pal = group_spec(dims)
    kde_by(ax, d_ind.assign(_grp=make_label(d_ind, dims)), 'Age', '_grp', order, pal, clip=(0, 100))
    style(ax, title='Individuals · age by ' + ' × '.join(dims), xlabel='years')
finish(fig, 'Age distribution of individual customers', 'Density; dashed = median', name='08_age_ind')

grp = make_label(d_ind, ['ForD', 'Select']); order4s, _ = group_spec(['ForD', 'Select'])
ct = pd.crosstab(grp, d_ind['AgeBand'], normalize='index').reindex(index=order4s, columns=AGE_BANDS).fillna(0) * 100
fig, ax = plt.subplots(figsize=(11, 3.2))
stacked_100(ax, ct, AGE_BANDS, dict(zip(AGE_BANDS, BLUES_7)), min_label=6); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=7, title='Age band', title_fontsize=8.5)
finish(fig, 'Individuals · age-band mix by ForD × Select', name='08_age_bands')
print('Individuals by age band (counts):')
display(pd.crosstab(d_ind['AgeBand'], [d_ind['ForD'], d_ind['Select']], margins=True, margins_name='All').reindex(AGE_BANDS + ['All']))

fig, axes = plt.subplots(1, 4, figsize=(19, 3.5))
order2, pal2 = group_spec(['ForD'])
med, lo, hi = median_table(d_ind, 'Deposits', 'AgeBand', 'ForD', AGE_BANDS, order2, holders=True)
grouped_bars(axes[0], med, order2, pal2, fmt=money, lo=lo, hi=hi); style(axes[0], title='Median deposits among holders (95% CI)', money_y=True)
axes[0].set_ylim(0, np.nanmax(hi.values) * 1.18 if np.isfinite(np.nanmax(hi.values)) else 1)
sel = d_ind.groupby(['AgeBand', 'ForD'])['Select'].apply(lambda s: is_(s, 'Select').mean() * 100).unstack().reindex(index=AGE_BANDS, columns=order2)
grouped_bars(axes[1], sel, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[1], title='Share Select', pct_y=True); axes[1].set_ylim(0, 105)
ln = d_ind.groupby(['AgeBand', 'ForD'])['Has_LN'].mean().unstack().reindex(index=AGE_BANDS, columns=order2) * 100
grouped_bars(axes[2], ln, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[2], title='Share with a loan', pct_y=True); axes[2].set_ylim(0, max(np.nanmax(ln.values) * 1.3, 5) if ln.notna().any().any() else 5)
lb = d_ind.groupby(['AgeBand', 'ForD'])['LinkedBusinesses'].apply(lambda s: s.gt(0).mean() * 100).unstack().reindex(index=AGE_BANDS, columns=order2)
grouped_bars(axes[3], lb, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[3], title='Share also behind a business account', pct_y=True)
axes[3].set_ylim(0, max(np.nanmax(lb.values) * 1.3, 5) if lb.notna().any().any() else 5)
finish(fig, 'Individuals · what changes with age', 'By age band and ForD · error bars = 95% bootstrap CI of the median', name='08_age_money')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (col, bands, ttl, sub) in zip(axes, [('AgeBand', AGE_BANDS, 'Individuals · age', d_ind), ('BusinessAgeBand', YEAR_BANDS, 'Business · years since incorporation', C[is_(C['IorB'], 'Business')])]):
    ct = pd.crosstab(sub[col], sub['ForD']).reindex(index=bands, columns=ORDER['ForD']).fillna(0)
    pct_ = ct / ct.sum(axis=0).replace(0, np.nan) * 100
    y = np.arange(len(bands))
    ax.barh(y, -pct_['Domestic'].fillna(0).values, color=BLUE, height=0.7, label=f'Domestic (n={int(ct["Domestic"].sum()):,})', edgecolor=INK['surface'])
    ax.barh(y, pct_['Foreign'].fillna(0).values, color=GREEN, height=0.7, label=f'Foreign (n={int(ct["Foreign"].sum()):,})', edgecolor=INK['surface'])
    for i, b in enumerate(bands):
        pd_, pf = pct_.loc[b, 'Domestic'], pct_.loc[b, 'Foreign']
        if not np.isnan(pd_): ax.text(-pd_ - 0.6, i, f'{pd_:.0f}% · {int(ct.loc[b, "Domestic"]):,}', ha='right', va='center', fontsize=8, color=INK['secondary'])
        if not np.isnan(pf):  ax.text(pf + 0.6, i, f'{pf:.0f}% · {int(ct.loc[b, "Foreign"]):,}', ha='left', va='center', fontsize=8, color=INK['secondary'])
    m_ = max(np.nanmax(pct_.values) * 1.5, 10) if pct_.notna().any().any() else 10
    ax.set_xlim(-m_, m_); ax.set_yticks(y); ax.set_yticklabels(bands); ax.axvline(0, color=INK['grid'])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{abs(v):.0f}%')); ax.grid(axis='y', visible=False)
    style(ax, title=ttl, xlabel='% of each side'); ax.legend(loc='lower right')
finish(fig, 'Age pyramids · Domestic vs Foreign', 'Labels = share of that side and count', name='08_pyramid')

## 8.2 Geography — country, region and the dominant country

Country is the only geography of the foreign book. The dominant foreign country gets its own level (`ForeignSplit`) so the rest of the foreign book can be seen on its own.

In [ ]:
fr = C[is_(C['ForD'], 'Foreign')].copy(); fr['Country'] = fr['Country'].fillna('(missing)')
print(f'Country coverage: {C["Country"].notna().mean():.1%} of active customers · {C["Country"].nunique()} distinct countries · foreign customers {len(fr):,}')
cc = fr.groupby('Country').agg(CIFs=('IorB', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').mean() * 100), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'),
                               MedianDep=('Deposits', lambda x: x[x > 0].median()), Select=('Select', lambda s: is_(s, 'Select').mean() * 100),
                               RealHH=('HHType', lambda s: is_(s, 'Real household').mean() * 100)).sort_values('CIFs', ascending=False)
cc['% foreign CIFs'] = cc['CIFs'] / max(cc['CIFs'].sum(), 1) * 100; cc['% foreign deposits'] = cc['Deposits'] / max(cc['Deposits'].sum(), 1e-9) * 100
cc = cc.rename(columns={'Individuals': '% Individual', 'MedianDep': 'Median deposits (holders)', 'Select': '% Select', 'RealHH': '% in real household'})
display(styled(cc.head(TOP_COUNTRIES), int_cols=['CIFs'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'],
               pct_cols=['% Individual', '% Select', '% in real household', '% foreign CIFs', '% foreign deposits'], gradient_cols=['% foreign CIFs', '% foreign deposits']))
top_c = cc.head(TOP_COUNTRIES)
if len(top_c):
    fig, axes = plt.subplots(1, 3, figsize=(17, 0.8 + 0.36 * len(top_c)), sharey=True)
    ct = pd.crosstab(fr['Country'], fr['IorB']).reindex(index=top_c.index, columns=ORDER['IorB']).fillna(0)
    left = np.zeros(len(ct))
    for g_ in ORDER['IorB']:
        axes[0].barh(np.arange(len(ct)), ct[g_].values, left=left, color=COLOR['IorB'][g_], label=g_, height=0.62, edgecolor=INK['surface']); left += ct[g_].values
    axes[0].set_yticks(np.arange(len(ct))); axes[0].set_yticklabels(ct.index); axes[0].invert_yaxis(); axes[0].legend(loc='lower right'); axes[0].grid(axis='y', visible=False)
    style(axes[0], title='Foreign customers by country and IorB', xlabel='CIFs')
    hbar(axes[1], top_c.index.tolist(), top_c['Deposits'].values, GREEN, fmt=money); style(axes[1], title='Deposits'); axes[1].xaxis.set_major_formatter(MONEY)
    hbar(axes[2], top_c.index.tolist(), top_c['Median deposits (holders)'].fillna(0).values, GREEN_T, fmt=money); style(axes[2], title='Median deposits per holder'); axes[2].xaxis.set_major_formatter(MONEY)
    for ax in axes[1:]: ax.tick_params(labelleft=False)
    finish(fig, f'Top {len(top_c)} foreign countries', name='08_countries')
dom_ = C[is_(C['ForD'], 'Domestic')]
print('Domestic customers by country:'); display(dom_['Country'].fillna('(missing)').value_counts().head(6).to_frame('CIFs'))
display(Markdown('#### Domestic / dominant country / other foreign'))
show_profile(profile_table(C, ['ForeignSplit']))
show_profile(profile_table(C, ['IorB', 'ForeignSplit']))
reg = C.groupby('Region').agg(CIFs=('IorB', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').mean() * 100), Select=('Select', lambda s: is_(s, 'Select').mean() * 100),
                              Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), Families=('ProductFamilies', 'mean')).reindex(REGION_ORDER).dropna(how='all')
reg['% CIFs'] = reg['CIFs'] / reg['CIFs'].sum() * 100; reg['% Deposits'] = reg['Deposits'] / max(reg['Deposits'].sum(), 1e-9) * 100
display(styled(reg.rename(columns={'Individuals': '% Individual', 'Select': '% Select', 'Families': 'Avg families'}), int_cols=['CIFs'], money_cols=['Deposits', 'Loans'],
               pct_cols=['% Individual', '% Select', '% CIFs', '% Deposits'], one_dec=['Avg families'], gradient_cols=['% CIFs', '% Deposits']))

## 8.3 Concentration & value at the customer level

Among holders (balance > 0). Then every active customer ranked by total balance into value bands, the top 100 depositors / borrowers profiled, and the tail above fixed thresholds.

In [ ]:
order4, pal4 = group_spec(['IorB', 'ForD'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, col in zip(axes, ['Deposits', 'Loans']):
    hold = C[C[col] > 0]
    x, y = lorenz(hold[col])
    if x is None: no_data(ax); continue
    ax.plot(x, y, color=INK['muted'], lw=2, ls='--', label=f'All holders (n={len(hold):,})')
    for g_ in order4:
        sg = hold.loc[is_(hold['Group2'], g_), col]; x, y = lorenz(sg)
        if x is not None: ax.plot(x, y, color=pal4[g_], lw=2, label=f'{g_} (n={len(sg):,})')
    p80, p50 = holders_for(C[col], 80), holders_for(C[col], 50)
    ax.axhline(80, color=INK['grid'], lw=1); ax.axvline(p80, color=RED, lw=1, ls=':')
    ax.annotate(f'{p80:.1f}% of holders = 80% of {col.lower()}\n{p50:.1f}% of holders = 50%', xy=(p80, 80), xytext=(min(p80 + 10, 55), 58), fontsize=8.5, color=INK['secondary'],
                arrowprops=dict(arrowstyle='-', color=INK['muted'], lw=0.8))
    ax.set_xlim(0, 100); ax.set_ylim(0, 101); ax.xaxis.set_major_formatter(PCT0); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=LABEL[col], xlabel='% of holders (largest first)', ylabel=f'cumulative % of {col.lower()}'); ax.legend(loc='lower right', fontsize=8)
finish(fig, 'Concentration among holders (customers)', 'Lorenz curves by IorB × ForD · red dotted line = share of holders that carries 80% of the balance', name='08_lorenz')
rows = []
for col in ['Deposits', 'Loans']:
    for g_ in ['All customers'] + order4:
        sg = C[col] if g_ == 'All customers' else C.loc[is_(C['Group2'], g_), col]
        rows.append({'balance': col, 'segment': g_, 'CIFs': len(sg), 'holders': int((sg > 0).sum()), '% holders': (sg > 0).mean() * 100 if len(sg) else np.nan, 'total $': sg.clip(lower=0).sum(),
                     'top 1% of holders': top_share(sg, .01), 'top 10%': top_share(sg, .10), 'holders for 80%': holders_for(sg, 80), 'Gini (holders)': gini(sg[sg > 0])})
conc = pd.DataFrame(rows).set_index(['balance', 'segment'])
display(styled(conc, pct_cols=['% holders', 'top 1% of holders', 'top 10%', 'holders for 80%'], money_cols=['total $'], int_cols=['CIFs', 'holders'], two_dec=['Gini (holders)']))

vb = C.groupby('ValueBand').agg(CIFs=('IorB', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), TotalBalance=('TotalBalance', 'sum'),
                                Entry=('TotalBalance', 'min'), Median=('TotalBalance', 'median'), Select=('Select', lambda x: is_(x, 'Select').mean() * 100),
                                Business=('IorB', lambda x: is_(x, 'Business').mean() * 100), Foreign=('ForD', lambda x: is_(x, 'Foreign').mean() * 100),
                                RealHH=('HHType', lambda s: is_(s, 'Real household').mean() * 100), Families=('ProductFamilies', 'mean'), Tenure=('LifetimeYears', 'median')).reindex(VB)
for c_ in ['CIFs', 'Deposits', 'Loans', 'TotalBalance']: vb[f'% {c_}'] = vb[c_] / max(vb[c_].sum(), 1e-9) * 100
vb = vb.rename(columns={'Entry': 'Entry threshold', 'Median': 'Median total balance', 'Select': '% Select', 'Business': '% Business', 'Foreign': '% Foreign', 'RealHH': '% in real household',
                        'Families': 'Avg families', 'Tenure': 'Median tenure (y)', '% TotalBalance': '% Total balance'})
display(styled(vb, pct_cols=['% CIFs', '% Deposits', '% Loans', '% Total balance', '% Business', '% Foreign', '% Select', '% in real household'],
               money_cols=['Entry threshold', 'Median total balance', 'Deposits', 'Loans', 'TotalBalance'], int_cols=['CIFs'], one_dec=['Avg families', 'Median tenure (y)']))
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={'width_ratios': [1.1, 1]})
stacked_100(axes[0], vb[['% CIFs', '% Deposits', '% Loans']].rename(columns=lambda c: c[2:]).T, VB, vb_colors); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=4)
style(axes[0], title='Share of customers vs. share of money, by value band')
comp = pd.crosstab(C['ValueBand'], C['Group2'], normalize='index').reindex(index=VB, columns=order4).fillna(0) * 100
stacked_100(axes[1], comp, order4, pal4); axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=4); style(axes[1], title='Who is in each band (IorB × ForD)')
finish(fig, 'Customer value pyramid', 'Bands = rank of every active customer by total balance', name='08_pyramid')
seg_band = pd.crosstab(C['Group3'], C['ValueBand'], normalize='index').reindex(index=GROUP3_ORDER, columns=VB).fillna(0) * 100
fig, ax = plt.subplots(figsize=(9, 0.6 + 0.45 * len(seg_band)))
heat(ax, seg_band, fmt='.0f', vmax=100, suffix='%'); ax.tick_params(axis='x', labelrotation=0)
finish(fig, 'Share of each segment that falls in each value band', 'Rows sum to 100%', name='08_segment_bands')

for col in ['Deposits', 'Loans']:
    t = C.nlargest(100, col)
    if t[col].sum() <= 0: continue
    print(f'Top 100 by {col}: {t[col].sum() / max(C[col].clip(lower=0).sum(), 1):.1%} of the book · {is_(t["IorB"], "Business").mean():.0%} business · {is_(t["ForD"], "Foreign").mean():.0%} foreign · '
          f'{is_(t["Select"], "Select").mean():.0%} Select · {is_(t["HHType"], "Real household").mean():.0%} in a real household · {is_(t["Segment"], "Unassigned").sum()} Unassigned segment')
    display(styled(t.groupby('Group3').agg(CIFs=(col, 'size'), Amount=(col, 'sum')).sort_values('Amount', ascending=False).assign(**{'% of top-100 $': lambda d: d['Amount'] / d['Amount'].sum() * 100}),
                   money_cols=['Amount'], pct_cols=['% of top-100 $'], int_cols=['CIFs']))
rows = []
for thr in [1e5, 1e6, 5e6, 10e6, 50e6]:
    m = C['TotalBalance'] >= thr
    rows.append({'Total balance ≥': money(thr), 'CIFs': int(m.sum()), '% of CIFs': m.mean() * 100, '% of deposits': C.loc[m, 'Deposits'].sum() / max(C['Deposits'].sum(), 1e-9) * 100,
                 '% of loans': C.loc[m, 'Loans'].sum() / C['Loans'].sum() * 100 if C['Loans'].sum() else np.nan, '% Business': is_(C.loc[m, 'IorB'], 'Business').mean() * 100 if m.any() else np.nan,
                 '% Foreign': is_(C.loc[m, 'ForD'], 'Foreign').mean() * 100 if m.any() else np.nan, '% Select': is_(C.loc[m, 'Select'], 'Select').mean() * 100 if m.any() else np.nan})
tail = pd.DataFrame(rows).set_index('Total balance ≥')
display(styled(tail, pct_cols=[c for c in tail.columns if c.startswith('%')], int_cols=['CIFs']))

**Observations §8** *(fill in after the run)*

- **IorB / ForD / Select:** …
- **Combinations:** the cells that stand out are … (… % of customers, … % of deposits, … % of loans).
- **Age:** individuals median …; domestic …, foreign …; deposits rise / stay flat with age …; … % of individuals 45–64 are also behind a business account.
- **Geography:** top 3 foreign countries = … % of foreign customers; the dominant country … holds … % of foreign deposits.
- **Concentration:** … % of deposit holders carry 80% of deposits; top 1% of customers (entry …) hold … %.

# 9. Roles — how CIFs connect to accounts

The owner is only one of the roles on an account. Joint owners share it, co-borrowers and guarantors back a loan, signers and attorneys operate it, beneficial owners control the business behind it. This section reads the file at its own grain (CIF × account × role): which roles exist, who plays them on whose accounts, how accounts are structured, which CIFs are only connected (own nothing), how often a role crosses household lines, and the **relationship networks** that shared accounts create.

## 9.1 Role catalogue — codes, descriptions and families

Every role code / description in the file, the family it was mapped to (`ROLE_RULES`), and where it appears. Check the `Other` rows: they are descriptions the rules did not recognize.

In [ ]:
r_ = rel.assign(_ind=is_(rel['CIF'].map(cif['IorB']), 'Individual'), _open=rel['AccountKey'].map(acct['IsOpen']).astype(bool),
                _bizacct=is_(rel['OwnerCIF'].map(cif['IorB']), 'Business'), _ln=rel['Domain'].eq('LN'))
r_['_open_acct'] = r_['AccountKey'].where(r_['_open'])
r_['Family (rules)'] = np.where(r_['OwnerRow'], 'Primary owner (role description)', r_['RoleFamilyRaw'])
cat = (r_.groupby([r_['RoleCode'].fillna('(blank)'), r_['RoleDesc'].fillna('(blank)'), 'Family (rules)'])
         .agg(Rows=('CIF', 'size'), Accounts=('AccountKey', 'nunique'), CIFs=('CIF', 'nunique'), Ind=('_ind', 'mean'), Open=('_open', 'mean'), BizAcct=('_bizacct', 'mean'), LN=('_ln', 'mean'))
         .sort_values('Rows', ascending=False))
for c_ in ['Ind', 'Open', 'BizAcct', 'LN']: cat[c_] *= 100
cat = cat.rename(columns={'Ind': '% held by individuals', 'Open': '% on open accounts', 'BizAcct': '% on business-owned accounts', 'LN': '% on loans'})
display(styled(cat, int_cols=['Rows', 'Accounts', 'CIFs'], pct_cols=['% held by individuals', '% on open accounts', '% on business-owned accounts', '% on loans'], gradient_cols=['Rows']))
oth_desc = rel.loc[rel['RoleFamilyRaw'].eq('Other'), 'RoleDesc'].fillna('(blank)').value_counts()
if len(oth_desc): print(f'⚠ Role descriptions mapped to "Other" (extend ROLE_RULES if they mean something): {oth_desc.head(15).to_dict()}')

fam = (r_.groupby('RoleFamily').agg(Rows=('CIF', 'size'), OpenAccounts=('_open_acct', 'nunique'), CIFs=('CIF', 'nunique'),
                                    Ind=('_ind', 'mean'), BizAcct=('_bizacct', 'mean'), LN=('_ln', 'mean')).reindex(ROLE_FAMILIES).dropna(subset=['Rows']))
for c_ in ['Ind', 'BizAcct', 'LN']: fam[c_] *= 100
fam = fam.rename(columns={'OpenAccounts': 'Open accounts', 'Ind': '% held by individuals', 'BizAcct': '% on business-owned accounts', 'LN': '% on loans'})
display(styled(fam, int_cols=['Rows', 'Open accounts', 'CIFs'], pct_cols=['% held by individuals', '% on business-owned accounts', '% on loans']))
fig, ax = plt.subplots(figsize=(10, 0.6 + 0.4 * len(fam)))
hbar(ax, fam.index.tolist(), fam['Open accounts'].values, [ROLE_COLOR[f] for f in fam.index], fmt=fmt_int)
style(ax, xlabel='open accounts where the role appears')
finish(fig, 'Role families on open accounts', 'Primary owner = one per account by construction (§3.4)', name='09_role_families')

## 9.2 Who plays which role on whose account

Non-owner links on **open** accounts, by the type of the CIF holding the role and the type of the account owner. "Individual on a business account" is the people behind companies; "business on a business account" is corporate structures (a holding signing for a subsidiary).

In [ ]:
lk = oth[oth['IsOpen']].drop_duplicates(['CIF', 'AccountKey', 'RoleFamily']).copy()
lk['HolderIorB'] = lk['CIF'].map(cif['IorB'])
lk['Direction'] = lk['HolderIorB'].astype(str) + ' on ' + lk['OwnerIorB'].astype(str).str.lower() + ' account'
DIR_ORDER = [f'{a} on {b.lower()} account' for a in ORDER['IorB'] for b in ORDER['IorB']]
mat = pd.crosstab(lk['RoleFamily'], lk['Direction']).reindex(index=[f for f in ROLE_FAMILIES[1:] if f in set(lk['RoleFamily'])], columns=[d for d in DIR_ORDER if d in set(lk['Direction'])]).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(17, 0.9 + 0.42 * max(len(mat), 4)), gridspec_kw={'width_ratios': [1, 1.25]})
if mat.size:
    heat(axes[0], mat.div(mat.sum(axis=0).replace(0, np.nan), axis=1) * 100, annot=mat.map(fmt_int), vmax=100); axes[0].tick_params(axis='x', labelrotation=15)
    style(axes[0], title='Links on open accounts · color = share of the column')
else: no_data(axes[0])
A_ = A.assign(_g=A['IorB'].astype(str) + SEP + A['Domain'])
pen_rows = [f'{a}{SEP}{f}' for a in ORDER['IorB'] for f in FAM]
pen = pd.DataFrame({f: A_.groupby('_g')[f'Has: {f}'].mean() * 100 for f in ROLE_FAMILIES[1:]}).reindex(pen_rows).dropna(how='all')
pen = pen.loc[:, pen.fillna(0).max() > 0]
if pen.size: heat(axes[1], pen, fmt='.0f', vmax=100, suffix='%'); axes[1].tick_params(axis='x', labelrotation=30)
else: no_data(axes[1])
style(axes[1], title='% of open accounts that carry each role (owner type · domain)')
finish(fig, 'Who plays which role on whose account', 'Left: counts of CIF × account links (color = share of the column) · right: penetration of each role', name='09_role_matrix')
print(f'Open accounts with at least one non-owner CIF: {A["OtherCIFs"].gt(0).mean():.1%} · business-owned: {A.loc[is_(A["IorB"], "Business"), "OtherCIFs"].gt(0).mean():.1%} · '
      f'individual-owned: {A.loc[is_(A["IorB"], "Individual"), "OtherCIFs"].gt(0).mean():.1%}')

## 9.3 Ownership structures — sole, joint, guaranteed, operated by others

Every open account gets one `Structure` (first that applies: sole owner → joint / co-borrowers → guaranteed → owner + signers / agents → owner + other roles). Then how much of the money sits in each structure, and how much of the loan book has a guarantor.

In [ ]:
SC = COLOR['Structure']
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6), gridspec_kw={'width_ratios': [1, 1.2, 1]})
ct = pd.crosstab(A['Domain'], A['Structure'], normalize='index').reindex(index=FAM, columns=STRUCT_ORDER).fillna(0) * 100
stacked_100(axes[0], ct, STRUCT_ORDER, SC); style(axes[0], title='Open accounts by structure · domain')
ct = pd.crosstab(A['Group2'], A['Structure'], normalize='index').reindex(index=order4, columns=STRUCT_ORDER).fillna(0) * 100
stacked_100(axes[1], ct, STRUCT_ORDER, SC); style(axes[1], title='… by owner IorB × ForD'); axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3)
ct = A.groupby(['Domain', 'Structure'])['OpenBalance'].sum().clip(lower=0).unstack().reindex(index=FAM, columns=STRUCT_ORDER).fillna(0)
stacked_100(axes[2], ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100, STRUCT_ORDER, SC); style(axes[2], title='Share of each domain\'s balance by structure')
finish(fig, 'How open accounts are structured', 'Structure = who else is on the account besides the owner', name='09_structures')
stt = A.groupby('Structure').agg(Accounts=('OpenBalance', 'size'), Balance=('OpenBalance', 'sum'), Median=('OpenBalance', lambda s: s[s > 0].median()),
                                 CIFs=('CIFsOnAccount', 'mean'), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100)).reindex(STRUCT_ORDER).dropna(subset=['Accounts'])
stt['% accounts'] = stt['Accounts'] / stt['Accounts'].sum() * 100; stt['% balance'] = stt['Balance'] / max(stt['Balance'].clip(lower=0).sum(), 1e-9) * 100
display(styled(stt.rename(columns={'Balance': 'Balance $', 'Median': 'Median balance (> 0)', 'CIFs': 'Avg CIFs on account', 'Business': '% business-owned'}),
               int_cols=['Accounts'], money_cols=['Balance $', 'Median balance (> 0)'], pct_cols=['% business-owned', '% accounts', '% balance'], two_dec=['Avg CIFs on account']))
L_ = A[A['Domain'].eq('LN')]
if len(L_):
    g_ = L_.groupby('Group2')
    gl = pd.DataFrame({'Open loans': g_.size(), 'Loan balance': g_['Loans'].sum(),
                       '% of loans with a guarantor': g_['Has: Guarantor'].mean() * 100,
                       '% of loan balance guaranteed': g_.apply(lambda d: d.loc[d['Has: Guarantor'], 'Loans'].sum() / max(d['Loans'].sum(), 1e-9) * 100),
                       '% with a co-borrower': g_['Has: Co-borrower'].mean() * 100}).reindex(order4)
    gl.loc['All loans'] = [len(L_), L_['Loans'].sum(), L_['Has: Guarantor'].mean() * 100, L_.loc[L_['Has: Guarantor'], 'Loans'].sum() / max(L_['Loans'].sum(), 1e-9) * 100, L_['Has: Co-borrower'].mean() * 100]
    print('Loans backed by someone else:'); display(styled(gl, int_cols=['Open loans'], money_cols=['Loan balance'], pct_cols=[c for c in gl.columns if c.startswith('%')]))

## 9.4 Owners, connected persons and the people who are both

Every CIF in the file by role profile and status. **Connected** CIFs own nothing open but sign, guarantee or co-own somebody else's account: they are inside the bank's relationships without being customers — the most direct cross-sell list the file can produce.

In [ ]:
rp = pd.crosstab(cif['RoleProfile'], cif['Status']).reindex(index=ORDER['RoleProfile'] + ['None'], columns=CIF_STATUS).fillna(0).astype(int).loc[lambda t: t.sum(axis=1) > 0]
rp['All'] = rp.sum(axis=1); display(rp)
con = cif[cif['Status'].eq('Connected (non-owner)')]
if len(con):
    ci = is_(con['IorB'], 'Individual')
    print(f'Connected CIFs: {len(con):,} · {ci.mean():.0%} individuals · {is_(con["ForD"], "Foreign").mean():.0%} foreign · median age {con.loc[ci, "Age"].median():.0f} · '
          f'{con["OwnedEver"].gt(0).mean():.0%} owned an account in the past · {con["LinkedBusinesses"].gt(0).mean():.0%} are behind a business account')
    t = pd.DataFrame({'Connected CIFs': con['MainOtherRole'].value_counts(), 'of which individuals': con.loc[ci, 'MainOtherRole'].value_counts(),
                      'linked to a business': con.loc[con['LinkedBusinesses'].gt(0), 'MainOtherRole'].value_counts(),
                      'same household as an owner they serve': None}).reindex([f for f in ROLE_FAMILIES if f in set(con['MainOtherRole'])]).fillna(0)
    same_hh = oth[oth['IsOpen']].assign(_same=lambda d: d['CIF'].map(cif['HouseholdID']).eq(d['OwnerCIF'].map(cif['HouseholdID'])))
    t['same household as an owner they serve'] = same_hh[same_hh['CIF'].isin(con.index)].groupby('CIF')['_same'].any().groupby(con['MainOtherRole']).sum().reindex(t.index).fillna(0)
    display(styled(t.astype(int), int_cols=list(t.columns)))
dual = C[C['RoleAcctsOpen'].gt(0)]
print(f'Active customers who also hold a role on someone else\'s open account: {len(dual):,} ({len(dual) / max(len(C), 1):.1%}) · '
      f'by IorB × ForD: ' + ' · '.join(f'{g_} {is_(C["Group2"], g_)[C["RoleAcctsOpen"].gt(0)].sum() / max(is_(C["Group2"], g_).sum(), 1):.0%}' for g_ in order4))
fig, axes = plt.subplots(1, 2, figsize=(14, 3.4))
ct = pd.crosstab(C['Group2'], C['RoleAcctsOpen'].gt(0).map({True: 'Also in other roles', False: 'Owner only'}), normalize='index').reindex(index=order4, columns=['Owner only', 'Also in other roles']).fillna(0) * 100
stacked_100(axes[0], ct, ['Owner only', 'Also in other roles'], {'Owner only': BLUE, 'Also in other roles': BLUE_T}); style(axes[0], title='Active customers: do they also sign / guarantee elsewhere?')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
if len(con):
    cc_ = pd.crosstab(con['MainOtherRole'], con['IorB']).reindex(index=[f for f in ROLE_FAMILIES if f in set(con['MainOtherRole'])], columns=ORDER['IorB']).fillna(0)
    left = np.zeros(len(cc_))
    for g_ in ORDER['IorB']:
        axes[1].barh(np.arange(len(cc_)), cc_[g_].values, left=left, color=COLOR['IorB'][g_], label=g_, height=0.62, edgecolor=INK['surface']); left += cc_[g_].values
    axes[1].set_yticks(np.arange(len(cc_))); axes[1].set_yticklabels(cc_.index); axes[1].invert_yaxis(); axes[1].legend(loc='lower right'); axes[1].grid(axis='y', visible=False)
    style(axes[1], title='Connected CIFs by their main role', xlabel='CIFs')
else: no_data(axes[1])
finish(fig, 'Owners and connected persons', name='09_connected')

## 9.5 Roles across household lines

A role holder who belongs to a different household than the owner is a relationship the household grouping does not capture. Measured on non-owner links on open accounts.

In [ ]:
x = oth[oth['IsOpen']].drop_duplicates(['CIF', 'AccountKey']).copy()
x['SameHH'] = x['CIF'].map(cif['HouseholdID']).eq(x['OwnerCIF'].map(cif['HouseholdID']))
x['OwnerHHType'] = x['OwnerCIF'].map(cif['HHType'])
if len(x):
    t = x.groupby('RoleFamily').agg(Links=('SameHH', 'size'), Same=('SameHH', 'mean')).reindex(ROLE_FAMILIES[1:]).dropna()
    t['% same household'] = t.pop('Same') * 100; t['% in another household'] = 100 - t['% same household']
    t2 = x.groupby('OwnerHHType')['SameHH'].mean().mul(100).reindex(ORDER['HHType'])
    display(styled(t, int_cols=['Links'], pct_cols=['% same household', '% in another household']))
    print(f'Non-owner links on open accounts that cross household lines: {(~x["SameHH"]).mean():.1%} '
          f'(owner in a real household: {100 - t2.get("Real household", np.nan):.1f}% · owner in a single-CIF household: {100 - t2.get("Single CIF", np.nan):.1f}%)')
    fig, ax = plt.subplots(figsize=(10, 0.6 + 0.4 * len(t)))
    stacked_100(ax, t[['% same household', '% in another household']], ['% same household', '% in another household'], {'% same household': BLUE, '% in another household': GRAY})
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
    finish(fig, 'Do role holders belong to the owner\'s household?', 'Gray = relationships the household ID does not capture', name='09_cross_household')

## 9.6 Relationship networks — what shared accounts connect

Two CIFs on the same open account are connected; a chain of shared accounts makes a **network** (a connected component of the CIF–account graph). A network can be bigger than a household (a signer who is also a joint owner elsewhere) or split one (household members with no account in common). Networks are the economic relationship the accounts reveal; households are the one the bank recorded.

In [ ]:
lk_all = rel[rel['AccountKey'].map(acct['IsOpen']).astype(bool)][['CIF', 'AccountKey']].drop_duplicates()
cif_ids = pd.Index(lk_all['CIF'].unique()); acc_ids = pd.Index(lk_all['AccountKey'].unique())
if len(lk_all):
    ii = cif_ids.get_indexer(lk_all['CIF']); jj = acc_ids.get_indexer(lk_all['AccountKey']) + len(cif_ids)
    n_nodes = len(cif_ids) + len(acc_ids)
    G = coo_matrix((np.ones(len(ii)), (ii, jj)), shape=(n_nodes, n_nodes))
    n_comp, labels_ = connected_components(G, directed=False)
    cif['NetworkID'] = pd.Series(labels_[:len(cif_ids)], index=cif_ids).reindex(cif.index)
else:
    cif['NetworkID'] = np.nan
nw = cif.dropna(subset=['NetworkID']).assign(_act=lambda d: d['Status'].eq('Active customer').astype(int), _ind=lambda d: is_(d['IorB'], 'Individual').astype(int), _biz=lambda d: is_(d['IorB'], 'Business').astype(int))
net = nw.groupby('NetworkID').agg(CIFs=('IorB', 'size'), Households=('HouseholdID', 'nunique'), Customers=('_act', 'sum'), Individuals=('_ind', 'sum'), Businesses=('_biz', 'sum'),
                                  Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'))
net['TotalBalance'] = net['Deposits'] + net['Loans']
cif['NetworkSize'] = cif['NetworkID'].map(net['CIFs']); cif['NetworkHouseholds'] = cif['NetworkID'].map(net['Households'])
C = cif[act_c]
multi_net = net[net['CIFs'] >= 2]
print(f'Networks (CIFs linked through open accounts): {len(net):,} · with 2+ CIFs: {len(multi_net):,} · largest: {net["CIFs"].max() if len(net) else 0:,} CIFs')
print(f'Active customers in a network of 2+ CIFs: {C["NetworkSize"].ge(2).mean():.1%} (vs. {is_(C["HHType"], "Real household").mean():.1%} in a real household)')
print(f'Networks spanning more than one household: {(net["Households"] > 1).sum():,} · they hold {net.loc[net["Households"] > 1, "TotalBalance"].sum() / max(net["TotalBalance"].clip(lower=0).sum(), 1):.1%} of total balance')
rh = cif[is_(cif['HHType'], 'Real household') & cif['NetworkID'].notna()].groupby('HouseholdID')['NetworkID'].nunique()
print(f'Real households whose members sit in more than one network (no shared account between them): {(rh > 1).sum():,} of {len(rh):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
nb_ = count_band(C['NetworkSize']); hb_ = count_band(C['HHMembers'])
t = pd.DataFrame({'Network size': nb_.value_counts(normalize=True) * 100, 'Household size': hb_.value_counts(normalize=True) * 100}).reindex([b for b in COUNT_BANDS if b != '0']).fillna(0)
grouped_bars(axes[0], t, ['Household size', 'Network size'], {'Household size': BLUE, 'Network size': GREEN}, fmt=lambda v: f'{v:.0f}%'); style(axes[0], title='Active customers: size of their household vs. their network (CIFs)', pct_y=True)
ct = pd.crosstab(count_band(net.loc[net['CIFs'] >= 2, 'Households']).rename('households'), 'n')
if len(ct): hbar(axes[1], ct.index.tolist(), ct['n'].values, GREEN, fmt=fmt_int); style(axes[1], title='Networks of 2+ CIFs by number of households they span', xlabel='networks')
else: no_data(axes[1])
finish(fig, 'Relationship networks vs. households', 'Network = CIFs connected by shared open accounts (any role)', name='09_networks')
top_net = net.nlargest(15, 'TotalBalance')
display(styled(top_net, int_cols=['CIFs', 'Households', 'Customers', 'Individuals', 'Businesses'], money_cols=['Deposits', 'Loans', 'TotalBalance']))

## 9.7 Anatomy of one relationship

The largest active real household that mixes individuals and businesses, laid out member by member and account by account (IDs only). It shows in one picture what the rules above do: one owner per account, the people behind the companies, joint owners, guarantors, and CIFs from outside the household who appear on its accounts.

In [ ]:
ROLE_ABBR = {'Primary owner': 'OWN', 'Joint / co-owner': 'JNT', 'Co-borrower': 'COB', 'Guarantor': 'GUA', 'Beneficial owner / controller': 'BO', 'Authorized signer': 'SIG',
             'Power of attorney / agent': 'POA', 'Fiduciary / custodial': 'FID', 'Beneficiary': 'BEN', 'Mailing / tax / interest': 'MAIL', 'Other': 'OTH'}
cand_h = H[is_(H['HHType'], 'Real household') & is_(H['Composition'], 'Mixed')]
if not len(cand_h): cand_h = H[is_(H['HHType'], 'Real household')]
if len(cand_h):
    hid = cand_h['TotalBalance'].idxmax()
    mem_ = cif[cif['HouseholdID'].eq(hid)]
    accs = acct[acct['OwnerCIF'].isin(mem_.index) & acct['IsOpen']].sort_values('OpenBalance', ascending=False).head(14)
    rr = rel[rel['AccountKey'].isin(accs.index)]
    who = pd.Index(mem_.index.tolist() + [c for c in rr['CIF'].unique() if c not in set(mem_.index)])
    print(f'Household {hid}: {len(mem_)} members · {len(acct[acct["OwnerCIF"].isin(mem_.index) & acct["IsOpen"]])} open accounts · deposits {money(hh.loc[hid, "Deposits"])} · loans {money(hh.loc[hid, "Loans"])}')
    mt = cif.loc[who, ['IorB', 'ForD', 'Status', 'Age', 'BusinessAge', 'KP_CIF', 'OwnedOpen', 'Deposits', 'Loans']].copy()
    mt.insert(0, 'In household', np.where(mt.index.isin(mem_.index), 'member', 'outside'))
    display(styled(mt, int_cols=['OwnedOpen'], money_cols=['Deposits', 'Loans'], one_dec=['Age', 'BusinessAge']))
    rr = rr.assign(_r=rr['RoleFamily'].map(ROLE_ABBR))
    grid_ = rr.pivot_table(index='CIF', columns='AccountKey', values='_r', aggfunc=lambda s: '/'.join(sorted(set(s)))).reindex(index=who, columns=accs.index).fillna('')
    grid_.columns = [f'{k}\n{money(accs.loc[k, "OpenBalance"])}' for k in grid_.columns]
    fig, ax = plt.subplots(figsize=(max(8, 1.05 * len(accs) + 3), 0.8 + 0.42 * len(who)))
    code_val = grid_.map(lambda v: 2 if 'OWN' in v else (1 if v else 0)).astype(float)
    sns.heatmap(code_val, annot=grid_.values, fmt='', cmap=LinearSegmentedColormap.from_list('anat', ['#FFFFFF', BLUE_T, BLUE]), cbar=False, ax=ax, linewidths=1, linecolor=INK['grid'], annot_kws={'size': 8}, vmin=0, vmax=2)
    for t_ in ax.texts: t_.set_color('white' if t_.get_text().startswith('OWN') else INK['primary'])
    ax.set_yticklabels([f'{c} · {cif.loc[c, "IorB"]}' + ('' if c in set(mem_.index) else ' (outside)') for c in who], rotation=0)
    ax.tick_params(axis='x', labelrotation=0, labelsize=7.5); ax.set_xlabel(''); ax.set_ylabel(''); ax.grid(False)
    finish(fig, 'Anatomy of one relationship', 'Rows = CIFs, columns = open accounts owned by the household (largest first). OWN owner · JNT joint · COB co-borrower · GUA guarantor · BO beneficial owner · SIG signer · POA attorney · FID fiduciary · BEN beneficiary', name='09_anatomy')

**Observations §9** *(fill in after the run)*

- **Roles in the file:** … role codes in … families; … % of open accounts have someone besides the owner (business-owned …, individual-owned …).
- **Who plays what:** … % of non-owner links are individuals on business accounts (signers …, guarantors …, beneficial owners …).
- **Structures:** sole owner … % of open accounts / … % of balance; joint …; guaranteed …. … % of the loan balance carries a guarantor.
- **Connected persons:** … CIFs own nothing open but hold roles — … % individuals, … % behind a business, … % used to own an account.
- **Across households / networks:** … % of role links cross household lines; … % of active customers sit in a network of 2+ CIFs vs. … % in a real household; … networks span several households and hold … % of the money.

# 10. The person behind the business — key-person age

A business CIF has its own CIF and an **incorporation** date, but the decisions are made by people: the beneficial owners, guarantors, joint owners and signers on its accounts. §5.5 picked one **key person** per business with an explicit priority rule. This section checks how well that works (coverage, which role decides it), compares the age behind businesses with the age of individual customers, sizes the **succession** exposure (businesses whose key person is 65+), counts key persons who do not bank with us personally, and puts every customer on one **decision-maker age** axis. Active customers only unless stated.

## 10.1 Coverage — how many businesses have an identifiable person behind them

In [ ]:
B = C[is_(C['IorB'], 'Business')].copy()
if len(B):
    B['KP_Label'] = B['KP_Label'].fillna('No individual linked')
    def kp_cov(d):
        return pd.Series({'Businesses': len(d), '% role-linked person': d['KP_Source'].eq('Account role').mean() * 100, '% household only': d['KP_Source'].eq('Household').mean() * 100,
                          '% nobody linked': d['KP_Source'].isna().mean() * 100, '% key-person age known': d['KP_Age'].notna().mean() * 100,
                          'Median linked persons': d['LinkedPersons'].median(), 'Median key-person age': d['KP_Age'].median(), 'Median business age': d['BusinessAge'].median(),
                          '% key person is a customer too': d['KP_IsCustomer'].astype(float).mean() * 100 if d['KP_IsCustomer'].notna().any() else np.nan,
                          'Deposits $': d['Deposits'].sum(), 'Loans $': d['Loans'].sum()})
    cov = pd.concat({f'{a}{SEP}{b}': kp_cov(B[is_(B['ForD'], a) & is_(B['Select'], b)]) for a in ORDER['ForD'] for b in ORDER['Select']}, axis=1).T
    cov.loc['All active businesses'] = kp_cov(B)
    display(styled(cov, int_cols=['Businesses'], pct_cols=[c for c in cov.columns if c.startswith('%')], one_dec=['Median linked persons', 'Median key-person age', 'Median business age'],
                   money_cols=['Deposits $', 'Loans $']))
    fig, axes = plt.subplots(1, 2, figsize=(16, 3.4), gridspec_kw={'width_ratios': [1.3, 1]})
    KPC = dict(zip(KP_LABELS, [ROLE_COLOR[f] for f in KEY_PERSON_PRIORITY] + [GRAY, GRAY_L]))
    ct = pd.crosstab(B['ForD'], B['KP_Label'], normalize='index').reindex(index=ORDER['ForD'], columns=KP_LABELS).fillna(0) * 100
    stacked_100(axes[0], ct, KP_LABELS, KPC); style(axes[0], title='Which role defines the key person, by ForD'); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=7.8)
    lp_ = count_band(B['LinkedPersons'].fillna(0))
    ct = pd.crosstab(lp_, B['ForD'], normalize='columns').reindex(index=[b for b in COUNT_BANDS if b in set(lp_)], columns=ORDER['ForD']) * 100
    grouped_bars(axes[1], ct, ORDER['ForD'], COLOR['ForD'], fmt=lambda v: f'{v:.0f}%'); style(axes[1], title='Individuals linked to each business', ylabel='% of businesses', pct_y=True)
    finish(fig, 'The people behind active businesses', 'Linked = individual with a qualifying role on the business\'s own accounts (or in its real household)', name='10_coverage')
else:
    print('No active businesses')

## 10.2 How old is the business vs. how old are the people behind it

Key-person age next to the age of individual customers (are business owners older than retail customers?), and next to the age of the company itself. A young company run by an older person is a new venture of an established client; an old company with an old key person is a succession question.

In [ ]:
if len(B):
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.6))
    d_ = pd.concat([C.loc[is_(C['IorB'], 'Individual'), ['Age', 'ForD']].rename(columns={'Age': 'years'}).assign(who='Individual customers'),
                    B[['KP_Age', 'ForD']].rename(columns={'KP_Age': 'years'}).assign(who='Key persons of businesses')])
    kde_by(axes[0], d_, 'years', 'who', ['Individual customers', 'Key persons of businesses'], {'Individual customers': BLUE, 'Key persons of businesses': GREEN}, clip=(18, 100))
    style(axes[0], title='Age: individual customers vs. key persons', xlabel='years')
    order_s, pal_s = group_spec(['ForD', 'Select'])
    kde_by(axes[1], B.assign(_g=make_label(B, ['ForD', 'Select'])), 'KP_Age', '_g', order_s, pal_s, clip=(18, 100)); style(axes[1], title='Key-person age by ForD × Select', xlabel='years')
    hm = pd.crosstab(B['BusinessAgeBand'], B['KP_AgeBand']).reindex(index=YEAR_BANDS, columns=AGE_BANDS).fillna(0)
    heat(axes[2], hm.div(hm.sum(axis=1).replace(0, np.nan), axis=0) * 100, annot=hm.map(fmt_int), vmax=100); style(axes[2], title='Business age (rows) × key-person age (columns)', xlabel='key-person age band', ylabel='years since incorporation')
    finish(fig, 'Age behind the business', 'Color in the heatmap = share of the row', name='10_ages')
    t = B.groupby('BusinessAgeBand').agg(Businesses=('KP_Age', 'size'), KnownKP=('KP_Age', lambda s: s.notna().mean() * 100), MedianKP=('KP_Age', 'median'),
                                         Old=('KP_Age', lambda s: s.ge(65).mean() * 100), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum')).reindex(YEAR_BANDS).dropna(subset=['Businesses'])
    display(styled(t.rename(columns={'KnownKP': '% key-person age known', 'MedianKP': 'Median key-person age', 'Old': '% key person 65+'}), int_cols=['Businesses'],
                   pct_cols=['% key-person age known', '% key person 65+'], one_dec=['Median key-person age'], money_cols=['Deposits', 'Loans']))

## 10.3 Succession exposure and key persons who are not customers

**Succession**: businesses whose key person is 65+ (and 75+), with the deposits and loans they carry — the relationship depends on a transition that has not happened yet. **Cross-sell**: key persons who hold no account of their own at the bank (connected only, former, or never owners).

In [ ]:
if len(B):
    rows = []
    for g_ in ORDER['ForD'] + ['All']:
        d_ = B if g_ == 'All' else B[is_(B['ForD'], g_)]
        k_ = d_['KP_Age'].notna()
        rows.append({'ForD': g_, 'Businesses with known key-person age': int(k_.sum()),
                     '% key person 65+': d_.loc[k_, 'KP_Age'].ge(65).mean() * 100 if k_.any() else np.nan, '% key person 75+': d_.loc[k_, 'KP_Age'].ge(75).mean() * 100 if k_.any() else np.nan,
                     'Deposits with key person 65+': d_.loc[d_['KP_Age'].ge(65), 'Deposits'].sum(), '% of these deposits': d_.loc[d_['KP_Age'].ge(65), 'Deposits'].sum() / max(d_['Deposits'].sum(), 1e-9) * 100,
                     'Loans with key person 65+': d_.loc[d_['KP_Age'].ge(65), 'Loans'].sum(), '% of these loans': d_.loc[d_['KP_Age'].ge(65), 'Loans'].sum() / d_['Loans'].sum() * 100 if d_['Loans'].sum() else np.nan})
    succ = pd.DataFrame(rows).set_index('ForD')
    display(styled(succ, int_cols=['Businesses with known key-person age'], pct_cols=[c for c in succ.columns if c.startswith('%')], money_cols=['Deposits with key person 65+', 'Loans with key person 65+']))
    kpc = B[B['KP_CIF'].notna()]
    st_kp = kpc['KP_Status'].value_counts().reindex(CIF_STATUS).fillna(0).astype(int)
    print(f'Status of the key persons of active businesses: {st_kp.to_dict()}')
    not_cust = kpc[~kpc['KP_Status'].eq('Active customer')]
    print(f'Key persons who do not own an open account themselves: {not_cust["KP_CIF"].nunique():,} people behind {len(not_cust):,} businesses '
          f'({len(not_cust) / max(len(kpc), 1):.0%}) holding {money(not_cust["Deposits"].sum())} of deposits and {money(not_cust["Loans"].sum())} of loans')
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
    ct = pd.crosstab(kpc['Group3'].str.split(SEP).str[1:].str.join(SEP), kpc['KP_Status'], normalize='index').reindex(columns=CIF_STATUS).fillna(0) * 100
    ct = ct.reindex([f'{a}{SEP}{b}' for a in ORDER['ForD'] for b in ORDER['Select']]).dropna(how='all')
    stacked_100(axes[0], ct, CIF_STATUS, COLOR['Status']); style(axes[0], title='Is the key person a customer too? (business ForD × Select)')
    axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=7.8)
    ab_ = B.assign(_b=B['KP_AgeBand'].fillna('unknown'))
    t = ab_.groupby('_b').agg(Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum')).reindex(AGE_BANDS + ['unknown']).fillna(0)
    grouped_bars(axes[1], t, ['Deposits', 'Loans'], {'Deposits': BLUE, 'Loans': GREEN}, fmt=money); style(axes[1], title='Business deposits and loans by key-person age band', money_y=True)
    finish(fig, 'Succession and cross-sell behind the business', name='10_succession')

## 10.4 Decision-maker age for every customer, and the household reference age

`DecisionAge` = own age for individuals, key-person age for businesses: one age axis for the whole base. The household reference age is the age of the member who holds the most money (or of the key person of its largest business).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
ct = pd.crosstab(C['Group2'], C['DecisionAgeBand'].fillna('unknown'), normalize='index').reindex(index=order4, columns=AGE_BANDS + ['unknown']).fillna(0) * 100
stacked_100(axes[0], ct, AGE_BANDS + ['unknown'], dict(zip(AGE_BANDS + ['unknown'], BLUES_7 + [GRAY_L])), min_label=8)
style(axes[0], title='Decision-maker age band by IorB × ForD'); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=4, fontsize=7.8)
m_, lo_, hi_ = median_table(C, 'Deposits', 'DecisionAgeBand', 'IorB', AGE_BANDS, ORDER['IorB'], holders=True)
grouped_bars(axes[1], m_, ORDER['IorB'], COLOR['IorB'], fmt=money, lo=lo_, hi=hi_); style(axes[1], title='Median deposits (holders) by decision-maker age · 95% CI', money_y=True)
axes[1].set_ylim(0, np.nanmax(hi_.values) * 1.18 if np.isfinite(np.nanmax(hi_.values)) else 1)
hh_a = H.assign(_t=H['HHType'])
kde_by(axes[2], hh_a, 'RefAge', 'Composition', ORDER['Composition'], COLOR['Composition'], clip=(18, 100)); style(axes[2], title='Household reference age by composition', xlabel='years')
finish(fig, 'One age axis for every customer and household', name='10_decision_age')
t = H.groupby('RefAgeBand').agg(HHs=('Members', 'size'), Members=('Members', 'mean'), Real=('HHType', lambda s: is_(s, 'Real household').mean() * 100),
                                Deposits=('Deposits', 'sum'), MedDep=('Deposits', lambda s: s[s > 0].median()), Loans=('Loans', 'sum'), Families=('ProductFamilies', 'mean')).reindex(AGE_BANDS).dropna(subset=['HHs'])
t['% HHs'] = t['HHs'] / t['HHs'].sum() * 100; t['% Deposits'] = t['Deposits'] / max(t['Deposits'].sum(), 1e-9) * 100
display(styled(t.rename(columns={'Members': 'Avg members', 'Real': '% real household', 'MedDep': 'Median deposits (holders)', 'Families': 'Avg families'}),
               int_cols=['HHs'], one_dec=['Avg members', 'Avg families'], pct_cols=['% real household', '% HHs', '% Deposits'], money_cols=['Deposits', 'Median deposits (holders)', 'Loans']))

**Observations §10** *(fill in after the run)*

- **Coverage:** … % of active businesses have a role-linked individual, … % only a household link, … % nobody; key-person age known for … %.
- **Which role decides:** beneficial owner … %, guarantor … %, signer … %.
- **Ages:** key persons median … vs. individual customers …; foreign businesses' key persons … vs. domestic ….
- **Succession:** … % of businesses (… of deposits, … of loans) depend on a key person aged 65+.
- **Cross-sell:** … key persons behind … businesses do not own an account of their own.
- **One age axis:** decision-maker age known for … % of customers; deposits peak at … .

# 11. Accounts & products

The account level on its own: which products exist and which are still sold, how Select products are spread, what a typical account holds, how long accounts live and how they end, dormancy, and the channel (branches and officers). Balances are open accounts only; product life uses every account.

## 11.1 Product catalogue — what is open, what is still being sold

Per product: accounts ever and open, open balance, median balance, Select share, who owns it, how long closed accounts lived, and the first / last opening date. A product with open accounts but no opening in the last two years is a **legacy** product.

In [ ]:
acct['_closed_life'] = acct['LifeYears'].where(~acct['IsOpen'])
acct['_open_bal_pos'] = acct['OpenBalance'].where(acct['IsOpen'] & acct['OpenBalance'].gt(0))
A = acct[acct['IsOpen']]
pc = acct.groupby(['Domain', acct['Product'].fillna('(blank)')]).agg(
    Ever=('IsOpen', 'size'), Open=('IsOpen', 'sum'), Balance=('OpenBalance', 'sum'), Median=('_open_bal_pos', 'median'),
    Select=('AccountSelect', lambda s: is_(s, 'Select').mean() * 100), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100),
    Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100), LifeClosed=('_closed_life', 'median'), First=('Opened', 'min'), Last=('Opened', 'max'))
pc['% open'] = pc['Open'] / pc['Ever'] * 100
pc['Legacy (no opening in 2y)'] = (pc['Open'] > 0) & (pc['Last'] < SNAPSHOT - pd.DateOffset(years=2))
pc = pc.rename(columns={'Ever': 'Accounts ever', 'Open': 'Open', 'Balance': 'Open balance $', 'Median': 'Median open balance', 'Select': '% Select', 'Business': '% business-owned',
                        'Foreign': '% foreign-owned', 'LifeClosed': 'Median life of closed (y)', 'First': 'First opened', 'Last': 'Last opened'})
for f in FAM:
    t = pc.loc[f] if f in pc.index.get_level_values(0) else None
    if t is None or not len(t): continue
    display(Markdown(f'#### {FAM_LABEL[f]} — {len(t)} products, top {TOP_PRODUCTS} by open accounts'))
    t = t.sort_values('Open', ascending=False).head(TOP_PRODUCTS).copy()
    t['First opened'] = t['First opened'].dt.date; t['Last opened'] = t['Last opened'].dt.date
    display(styled(t, int_cols=['Accounts ever', 'Open'], money_cols=['Open balance $', 'Median open balance'], pct_cols=['% Select', '% business-owned', '% foreign-owned', '% open'],
                   one_dec=['Median life of closed (y)'], gradient_cols=['Open']))
leg = pc[pc['Legacy (no opening in 2y)']]
print(f'Legacy products (open accounts, no opening in 2 years): {len(leg):,} products · {int(leg["Open"].sum()):,} open accounts · {money(leg["Open balance $"].sum())}')
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, f in zip(axes, FAM):
    t = pc.loc[f].sort_values('Open balance $', ascending=False).head(10) if f in pc.index.get_level_values(0) else pd.DataFrame()
    if len(t): hbar(ax, [str(i)[:28] for i in t.index], t['Open balance $'].clip(lower=0).values, COLOR['Domain'][f], fmt=money); ax.xaxis.set_major_formatter(MONEY)
    else: no_data(ax)
    style(ax, title=f'{FAM_LABEL[f]} · top 10 products by open balance')
finish(fig, 'Products that carry the money', name='11_products')

## 11.2 Select products

Account-level Select (the product is a Select product). Who owns Select products, what share of each domain's balance they carry, and how the product flag lines up with the customer's segment.

In [ ]:
sp = A.groupby(['Domain', 'AccountSelect']).agg(Accounts=('OpenBalance', 'size'), Balance=('OpenBalance', 'sum'), Median=('_open_bal_pos', 'median')).unstack('AccountSelect')
sp = sp.reindex(FAM)
tbl = pd.DataFrame({'Open accounts · Select': sp[('Accounts', 'Select')] if ('Accounts', 'Select') in sp.columns else 0,
                    'Open accounts · Regular': sp[('Accounts', 'Regular')] if ('Accounts', 'Regular') in sp.columns else 0,
                    'Balance · Select': sp[('Balance', 'Select')] if ('Balance', 'Select') in sp.columns else 0,
                    'Balance · Regular': sp[('Balance', 'Regular')] if ('Balance', 'Regular') in sp.columns else 0,
                    'Median · Select': sp[('Median', 'Select')] if ('Median', 'Select') in sp.columns else np.nan,
                    'Median · Regular': sp[('Median', 'Regular')] if ('Median', 'Regular') in sp.columns else np.nan}).fillna({'Open accounts · Select': 0, 'Open accounts · Regular': 0})
tbl['% of accounts Select'] = tbl['Open accounts · Select'] / (tbl['Open accounts · Select'] + tbl['Open accounts · Regular']).replace(0, np.nan) * 100
tbl['% of balance Select'] = tbl['Balance · Select'] / (tbl['Balance · Select'].fillna(0) + tbl['Balance · Regular'].fillna(0)).replace(0, np.nan) * 100
display(styled(tbl, int_cols=['Open accounts · Select', 'Open accounts · Regular'], money_cols=['Balance · Select', 'Balance · Regular', 'Median · Select', 'Median · Regular'],
               pct_cols=['% of accounts Select', '% of balance Select']))
fig, axes = plt.subplots(1, 2, figsize=(15, 3.3))
pen_ = A.assign(_s=is_(A['AccountSelect'], 'Select')).pivot_table(index='Group2', columns='Domain', values='_s', aggfunc='mean').reindex(index=order4, columns=FAM) * 100
heat(axes[0], pen_, fmt='.0f', vmax=100, suffix='%'); style(axes[0], title='% of open accounts that are Select products (owner × domain)')
ct = pd.crosstab(A['AccountSelect'], A['SelectSegment'], normalize='index').reindex(index=ORDER['AccountSelect'], columns=ORDER['Select']).fillna(0) * 100
stacked_100(axes[1], ct, ORDER['Select'], {'Select': GREEN, 'Non-Select': GREEN_T}); style(axes[1], title='Owner\'s segment by product type (Select segment?)')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
finish(fig, 'Select products', 'Product flag (account) vs. segment flag (customer)', name='11_select')

## 11.3 What a typical account holds

Open accounts with a positive balance, per domain and owner (IorB × ForD), on a log scale. One hump per group means one kind of account; two humps mean two different uses of the same product.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 0.9 + 0.55 * len(order4) + 1.6))
for ax, f in zip(axes, FAM):
    d_ = A[A['Domain'].eq(f)]
    box_by(ax, d_.assign(_bal=d_['OpenBalance']), '_bal', 'Group2', order4, pal4); style(ax, title=f'{FAM_LABEL[f]} · balance per open account')
    if f != FAM[0]: ax.set_yticklabels([])
finish(fig, 'Balance per open account', 'Among accounts with a positive balance, log scale. Label = median; right = share of accounts with a positive balance.', name='11_acct_balances')
fig, axes = plt.subplots(1, 3, figsize=(17, 3.3))
for ax, f in zip(axes, FAM):
    s = A.loc[A['Domain'].eq(f) & A['OpenBalance'].gt(0), 'OpenBalance']
    if len(s) >= 10:
        sns.histplot(x=s, log_scale=True, bins=40, stat='percent', color=COLOR['Domain'][f], edgecolor=INK['surface'], linewidth=0.4, ax=ax)
        ymax = ax.get_ylim()[1]
        for q, lab, ls in [(.5, 'P50', '--'), (.9, 'P90', ':'), (.99, 'P99', ':')]:
            v = s.quantile(q); ax.axvline(v, color=INK['secondary'], lw=1, ls=ls); ax.text(v, ymax * 0.98, f' {lab} {money(v)}', rotation=90, ha='left', va='top', fontsize=7.2, color=INK['secondary'])
        ax.xaxis.set_major_formatter(MONEY)
    else: no_data(ax)
    style(ax, title=f'{f} (n={len(s):,})', ylabel='% of accounts')
finish(fig, 'Shape of the account balance distributions', 'Log scale · dashed = median, dotted = P90 and P99', name='11_acct_hist')

## 11.4 Account life — how long accounts live and how they end

Kaplan–Meier survival over every account with a known life (open accounts are censored at the snapshot), then how closed accounts ended. CDs and loans are *supposed* to end (maturity, payoff); a DDA ending is a relationship decision.

In [ ]:
sv = acct[acct['LifeYears'].notna()]
fig, axes = plt.subplots(1, 3, figsize=(18, 3.8), gridspec_kw={'width_ratios': [1.2, 1, 1]})
km_rows = []
for f in FAM:
    s_ = sv[sv['Domain'].eq(f)]
    if len(s_) < 20: continue
    x, s = kaplan_meier(s_['LifeYears'], ~s_['IsOpen'])
    axes[0].step(x, s * 100, where='post', color=COLOR['Domain'][f], lw=2, label=f'{f} (n={len(s_):,})')
    km_rows.append({'domain': f, 'accounts': len(s_), 'closed': int((~s_['IsOpen']).sum()), 'open after 1y %': km_at(x, s, 1), 'after 3y %': km_at(x, s, 3),
                    'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10), 'median life (y)': km_median(x, s)})
axes[0].axhline(50, color=INK['grid'], lw=1); axes[0].set_xlim(0, 30); axes[0].set_ylim(0, 101); axes[0].yaxis.set_major_formatter(PCT0)
style(axes[0], title='Share of accounts still open, by years since opening', xlabel='years', ylabel='% still open'); axes[0].legend()
cl_ = acct[~acct['IsOpen']]
how = pd.crosstab(cl_['Domain'], cl_['StatusDesc'].fillna('(blank)'), normalize='index').reindex(FAM) * 100
how = how.loc[:, how.max() > 0]
heat(axes[1], how, fmt='.0f', vmax=100, suffix='%'); axes[1].tick_params(axis='x', labelrotation=30); style(axes[1], title='How closed accounts ended (status)')
kde_by(axes[2], cl_, 'LifeYears', 'Domain', FAM, COLOR['Domain'], clip=(0, 30)); style(axes[2], title='Life of closed accounts', xlabel='years open')
finish(fig, 'Account life', 'Kaplan–Meier: open accounts censored at the snapshot · closed accounts without a close date excluded', name='11_life')
display(styled(pd.DataFrame(km_rows).set_index('domain'), int_cols=['accounts', 'closed'], pct_cols=['open after 1y %', 'after 3y %', 'after 5y %', 'after 10y %'], one_dec=['median life (y)']))

## 11.5 Dormancy — open on the books, silent in practice

Open accounts in a dormant / restricted status, and customers whose **every** open account is dormant: they still count as active customers but behave like the lost base.

In [ ]:
dm = A.groupby(['Domain']).agg(Open=('Dormant', 'size'), Dormant=('Dormant', 'sum'), DormBal=('OpenBalance', lambda s: s[A.loc[s.index, 'Dormant']].sum())).reindex(FAM)
dm['% dormant'] = dm['Dormant'] / dm['Open'].replace(0, np.nan) * 100
display(styled(dm.rename(columns={'Open': 'Open accounts', 'Dormant': 'Dormant / restricted', 'DormBal': 'Balance in dormant $'}), int_cols=['Open accounts', 'Dormant / restricted'],
               money_cols=['Balance in dormant $'], pct_cols=['% dormant']))
od = C.groupby('Group3').agg(Customers=('OnlyDormant', 'size'), OnlyDormant=('OnlyDormant', 'mean'), Deposits=('Deposits', lambda s: s[C.loc[s.index, 'OnlyDormant']].sum())).reindex(GROUP3_ORDER)
od['OnlyDormant'] *= 100
display(styled(od.rename(columns={'OnlyDormant': '% with every open account dormant', 'Deposits': 'Their deposits $'}), int_cols=['Customers'], pct_cols=['% with every open account dormant'], money_cols=['Their deposits $']))
print(f'Active customers whose every open account is dormant / restricted: {C["OnlyDormant"].sum():,} ({C["OnlyDormant"].mean():.1%}) · {money(C.loc[C["OnlyDormant"], "Deposits"].sum())} of deposits')

## 11.6 Channel — branches and officers

Open accounts, deposits and loans by branch, the branch × segment map, how concentrated officer portfolios are, which officer codes are pooled **house** codes (each holding more than `HOUSE_CODE_SHARE` % of open accounts), and how much of the book is still assigned to officers marked retired / inactive.

In [ ]:
bg = A.groupby(A['Branch'].fillna('(missing)'))
br = pd.DataFrame({'Open accounts': bg.size(), 'Customers': bg['OwnerCIF'].nunique(), 'Deposits': bg['Deposits'].sum(), 'Loans': bg['Loans'].sum(),
                   '% Individual-owned': bg['IorB'].apply(lambda s: is_(s, 'Individual').mean() * 100), '% Foreign-owned': bg['ForD'].apply(lambda s: is_(s, 'Foreign').mean() * 100),
                   '% Select products': bg['AccountSelect'].apply(lambda s: is_(s, 'Select').mean() * 100)}).sort_values('Open accounts', ascending=False)
for f in FAM: br[f'% {f}'] = bg['Domain'].apply(lambda s: s.eq(f).mean() * 100)
br['% Deposits'] = br['Deposits'] / max(br['Deposits'].sum(), 1e-9) * 100; br['% Loans'] = br['Loans'] / br['Loans'].sum() * 100 if br['Loans'].sum() else np.nan
display(styled(br, int_cols=['Open accounts', 'Customers'], money_cols=['Deposits', 'Loans'], pct_cols=[c for c in br.columns if c.startswith('%')], gradient_cols=['% Deposits', '% Loans']))
top_b = br.head(20)
fig, axes = plt.subplots(1, 3, figsize=(17, 0.8 + 0.38 * len(top_b)), sharey=True)
ct = pd.crosstab(A['Branch'].fillna('(missing)'), A['Domain']).reindex(index=top_b.index, columns=FAM).fillna(0)
left = np.zeros(len(ct))
for f in FAM:
    axes[0].barh(np.arange(len(ct)), ct[f].values, left=left, color=COLOR['Domain'][f], label=f, height=0.62, edgecolor=INK['surface']); left += ct[f].values
axes[0].set_yticks(np.arange(len(ct))); axes[0].set_yticklabels(ct.index); axes[0].invert_yaxis(); axes[0].legend(loc='lower right'); axes[0].grid(axis='y', visible=False)
style(axes[0], title='Open accounts by branch and domain', xlabel='accounts')
hbar(axes[1], top_b.index.tolist(), top_b['Deposits'].clip(lower=0).values, BLUE, fmt=money); style(axes[1], title='Deposits'); axes[1].xaxis.set_major_formatter(MONEY)
hbar(axes[2], top_b.index.tolist(), top_b['Loans'].clip(lower=0).values, GREEN, fmt=money); style(axes[2], title='Loans'); axes[2].xaxis.set_major_formatter(MONEY)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, 'Branches', f'Top {len(top_b)} by open accounts', name='11_branches')
bseg = pd.crosstab(A['Branch'].fillna('(missing)'), A['Group2']).reindex(index=top_b.index, columns=order4).fillna(0)
fig, ax = plt.subplots(figsize=(10, 0.7 + 0.4 * len(bseg)))
heat(ax, bseg.div(bseg.sum(axis=0).replace(0, np.nan), axis=1) * 100, annot=bseg.map(fmt_int), vmax=100); ax.tick_params(axis='x', labelrotation=0)
finish(fig, 'Branch × owner segment', 'Color = share of each segment\'s open accounts booked in the branch (columns sum to 100%) · cells = accounts', name='11_branch_segment')

off = A.groupby(A['OfficerCode'].fillna('(missing)')).agg(Accounts=('OpenBalance', 'size'), Customers=('OwnerCIF', 'nunique'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'),
                                                         Inactive=('OfficerInactive', 'max')).sort_values('Accounts', ascending=False)
off['% accounts'] = off['Accounts'] / off['Accounts'].sum() * 100; off['% deposits'] = off['Deposits'] / max(off['Deposits'].sum(), 1e-9) * 100
HOUSE = off.index[off['% accounts'] > HOUSE_CODE_SHARE].tolist()
A_house = A['OfficerCode'].fillna('(missing)').isin(HOUSE)
print(f'{len(off):,} officer codes on open accounts · median {off["Accounts"].median():.0f} accounts each · top 5 codes = {off["% accounts"].head(5).sum():.1f}% of accounts, {off["% deposits"].head(5).sum():.1f}% of deposits')
print(f'House codes (> {HOUSE_CODE_SHARE}% of open accounts each): {HOUSE} → {A_house.mean():.1%} of open accounts, {A.loc[A_house, "Deposits"].sum() / max(A["Deposits"].sum(), 1e-9):.1%} of deposits')
inact = A[A['OfficerInactive']]
print(f'Open accounts assigned to officers marked retired / inactive: {len(inact):,} ({len(inact) / max(len(A), 1):.1%}) · {inact["OwnerCIF"].nunique():,} customers · '
      f'{money(inact["Deposits"].sum())} deposits · {money(inact["Loans"].sum())} loans · {inact["OfficerCode"].nunique()} officer codes')
display(styled(off.head(12).assign(Inactive=lambda d: d['Inactive'].map({True: 'retired / inactive', False: ''})), int_cols=['Accounts', 'Customers'], money_cols=['Deposits', 'Loans'], pct_cols=['% accounts', '% deposits']))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
t = off.head(15)
hbar(axes[0], t.index.tolist(), t['Accounts'].values, [GRAY if c in HOUSE else (RED if t.loc[c, 'Inactive'] else BLUE) for c in t.index], fmt=lambda v: f'{v:,.0f} ({v / len(A):.0%})')
style(axes[0], title='Top 15 officer codes by open accounts', xlabel='accounts')
x, y = lorenz(off['Accounts'].astype(float)); axes[1].plot(x, y, color=BLUE, lw=2, label='Open accounts') if x is not None else None
x, y = lorenz(off['Deposits']); axes[1].plot(x, y, color=GREEN, lw=2, label='Deposits') if x is not None else None
axes[1].set_xlim(0, 100); axes[1].set_ylim(0, 100); axes[1].xaxis.set_major_formatter(PCT0); axes[1].yaxis.set_major_formatter(PCT0); axes[1].legend(loc='lower right')
style(axes[1], title='Portfolio concentration across officer codes', xlabel='% of officer codes (largest first)', ylabel='cumulative %')
finish(fig, 'Officer portfolios', 'Gray = house codes · red = officer marked retired / inactive', name='11_officers')

## 11.7 Large Relationship flag

The flag is used at the grain detected in §3.5. Who carries it, and how much of the book it covers.

In [ ]:
if LARGE_GRAIN in ('CIF', 'Household'):
    lrc = C.assign(_lr=C['LargeRel'].fillna(False).astype(bool).map({True: 'Large relationship', False: 'Other'}))
    t = lrc.groupby('_lr').agg(Customers=('IorB', 'size'), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100),
                               Select=('Select', lambda s: is_(s, 'Select').mean() * 100), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), MedianTotal=('TotalBalance', 'median'),
                               TopDecile=('TopDecile', 'mean'))
    t['TopDecile'] *= 100; t['% deposits'] = t['Deposits'] / max(t['Deposits'].sum(), 1e-9) * 100; t['% loans'] = t['Loans'] / t['Loans'].sum() * 100 if t['Loans'].sum() else np.nan
    display(styled(t.rename(columns={'Business': '% Business', 'Foreign': '% Foreign', 'Select': '% Select', 'MedianTotal': 'Median total balance', 'TopDecile': '% in top 10% by balance'}),
                   int_cols=['Customers'], money_cols=['Deposits', 'Loans', 'Median total balance'], pct_cols=['% Business', '% Foreign', '% Select', '% in top 10% by balance', '% deposits', '% loans']))
    print(f'Agreement between the flag and being in the top 10% by total balance: {(lrc["_lr"].eq("Large relationship") == lrc["TopDecile"]).mean():.1%}')
elif LARGE_GRAIN == 'Account':
    t = A.groupby(A['LargeRel'].fillna(False).astype(bool)).agg(Accounts=('OpenBalance', 'size'), Balance=('OpenBalance', 'sum'), Median=('_open_bal_pos', 'median'))
    display(styled(t, int_cols=['Accounts'], money_cols=['Balance', 'Median']))
else:
    print('Large Relationship not available in the file')

**Observations §11** *(fill in after the run)*

- **Products:** … DD, … CD and … LN products; top 3 products per domain carry … % of the open balance; … legacy products still hold … .
- **Select products:** … % of open DD balance and … % of CD balance are Select products; … % of Select products sit with customers whose segment is not Select.
- **Account life:** median life DD … years, CD …, LN …; … % of closed DDAs were charged off.
- **Dormancy:** … % of open accounts are dormant; … customers (… of deposits) have every open account dormant.
- **Channel:** branch … books … % of foreign accounts; … house codes hold … % of accounts; … accounts (… deposits) are still assigned to retired / inactive officers.

# 12. History — the bank year by year

The file holds every account ever opened with its open and close dates, so the bank can be rebuilt at every year end: how many accounts were open, how many customers owned at least one, how many households had an active member. **There is no balance history** (one current balance per account), so every view here counts entities, not money.

**Select in the time views is read from the account, not from the customer.** Whether a customer is Select is known today, not in the past; whether an *account* is a Select product is fixed when it opens. So every year-by-year and survival view below splits by the **product** flag (`AccountSelect`): open Select accounts per year, their closure rates and their survival.

**Rules.** An account is alive at date *t* if it opened on or before *t* and was not closed by *t*. Customers and households are alive when they own at least one live account. Accounts closed without a close date are left out (§4.2), and attributes such as IorB, ForD, segment or household composition are today's values. **Caveat:** if the core system purges old closed accounts, early years are understated — a rising curve can be partly an artefact of what the file keeps.

In [ ]:
hist_a = acct[~acct['EndUnknown'] & acct['Opened'].notna()].copy()
y_first = int(hist_a['Opened'].dt.year.min()) if len(hist_a) else SNAPSHOT.year
YEARS = list(range(max(HISTORY_START, y_first), SNAPSHOT.year + 1))
ENDS = [pd.Timestamp(y, 12, 31) if y < SNAPSHOT.year else SNAPSHOT for y in YEARS]
print(f'Accounts usable for history: {len(hist_a):,} of {len(acct):,} (excluded: {acct["EndUnknown"].sum():,} closed without a date, {acct["Opened"].isna().sum():,} without an open date)')
print(f'Earliest account opening in the file: {hist_a["Opened"].min().date() if len(hist_a) else "-"} · years shown: {YEARS[0]}–{YEARS[-1]} ({SNAPSHOT.year} = to {SNAPSHOT.date()})')

rows = []
for y, t in zip(YEARS, ENDS):
    aa = hist_a[alive_at(hist_a['Opened'], hist_a['EndDate'], t)]
    r = {'year': y, 'Open accounts': len(aa)}
    for f in FAM: r[f] = int(aa['Domain'].eq(f).sum())
    owners = pd.Index(aa['OwnerCIF'].dropna().unique())
    oc = cif.loc[owners]
    r['Active customers'] = len(owners)
    for g_ in GROUP2_ORDER: r[g_] = int(is_(oc['Group2'], g_).sum())
    r['Customers with an open Select account'] = aa.loc[aa['AccountSelect'].eq('Select'), 'OwnerCIF'].nunique()
    for f in FAM: r[f'Select {f}'] = int((aa['Domain'].eq(f) & aa['AccountSelect'].eq('Select')).sum())
    r['Open Select accounts'] = int(aa['AccountSelect'].eq('Select').sum())
    hids = pd.Index(oc['HouseholdID'].unique())
    r['Active households'] = len(hids)
    hc = hh.loc[hids]
    for comp_ in ORDER['Composition']: r[f'HH · {comp_}'] = int(is_(hc['Composition'], comp_).sum())
    r['Real households'] = int(is_(hc['HHType'], 'Real household').sum())
    rows.append(r)
stock = pd.DataFrame(rows).set_index('year')

# ---- Flows ----
opened = pd.crosstab(acct['OpenYear'], acct['Domain']).reindex(index=YEARS, columns=FAM).fillna(0)
closed = pd.crosstab(hist_a['CloseYear'], hist_a['Domain']).reindex(index=YEARS, columns=FAM).fillna(0)
owners_ever = cif[cif['OwnedEver'].gt(0)]
acq_c = pd.crosstab(owners_ever['FirstOwnedOpened'].dt.year, owners_ever['IorB']).reindex(index=YEARS, columns=ORDER['IorB']).fillna(0)
lost_m = cif['Status'].eq('Former customer') & cif['ChurnDate'].notna()
lost_c = pd.crosstab(cif.loc[lost_m, 'ChurnDate'].dt.year, cif.loc[lost_m, 'IorB']).reindex(index=YEARS, columns=ORDER['IorB']).fillna(0)
hh_first_own = owners_ever.groupby('HouseholdID')['FirstOwnedOpened'].min()
acq_h = hh_first_own.dt.year.value_counts().reindex(YEARS).fillna(0)
lost_h = hh.loc[hh['Status'].eq('Former') & hh['EndDate'].notna(), 'EndDate'].dt.year.value_counts().reindex(YEARS).fillna(0)
flows = pd.DataFrame({'Accounts opened': opened.sum(axis=1), 'Accounts closed': closed.sum(axis=1), 'Customers acquired': acq_c.sum(axis=1), 'Customers lost': lost_c.sum(axis=1),
                      'Households acquired': acq_h, 'Households lost': lost_h})
flows['Net customers'] = flows['Customers acquired'] - flows['Customers lost']
year_tbl = stock[['Open accounts', 'Open Select accounts', 'Active customers', 'Active households', 'Real households', 'Customers with an open Select account']].join(flows)
display(styled(year_tbl, int_cols=list(year_tbl.columns)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
stacked_bars(axes[0], stock[FAM], FAM, COLOR['Domain']); style(axes[0], title='Open accounts at year end, by domain', ylabel='accounts'); axes[0].legend(fontsize=8)
stacked_bars(axes[1], stock[GROUP2_ORDER], GROUP2_ORDER, pal4); style(axes[1], title='Active customers at year end, by IorB × ForD', ylabel='customers'); axes[1].legend(fontsize=7.5)
hcols = [f'HH · {c_}' for c_ in ORDER['Composition']]
stacked_bars(axes[2], stock[hcols].rename(columns=lambda c: c[5:]), ORDER['Composition'], COLOR['Composition']); style(axes[2], title='Active households at year end, by composition', ylabel='households'); axes[2].legend(fontsize=8)
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'The bank year by year — stock', f'Year end (last point = snapshot {SNAPSHOT.date()}) · today\'s attributes', name='12_stock')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
ax = axes[0]
stacked_bars(ax, opened, FAM, COLOR['Domain'])
bottom = np.zeros(len(closed))
for f in FAM:
    ax.bar(closed.index, -closed[f].values, bottom=-bottom, color=COLOR['Domain'][f], alpha=0.45, edgecolor=INK['surface'], linewidth=0.4); bottom += closed[f].values
ax.axhline(0, color=INK['secondary'], lw=0.8); ax.plot(opened.index, opened.sum(axis=1) - closed.sum(axis=1), color=INK['primary'], lw=1.5, marker='o', ms=3, label='net')
style(ax, title='Accounts opened (up) and closed (down, faded)', ylabel='accounts'); ax.legend(fontsize=7.5)
for ax, (acq, lost, ttl, cols_, pal_) in zip(axes[1:], [(acq_c, lost_c, 'Customers acquired (up) and lost (down)', ORDER['IorB'], COLOR['IorB']),
                                                        (acq_h.to_frame('Households'), lost_h.to_frame('Households'), 'Households acquired (up) and lost (down)', ['Households'], {'Households': BLUE})]):
    stacked_bars(ax, acq, cols_, pal_)
    bottom = np.zeros(len(lost))
    for c_ in cols_:
        ax.bar(lost.index, -lost[c_].values, bottom=-bottom, color=RED if len(cols_) == 1 else pal_[c_], alpha=0.55, edgecolor=INK['surface'], linewidth=0.4); bottom += lost[c_].values
    ax.axhline(0, color=INK['secondary'], lw=0.8); ax.plot(acq.index, acq.sum(axis=1) - lost.sum(axis=1), color=INK['primary'], lw=1.5, marker='o', ms=3, label='net')
    style(ax, title=ttl); ax.legend(fontsize=7.5)
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'The bank year by year — flows', 'Acquired = first account owned · lost = last account (any role) closed · the snapshot year is partial', name='12_flows')

fig, axes = plt.subplots(1, 2, figsize=(16, 3.5))
stacked_bars(axes[0], stock[[f'Select {f}' for f in FAM]].rename(columns=lambda c: c[7:]), FAM, COLOR['Domain'])
style(axes[0], title='Open Select accounts at year end, by domain', ylabel='accounts'); axes[0].legend(fontsize=8)
ax = axes[1]
for f in FAM:
    ax.plot(stock.index, stock[f'Select {f}'] / stock[f].replace(0, np.nan) * 100, color=COLOR['Domain'][f], lw=2, marker='o', ms=3.5, label=f'% of open {f} that are Select')
ax.plot(stock.index, stock['Customers with an open Select account'] / stock['Active customers'].replace(0, np.nan) * 100, color=GREEN, lw=1.5, ls='--', label='% of active customers with an open Select account')
ax.plot(stock.index, stock['Real households'] / stock['Active households'].replace(0, np.nan) * 100, color=GRAY, lw=1.5, ls=':', label='% of active households that are real groups')
ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(PCT0); ax.legend(fontsize=7.5); style(ax, title='Mix over time')
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'Select products over time', 'Select = product flag of the account (known at opening) · household type = today\'s grouping', name='12_select_time')

## 12.1 Annual attrition rates — accounts, customers, households

For each year *Y*: entities that ended during *Y* ÷ entities alive on 1 January *Y* (the snapshot year is annualized). A **flow** measure, comparable across years.

- **Accounts**: closure rate per domain. CDs and loans close by design (maturity, payoff), so their rate is not churn.
- **Customers**: every CIF that ever owned an account; it ends when nothing remains open in any role (`Former customer`). An ex-owner who is still a signer or joint owner elsewhere is still inside the relationship.
- **Households**: every household with an owner ever; it ends when no member has anything open.

In [ ]:
CH_YEARS = [y for y in YEARS if y >= SNAPSHOT.year - 14]
cb = cif[cif['OwnedEver'].gt(0) & ~(cif['Status'].eq('Former customer') & cif['ChurnDate'].isna())].copy()
cb['End'] = cb['ChurnDate'].where(cb['Status'].eq('Former customer'))
hb_ = hh[hh['Owners'].gt(0) & ~(hh['Status'].eq('Former') & hh['EndDate'].isna())].copy()
hb_['End'] = hb_['EndDate'].where(hb_['Status'].eq('Former'))
rates_c = annual_rates(cb['RelationshipStart'], cb['End'], CH_YEARS, SNAPSHOT)
rates_h = annual_rates(hb_['RelationshipStart'], hb_['End'], CH_YEARS, SNAPSHOT)
rates_a = {f: annual_rates(hist_a.loc[hist_a['Domain'].eq(f), 'Opened'], hist_a.loc[hist_a['Domain'].eq(f), 'EndDate'], CH_YEARS, SNAPSHOT) for f in FAM}
rt = pd.DataFrame({'Customer attrition %': rates_c['annual rate %'], 'Customers alive 1 Jan': rates_c['alive 1 Jan'], 'Customers lost': rates_c['ended in year'],
                   'Household attrition %': rates_h['annual rate %'], 'Households lost': rates_h['ended in year'],
                   **{f'{f} closure %': rates_a[f]['annual rate %'] for f in FAM}})
display(styled(rt, pct_cols=[c for c in rt.columns if c.endswith('%')], int_cols=['Customers alive 1 Jan', 'Customers lost', 'Households lost']))
print(f'{SNAPSHOT.year} is partial (to {SNAPSHOT.date()}): its rates are annualized.')

fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
for f in FAM:
    r_ = rates_a[f]; r_ = r_[r_['alive 1 Jan'] >= 30]
    axes[0].plot(r_.index, r_['annual rate %'], color=COLOR['Domain'][f], lw=2, marker='o', ms=3.5, label=f)
style(axes[0], title='Accounts · annual closure rate by domain'); axes[0].legend(fontsize=7.5)
for ax, dim in zip(axes[1:3], ['IorB', 'ForD']):
    order, pal = group_spec([dim])
    for g_ in order:
        r_ = annual_rates(cb.loc[is_(cb[dim], g_), 'RelationshipStart'], cb.loc[is_(cb[dim], g_), 'End'], CH_YEARS, SNAPSHOT); r_ = r_[r_['alive 1 Jan'] >= 30]
        ax.plot(r_.index, r_['annual rate %'], color=pal[g_], lw=2, marker='o', ms=3.5, label=g_)
    ax.plot(rates_c.index, rates_c['annual rate %'], color=INK['muted'], lw=1.3, ls='--', label='All customers')
    style(ax, title=f'Customers · annual attrition by {dim}'); ax.legend(fontsize=7.5)
for g_ in ORDER['HHType']:
    r_ = annual_rates(hb_.loc[is_(hb_['HHType'], g_), 'RelationshipStart'], hb_.loc[is_(hb_['HHType'], g_), 'End'], CH_YEARS, SNAPSHOT); r_ = r_[r_['alive 1 Jan'] >= 30]
    axes[3].plot(r_.index, r_['annual rate %'], color=COLOR['HHType'][g_], lw=2, marker='o', ms=3.5, label=g_)
axes[3].plot(rates_c.index, rates_c['annual rate %'], color=RED, lw=1.3, ls=':', label='Customers (for reference)')
style(axes[3], title='Households · annual attrition by type'); axes[3].legend(fontsize=7.5)
for ax in axes: ax.set_ylim(0, None); ax.yaxis.set_major_formatter(PCT0); ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'Annual attrition rates', f'Ended in the year ÷ alive on 1 January · {SNAPSHOT.year} annualized · points with < 30 alive not drawn', name='12_rates')

rates8 = pd.DataFrame({g_: annual_rates(cb.loc[is_(cb['Group2'], g_), 'RelationshipStart'], cb.loc[is_(cb['Group2'], g_), 'End'], CH_YEARS, SNAPSHOT)['annual rate %'] for g_ in GROUP2_ORDER}).T
if rates8.size:
    fig, ax = plt.subplots(figsize=(1.0 * len(CH_YEARS) + 3, 0.6 + 0.45 * len(GROUP2_ORDER)))
    heat(ax, rates8, fmt='.1f', cmap=CMAP_RED)
    finish(fig, 'Customer attrition rate (%) — IorB × ForD', f'Row = segment, column = year · {SNAPSHOT.year} annualized', name='12_rates_heatmap')

# Account-level Select: closure rate of Select vs. Regular products, per domain
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
sel_rate_rows = []
for ax, f in zip(axes, FAM):
    for sv, ls in [('Select', '-'), ('Regular', '--')]:
        m_ = hist_a['Domain'].eq(f) & hist_a['AccountSelect'].eq(sv)
        r_ = annual_rates(hist_a.loc[m_, 'Opened'], hist_a.loc[m_, 'EndDate'], CH_YEARS, SNAPSHOT); r_ = r_[r_['alive 1 Jan'] >= 30]
        if len(r_): ax.plot(r_.index, r_['annual rate %'], color=COLOR['AccountSelect'][sv], lw=2, ls=ls, marker='o', ms=3.5, label=f'{sv} products')
        sel_rate_rows += [{'domain': f, 'product': sv, 'year': y_, 'closure %': v_} for y_, v_ in r_['annual rate %'].items()]
    ax.set_ylim(0, None); ax.yaxis.set_major_formatter(PCT0); ax.tick_params(axis='x', labelrotation=45); style(ax, title=f'{FAM_LABEL[f]} · annual closure rate'); ax.legend(fontsize=7.5)
finish(fig, 'Select vs. Regular products — annual closure rate', 'Account-level Select flag (fixed when the account opens) · points with < 30 open accounts on 1 Jan not drawn', name='12_rates_select')
if sel_rate_rows:
    display(pd.DataFrame(sel_rate_rows).pivot_table(index=['domain', 'product'], columns='year', values='closure %').style.format(fmt_1, na_rep=''))

## 12.2 Vintages — of the customers acquired each year, how many are still here?

Cohort = year the relationship started. Retention after *k* years = share of the cohort not yet lost *k* years after its start (cells are blank until the cohort is old enough). The right-hand chart is the same for households.

In [ ]:
KS = [1, 2, 3, 5, 10, 15]
def retention(d, start_col, end_col):
    out = {}
    s, e = d[start_col], d[end_col]
    for k in KS:
        tk = s + pd.DateOffset(years=k)
        elig = tk <= SNAPSHOT
        alive = (e.isna() | (e > tk))
        out[f'after {k}y'] = (alive & elig).groupby(d['_coh']).sum() / elig.groupby(d['_coh']).sum().replace(0, np.nan) * 100
    t = pd.DataFrame(out); t.insert(0, 'cohort size', d.groupby('_coh').size())
    return t
COH_YEARS = [y for y in YEARS if y >= SNAPSHOT.year - 24]
ret_c = retention(cb.assign(_coh=cb['RelationshipStart'].dt.year), 'RelationshipStart', 'End').reindex(COH_YEARS)
ret_h = retention(hb_.assign(_coh=hb_['RelationshipStart'].dt.year), 'RelationshipStart', 'End').reindex(COH_YEARS)
fig, axes = plt.subplots(1, 2, figsize=(16, 0.8 + 0.3 * len(COH_YEARS)))
for ax, t, ttl in zip(axes, [ret_c, ret_h], ['Customers', 'Households']):
    heat(ax, t.drop(columns='cohort size'), fmt='.0f', cmap=CMAP_BLUE, vmax=100, suffix='%', size=7.8)
    ax.set_yticklabels([f'{int(y)}  (n={int(n):,})' if not pd.isna(n) else f'{int(y)}' for y, n in zip(t.index, t['cohort size'])], rotation=0)
    style(ax, title=f'{ttl} · % still here k years after the start')
finish(fig, 'Retention by vintage', 'Rows = year the relationship started · blank = cohort not old enough yet', name='12_vintages')
print('Customers: retention by vintage'); display(styled(ret_c, int_cols=['cohort size'], pct_cols=[c for c in ret_c.columns if c.startswith('after')]))

## 12.3 Survival curves — how long relationships last

Kaplan–Meier over everyone who ever owned an account: current customers count as "still here" up to the snapshot (censored), so the curve is not biased toward short relationships the way a median over the lost base alone is.

In [ ]:
def km_block(d, dims, label_col=None):
    rows_, curves = [], []
    order, pal = group_spec(dims) if label_col is None else (ORDER[label_col], COLOR[label_col])
    lab = make_label(d, dims) if label_col is None else d[label_col]
    for g_ in order:
        sg = d[lab.eq(g_)]
        if len(sg) < 30: continue
        x, s = kaplan_meier(sg['LifetimeYears'], sg['_event'])
        curves.append((g_, x, s, pal[g_], len(sg)))
        rows_.append({'split': ' × '.join(dims), 'segment': g_, 'n': len(sg), 'lost': int(sg['_event'].sum()), 'still here after 1y %': km_at(x, s, 1), 'after 3y %': km_at(x, s, 3),
                      'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10), 'median life (y)': km_median(x, s)})
    return rows_, curves
cbk = cb.assign(_event=cb['Status'].eq('Former customer').astype(int))
hbk = hb_.assign(_event=hb_['Status'].eq('Former').astype(int))
km_tbl_rows = []
fig, axes = plt.subplots(1, 5, figsize=(22, 3.9))
akm = hist_a.assign(LifetimeYears=hist_a['LifeYears'], _event=(~hist_a['IsOpen']).astype(int))
for ax, (d_, dims, lc, who) in zip(axes, [(cbk, ['IorB'], None, 'Customers'), (cbk, ['ForD'], None, 'Customers'), (cbk, ['IorB', 'ForD'], None, 'Customers'),
                                          (akm, ['AccountSelect'], 'AccountSelect', 'Accounts'), (hbk, ['HHType'], 'HHType', 'Households')]):
    rows_, curves = km_block(d_, dims, lc)
    for r in rows_: r['level'] = who
    km_tbl_rows += rows_
    for g_, x, s, colr, n_ in curves: ax.step(x, s * 100, where='post', color=colr, lw=2, label=f'{g_} (n={n_:,})')
    ax.axhline(50, color=INK['grid'], lw=1); ax.set_xlim(0, 30); ax.set_ylim(0, 101); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=f'{who} · by {" × ".join(dims)}', xlabel='years since start', ylabel='% still here'); ax.legend(fontsize=7.2, loc='lower left')
finish(fig, 'Survival curves', 'Kaplan–Meier · customers, accounts and households still here are censored at the snapshot · Select = product flag of the account', name='12_km')
for f in FAM:
    for sv in ORDER['AccountSelect']:
        sg = akm[akm['Domain'].eq(f) & akm['AccountSelect'].eq(sv)]
        if len(sg) < 30: continue
        x, s = kaplan_meier(sg['LifetimeYears'], sg['_event'])
        km_tbl_rows.append({'level': 'Accounts', 'split': 'Domain × Select product', 'segment': f'{f}{SEP}{sv}', 'n': len(sg), 'lost': int(sg['_event'].sum()),
                            'still here after 1y %': km_at(x, s, 1), 'after 3y %': km_at(x, s, 3), 'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10), 'median life (y)': km_median(x, s)})
for g_ in GROUP2_ORDER:
    sg = cbk[is_(cbk['Group2'], g_)]
    if len(sg) < 30: continue
    x, s = kaplan_meier(sg['LifetimeYears'], sg['_event'])
    km_tbl_rows.append({'level': 'Customers', 'split': 'IorB × ForD', 'segment': g_, 'n': len(sg), 'lost': int(sg['_event'].sum()), 'still here after 1y %': km_at(x, s, 1),
                        'after 3y %': km_at(x, s, 3), 'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10), 'median life (y)': km_median(x, s)})
for comp_ in ORDER['Composition']:
    sg = hbk[is_(hbk['Composition'], comp_)]
    if len(sg) < 30: continue
    x, s = kaplan_meier(sg['LifetimeYears'], sg['_event'])
    km_tbl_rows.append({'level': 'Households', 'split': 'Composition', 'segment': comp_, 'n': len(sg), 'lost': int(sg['_event'].sum()), 'still here after 1y %': km_at(x, s, 1),
                        'after 3y %': km_at(x, s, 3), 'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10), 'median life (y)': km_median(x, s)})
km_tbl = pd.DataFrame(km_tbl_rows).set_index(['level', 'split', 'segment'])
display(styled(km_tbl, int_cols=['n', 'lost'], pct_cols=[c for c in km_tbl.columns if c.endswith('%')], one_dec=['median life (y)']))

## 12.4 Are recent customers staying longer? CIF creation year vs. churn year, in bands

Every customer that ever owned an account, grouped by the **year its CIF was created** (in bands of `COHORT_BAND_YEARS`; a blank creation date falls back to the relationship start) and by the **year it churned** (the last of its accounts closed; still-here customers have no churn year). Three readings of the same question:

1. **Survival by creation band (Kaplan–Meier).** Each curve is one band of creation years: % of the band still a customer *k* years after the CIF was created. If the recent bands sit **above** the older ones in the first years, recent customers are staying longer; **below**, they leave sooner. Still-here customers are censored, so young bands are not penalized for being young.
2. **Creation band × churn band.** Where in time each band's losses happened (row = creation band, column = churn band, last column = still here). Cells show the count and the color the share of the row.
3. **% lost within 1, 2, 3, 5 and 10 years.** One line per horizon across creation bands, read from the survival curves. A point is drawn only when the band has had at least that many years of history, so the comparison is fair.

**Caveat.** If the core purged CIFs that closed long ago, old bands only keep the customers who stayed (survivorship bias): their curves look better than they were. Read the comparison between recent bands first.

In [ ]:
COHORT_BAND_YEARS = 5                                     # width of the creation-year and churn-year bands
def year_bands(years, start=HISTORY_START, width=COHORT_BAND_YEARS, last=SNAPSHOT.year):
    edges = list(range(start, last + 1, width))
    labels = [f'< {start}'] + [f'{e}–{str(min(e + width - 1, last))[-2:]}' if min(e + width - 1, last) != e else f'{e}' for e in edges]
    out = pd.cut(years, [-np.inf] + edges + [np.inf], right=False, labels=labels).astype(object)
    return out, labels
cr = cb.copy()
cr['Created'] = cr['CIFCreated'].fillna(cr['RelationshipStart'])
cr['CreatedYear'] = cr['Created'].dt.year
cr['ChurnYear'] = cr['End'].dt.year
cr['T'] = (cr['End'].fillna(SNAPSHOT) - cr['Created']).dt.days / 365.25
cr['_event'] = cr['End'].notna().astype(int)
cr = cr[cr['CreatedYear'].notna() & cr['T'].ge(0)]
cr['CreatedBand'], BANDS = year_bands(cr['CreatedYear'])
cr['ChurnBand'] = year_bands(cr['ChurnYear'])[0].where(cr['End'].notna(), 'Still a customer')
BANDS = [b for b in BANDS if b in set(cr['CreatedBand'])]
band_col = dict(zip(BANDS, [CMAP_BLUE(x) for x in np.linspace(0.3, 1.0, len(BANDS))]))
print(f'Customers in the view: {len(cr):,} (creation date from the CIF for {cb["CIFCreated"].notna().mean():.0%}, relationship start for the rest) · churned {cr["_event"].sum():,}')

HORIZONS = [1, 2, 3, 5, 10]
fig, axes = plt.subplots(1, 3, figsize=(20, 4.6), gridspec_kw={'width_ratios': [1.15, 1.25, 1]})
rows = []
for b in BANDS:
    sg = cr[cr['CreatedBand'].eq(b)]
    if len(sg) < 30: continue
    x, s = kaplan_meier(sg['T'], sg['_event'])
    axes[0].step(x, s * 100, where='post', color=band_col[b], lw=2.2, label=f'{b} (n={len(sg):,})')
    r = {'created': b, 'customers': len(sg), 'churned': int(sg['_event'].sum()), 'still here': int((1 - sg['_event']).sum()),
         'median life (y)': km_median(x, s), 'max years observed': sg['T'].max()}
    for k in HORIZONS: r[f'lost ≤ {k}y %'] = 100 - km_at(x, s, k) if sg['T'].max() >= k else np.nan
    rows.append(r)
axes[0].axhline(50, color=INK['grid'], lw=1); axes[0].set_xlim(0, 25); axes[0].set_ylim(0, 101); axes[0].yaxis.set_major_formatter(PCT0)
style(axes[0], title='Survival by CIF creation band', xlabel='years since the CIF was created', ylabel='% still a customer'); axes[0].legend(fontsize=7.5, loc='upper right', title='created', title_fontsize=8)

churn_cols = [b for b in year_bands(pd.Series([HISTORY_START]))[1] if b in set(cr['ChurnBand'])] + ['Still a customer']
mat = pd.crosstab(cr['CreatedBand'], cr['ChurnBand']).reindex(index=BANDS, columns=churn_cols).fillna(0)
share = mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0) * 100
annot = mat.astype(int).map(lambda v: '' if v == 0 else f'{v:,}') + share.map(lambda v: '' if not v else f'\n{v:.0f}%')
heat(axes[1], share.where(mat > 0), annot=annot.where(mat > 0, ''), cmap=CMAP_RED, vmax=max(float(share[churn_cols[:-1]].max().max()) if len(churn_cols) > 1 else 1, 1), size=7.5)
axes[1].tick_params(axis='x', labelrotation=30)
style(axes[1], title='Created (rows) × churned (columns) · color = % of the row', xlabel='churn year band', ylabel='CIF creation band')

lost = pd.DataFrame(rows).set_index('created') if rows else pd.DataFrame()
if len(lost):
    hz_col = dict(zip(HORIZONS, [CMAP_RED(x) for x in np.linspace(0.35, 1.0, len(HORIZONS))]))
    xs = np.arange(len(lost))
    for k in HORIZONS:
        v = lost[f'lost ≤ {k}y %']
        axes[2].plot(xs, v, color=hz_col[k], lw=2, marker='o', ms=4, label=f'within {k} year{"s" if k > 1 else ""}')
    axes[2].set_xticks(xs); axes[2].set_xticklabels(lost.index, rotation=30); axes[2].set_ylim(0, 100); axes[2].yaxis.set_major_formatter(PCT0)
    style(axes[2], title='% of each creation band lost within k years', xlabel='CIF creation band'); axes[2].legend(fontsize=7.5)
finish(fig, 'Are recent customers staying longer?', 'Kaplan–Meier by CIF creation band · churn = last account closed · still-here customers censored at the snapshot', name='12_cohort_bands')
if len(lost):
    display(styled(lost, int_cols=['customers', 'churned', 'still here'], one_dec=['median life (y)', 'max years observed'], pct_cols=[f'lost ≤ {k}y %' for k in HORIZONS],
                   gradient_cols=[f'lost ≤ {k}y %' for k in HORIZONS], cmap=CMAP_RED))
print('Created band × churn band (counts):'); display(mat.astype(int))

# The same survival curves, inside each IorB × ForD cell
fig, axes = plt.subplots(1, len(order4), figsize=(5 * len(order4), 3.8), sharey=True)
for ax, g_ in zip(np.atleast_1d(axes), order4):
    d_ = cr[is_(cr['Group2'], g_)]
    for b in BANDS:
        sg = d_[d_['CreatedBand'].eq(b)]
        if len(sg) < 30: continue
        x, s = kaplan_meier(sg['T'], sg['_event'])
        ax.step(x, s * 100, where='post', color=band_col[b], lw=2, label=f'{b} (n={len(sg):,})')
    ax.axhline(50, color=INK['grid'], lw=1); ax.set_xlim(0, 25); ax.set_ylim(0, 101); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=g_, xlabel='years since CIF creation', ylabel='% still a customer' if g_ == order4[0] else ''); ax.legend(fontsize=7, loc='lower left')
finish(fig, 'Survival by CIF creation band, inside each IorB × ForD cell', 'Darker = more recent creation band · bands with < 30 customers in the cell not drawn', name='12_cohort_bands_segments')

**Observations §12** *(fill in after the run)*

- **Stock:** open accounts went from … (…) to … (…); active customers peaked at … in …; active households ….
- **Flows:** the record acquisition year was … (… customers); losses exceeded acquisitions in ….
- **Rates:** customer attrition … % a year recently (individuals …, businesses …; domestic …, foreign …); household attrition … % — lower than customer attrition by … points, because ….
- **Vintages:** of the customers acquired in …, … % are still here after 5 years; the weakest vintages are …; real households retain better / worse than single CIFs.
- **Survival:** median relationship life … years for customers, … for households; Select products live … vs. Regular ….
- **Recent vs. older customers (§12.4):** of the CIFs created in …, … % were lost within 3 years vs. … % for those created in …; recent customers stay longer / leave sooner, mainly in ….

# 13. Attrition — the customers and households that left

§6–§11 describe the relationships the bank **has**; this section describes the ones it **lost**, at both levels. The file does not need a separate status column for this: a customer is lost when nothing it was linked to remains open.

| Term | Rule |
|---|---|
| **Base** | Every CIF that ever owned an account: active customers + connected ex-owners (still a signer / joint owner somewhere) + former customers. Households: every household with an owner ever. |
| **Churned CIF** | `Former customer`: owned at least one account and has nothing open in any role. |
| **Churn date** | Close date of the last of its accounts (any role) to close. Former customers with an account closed **without** a date have an uncertain churn date (counted in §13.1). |
| **Attrition share** | Churned ÷ base: the share of everyone ever onboarded in a segment that is gone today (a *stock* measure; the annual *flow* is §12.1). |
| **Exit profile / type** | What the lost customer ever owned, and how it ended: `Involuntary (charge-off)` (an account charged off in the last 12 months of the relationship), `Loan run-off` (only loans — paid off or matured), `CD run-off` (only CDs — matured, not renewed), `Deposit relationship lost` (the regrettable part). |
| **Select** | Here always the **product** definition: the customer owned at least one Select account (former customers: at any time). Whether a customer was Select in the past is not known; whether its accounts were Select products is. |
| **Age at exit** | Age when the relationship ended (decision-maker age for businesses), so a customer who left at 40 is not counted as 50 today. |

## 13.1 The lost base — size and data checks

In [ ]:
base = cif[cif['OwnedEver'].gt(0)].copy()
if SELECT_BASIS != 'product':     # the segment is today's label; a former customer's past Select status is only known through the products it owned
    base['Select'] = base['SelectProduct']; base['Group3'] = make_label(base, ['IorB', 'ForD', 'Select'])
base['Outcome'] = np.where(base['Status'].eq('Former customer'), 'Churned', np.where(base['Status'].eq('Active customer'), 'Active', 'Connected ex-owner'))
OUT_ORDER = ['Active', 'Connected ex-owner', 'Churned']; OUT_COLOR = {'Active': BLUE, 'Connected ex-owner': BLUE_T, 'Churned': RED}
ch = base[base['Outcome'].eq('Churned')].copy()
st_tbl = pd.crosstab(base['Group3'], base['Outcome']).reindex(index=GROUP3_ORDER, columns=OUT_ORDER).fillna(0).astype(int)
st_tbl.loc['All ever-owners'] = st_tbl.sum()
st_tbl['Attrition share %'] = st_tbl['Churned'] / st_tbl[OUT_ORDER].sum(axis=1).replace(0, np.nan) * 100
display(styled(st_tbl, int_cols=OUT_ORDER, pct_cols=['Attrition share %'], gradient_cols=['Attrition share %'], cmap=CMAP_RED))

rows = []
def add_chk(name, mask, note=''):
    mask = pd.Series(mask, index=ch.index).fillna(False).astype(bool)
    rows.append({'check (churned CIFs)': name, 'CIFs': int(mask.sum()), '% of churned': mask.mean() * 100 if len(ch) else np.nan, 'note': note})
res_bal = acct[~acct['IsOpen'] & acct['Balance'].fillna(0).ne(0)].groupby('OwnerCIF')['Balance'].sum()
add_chk('Residual balance on closed accounts', ch.index.isin(res_bal.index), f'total {money(res_bal.reindex(ch.index).sum())} — closure not completed?')
add_chk('Churn date unknown (every account closed without a date)', ch['ChurnDate'].isna(), 'left out of timing and tenure views')
add_chk('Churn date uncertain (some account closed without a date)', ch['ChurnDateUncertain'] & ch['ChurnDate'].notna(), 'latest known close date used')
add_chk('Churn date before relationship start', ch['ChurnDate'] < ch['RelationshipStart'], 'lifetime set to missing')
add_chk('Any owned account charged off (ever)', ch['ChargedOffEver'].gt(0), 'context only')
add_chk('Charge-off in the last 12 months before leaving', ch['ChargedOffAtExit'], 'involuntary exit — read separately')
add_chk('IorB or ForD missing', ch['IorB'].isna() | ch['ForD'].isna())
display(styled(pd.DataFrame(rows).set_index('check (churned CIFs)'), int_cols=['CIFs'], pct_cols=['% of churned']))
sel_a, sel_c = is_(base.loc[base['Outcome'].eq('Active'), 'SelectSegment'], 'Select').mean() * 100, is_(ch['SelectSegment'], 'Select').mean() * 100
print(f'Lost base: {len(ch):,} churned CIFs of {len(base):,} ever-owners → attrition share {len(ch) / max(len(base), 1):.1%} · connected ex-owners {base["Outcome"].eq("Connected ex-owner").sum():,}')
print(f'Select (segment) among active customers {sel_a:.1f}% vs churned {sel_c:.1f}%.' + ('  ⚠ The segment may be reset when a customer leaves — read segment-based Select as "Select at the snapshot".' if sel_c < sel_a / 4 else '')
      + f' Product-based Select is read from the accounts the customer owned, so it does not have this problem (SELECT_BASIS = "{SELECT_BASIS}").')
hh_base = hh[hh['Owners'].gt(0)]
print(f'Households: {hh_base["Status"].eq("Former").sum():,} former of {len(hh_base):,} with an owner ever → household attrition share {hh_base["Status"].eq("Former").mean():.1%}')

## 13.2 Which segments lose the most — every granularity, both levels

Attrition share = churned ÷ base, for the seven CIF granularities and the household dimensions. *Over-representation* compares the mix of the lost base with the mix of today's base: a positive number means the segment is a larger share of the leavers than of the customers who stay.

In [ ]:
def attrition_table(d, dims, churn_col='Outcome', churn_val='Churned', active_val='Active'):
    g = d.groupby(dims, dropna=False)
    t = pd.DataFrame({'Base': g.size(), 'Active': g[churn_col].apply(lambda s: int(s.eq(active_val).sum())), 'Churned': g[churn_col].apply(lambda s: int(s.eq(churn_val).sum()))})
    t['Attrition share %'] = t['Churned'] / t['Base'].replace(0, np.nan) * 100
    t['% of churned base'] = t['Churned'] / max(t['Churned'].sum(), 1) * 100
    t['% of active base'] = t['Active'] / max(t['Active'].sum(), 1) * 100
    t['Over-representation (pp)'] = t['% of churned base'] - t['% of active base']
    lost = d[d[churn_col].eq(churn_val)]
    t['Median lifetime of churned (y)'] = lost.groupby(dims)['LifetimeYears'].median()
    t['% churned in first year'] = lost.groupby(dims)['LifetimeYears'].apply(lambda s: (s < 1).mean() * 100)
    return t.reindex(_index_for(dims)).dropna(subset=['Base'])
ATTR_PCT = ['Attrition share %', '% of churned base', '% of active base', '% churned in first year']
attr_tables = {}
for dims in GRANULARITIES:
    display(Markdown(f'#### Customers · {" × ".join(dims)}'))
    t = attrition_table(base, dims); attr_tables[' × '.join(dims)] = t
    display(styled(t, pct_cols=ATTR_PCT, int_cols=['Base', 'Active', 'Churned'], one_dec=['Over-representation (pp)', 'Median lifetime of churned (y)'], gradient_cols=['Attrition share %'], cmap=CMAP_RED))
hhb = hh_base.assign(Outcome=np.where(hh_base['Status'].eq('Former'), 'Churned', np.where(hh_base['Status'].eq('Active'), 'Active', 'Connected')))
for dims in [['HHType'], ['Composition'], ['HHForD'], ['SizeBand'], ['HHType', 'Composition']]:
    display(Markdown(f'#### Households · {" × ".join(dims)}'))
    t = attrition_table(hhb, dims); attr_tables['HH ' + ' × '.join(dims)] = t
    display(styled(t, pct_cols=ATTR_PCT, int_cols=['Base', 'Active', 'Churned'], one_dec=['Over-representation (pp)', 'Median lifetime of churned (y)'], gradient_cols=['Attrition share %'], cmap=CMAP_RED))

overall = len(ch) / max(len(base), 1) * 100
fig, axes = plt.subplots(1, 4, figsize=(19, 3.6), gridspec_kw={'width_ratios': [1, 1, 1, 1.6]})
for ax, dim in zip(axes[:3], ['IorB', 'ForD', 'Select']):
    t = attr_tables[dim]; order, pal = group_spec([dim])
    hbar(ax, list(t.index), t['Attrition share %'].values, [pal[g_] for g_ in t.index], fmt=fmt_pct)
    ax.axvline(overall, color=RED, lw=1, ls=':'); style(ax, title=f'By {dim}'); ax.xaxis.set_major_formatter(PCT0)
t8 = attr_tables['IorB × ForD × Select']
hbar(axes[3], [SEP.join(ix) for ix in t8.index], t8['Attrition share %'].values, [pal4[SEP.join(ix[:2])] for ix in t8.index], fmt=fmt_pct, hatches=['' if ix[2] == 'Select' else '///' for ix in t8.index])
axes[3].axvline(overall, color=RED, lw=1, ls=':'); style(axes[3], title='IorB × ForD × Select (hatched = Non-Select)'); axes[3].xaxis.set_major_formatter(PCT0)
finish(fig, 'Attrition share by segment', f'Churned ÷ everyone who ever owned an account · red dotted line = all ({overall:.1f}%)', name='13_attrition_share')
fig, ax = plt.subplots(figsize=(11, 0.9 + 0.5 * len(t8)))
y = np.arange(len(t8)); hgt = 0.38
ax.barh(y - hgt / 2, t8['% of active base'].values, height=hgt, color=BLUE, label='Active base')
ax.barh(y + hgt / 2, t8['% of churned base'].values, height=hgt, color=RED, label='Churned base')
for yi, (a_, c_) in enumerate(zip(t8['% of active base'].values, t8['% of churned base'].values)):
    ax.text(a_ + 0.3, yi - hgt / 2, fmt_pct(a_), va='center', fontsize=7.8, color=INK['secondary']); ax.text(c_ + 0.3, yi + hgt / 2, fmt_pct(c_), va='center', fontsize=7.8, color=INK['secondary'])
ax.set_yticks(y); ax.set_yticklabels([SEP.join(ix) for ix in t8.index]); ax.invert_yaxis(); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid']); ax.xaxis.set_major_formatter(PCT0)
ax.legend(loc='lower right'); style(ax, xlabel='% of the base')
finish(fig, 'Who the leavers are vs. who stays', 'Each color sums to 100% across the eight segments', name='13_mix')

## 13.3 Customer churn vs. household churn

A customer who closes everything while the rest of the household stays has not taken the relationship away — the family or the group is still here. Only this file can separate the two: of the CIFs lost inside **real** households, how many left a household that is still active?

In [ ]:
ch['HHStatusNow'] = ch['HouseholdID'].map(hh['Status'])
in_real = is_(ch['HHType'], 'Real household')
rows = []
for g_ in ['All'] + order4:
    d_ = ch if g_ == 'All' else ch[is_(ch['Group2'], g_)]
    r_ = d_[is_(d_['HHType'], 'Real household')]
    rows.append({'segment': g_, 'Churned CIFs': len(d_), '% in a real household': is_(d_['HHType'], 'Real household').mean() * 100 if len(d_) else np.nan,
                 'Churned in real households': len(r_), '% whose household is still active': r_['HHStatusNow'].eq('Active').mean() * 100 if len(r_) else np.nan,
                 '% whose household is still connected': r_['HHStatusNow'].eq('Connected only').mean() * 100 if len(r_) else np.nan,
                 '% whose household is gone too': r_['HHStatusNow'].isin(['Former', 'Former connected']).mean() * 100 if len(r_) else np.nan})
cvh = pd.DataFrame(rows).set_index('segment')
display(styled(cvh, int_cols=['Churned CIFs', 'Churned in real households'], pct_cols=[c for c in cvh.columns if c.startswith('%')]))
fig, axes = plt.subplots(1, 2, figsize=(15, 3.4))
t2_ = attr_tables['IorB × ForD']
lvl = pd.DataFrame({'Customer level': [t2_['Attrition share %'].get(tuple(g_.split(SEP)), np.nan) for g_ in order4],
                    'Household level (household of the CIF)': [hh_base.loc[is_(hh_base['LeadIorB'], g_.split(SEP)[0]) & is_(hh_base['LeadForD'], g_.split(SEP)[1]), 'Status'].eq('Former').mean() * 100
                                                               for g_ in order4]}, index=order4)
grouped_bars(axes[0], lvl, list(lvl.columns), {'Customer level': RED, 'Household level (household of the CIF)': RED_T}, fmt=lambda v: f'{v:.0f}%')
style(axes[0], title='Attrition share: customers vs. households (lead member\'s IorB × ForD)', pct_y=True); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
if len(cvh):
    ct = cvh.loc[order4, ['% whose household is still active', '% whose household is still connected', '% whose household is gone too']].fillna(0)
    ct.columns = ['Household still active', 'Household still connected', 'Household gone too']
    stacked_100(axes[1], ct, list(ct.columns), {'Household still active': BLUE, 'Household still connected': BLUE_T, 'Household gone too': RED})
    style(axes[1], title='Customers lost inside real households: what happened to the household'); axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3)
finish(fig, 'Losing a customer is not always losing the relationship', name='13_cif_vs_hh')

## 13.4 What they had and how it ended

Read from the accounts each lost customer owned. A `LN only` exit is usually a loan paid off; a `CD only` exit a CD that matured and was not renewed — neither is the same problem as a lost DDA relationship. The last product to close says which relationship ended last.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 0.9 + 0.5 * len(GROUP3_ORDER)))
ex = pd.crosstab(ch['Group3'], ch['ExitProfile'], normalize='index').reindex(index=GROUP3_ORDER, columns=EXIT_ORDER).fillna(0) * 100
stacked_100(axes[0], ex, EXIT_ORDER, EXIT_COLOR); style(axes[0], title='Exit profile — what they ever owned'); axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
et = pd.crosstab(ch['Group3'], ch['ExitType'], normalize='index').reindex(index=GROUP3_ORDER, columns=EXIT_TYPES).fillna(0) * 100
stacked_100(axes[1], et, EXIT_TYPES, EXIT_TYPE_COLOR); style(axes[1], title='Exit type — how it ended'); axes[1].tick_params(labelleft=False)
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
finish(fig, 'Which relationship ended', 'LN only ≈ loan paid off / matured · CD only ≈ CD matured, not renewed · charge-off = involuntary', name='13_exit')
ex_life = ch.groupby('ExitProfile').agg(CIFs=('IorB', 'size'), Life=('LifetimeYears', 'median'), Under1=('LifetimeYears', lambda s: (s < 1).mean() * 100),
                                        Accounts=('OwnedEver', 'mean'), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100),
                                        ChargedOff=('ChargedOffEver', lambda s: s.gt(0).mean() * 100)).reindex(EXIT_ORDER).dropna(subset=['CIFs'])
display(styled(ex_life.rename(columns={'Life': 'Median lifetime (y)', 'Under1': '% left in < 1 y', 'Accounts': 'Avg accounts owned ever', 'Business': '% Business', 'Foreign': '% Foreign',
                                       'ChargedOff': '% with a charge-off'}), int_cols=['CIFs'], one_dec=['Median lifetime (y)', 'Avg accounts owned ever'],
               pct_cols=['% left in < 1 y', '% Business', '% Foreign', '% with a charge-off']))
lc = pd.crosstab(ch['Group2'], ch['LastClosedDomain'].fillna('unknown'), normalize='index').reindex(index=order4, columns=FAM + ['unknown']).fillna(0) * 100
print('Last product to close, by IorB × ForD (% of churned):'); display(lc.style.format(fmt_pct))
print(f'Exit types (all churned): ' + ' · '.join(f'{k} {v:.0%}' for k, v in ch['ExitType'].value_counts(normalize=True).reindex(EXIT_TYPES).dropna().items()))

## 13.5 How long the leavers stayed

Lifetime bands of the customers who left next to the relationship age of the ones who stay. A large `<1` band is an onboarding problem; a large `11–20` band is a legacy book running off.

In [ ]:
band_colors = dict(zip(YEAR_BANDS, [CMAP_BLUE(x) for x in np.linspace(0.18, 1.0, len(YEAR_BANDS))]))
fig, axes = plt.subplots(1, 2, figsize=(16, 0.9 + 0.5 * len(GROUP3_ORDER)))
for ax, (d_, ttl) in zip(axes, [(ch, 'Churned · lifetime when they left'), (C, 'Active · relationship age today')]):
    ct = pd.crosstab(d_['Group3'], d_['TenureBand'], normalize='index').reindex(index=GROUP3_ORDER, columns=YEAR_BANDS).fillna(0) * 100
    stacked_100(ax, ct, YEAR_BANDS, band_colors); style(ax, title=ttl)
axes[1].tick_params(labelleft=False); axes[1].legend(loc='upper center', bbox_to_anchor=(0.0, -0.12), ncol=6, title='Years', title_fontsize=8.5)
finish(fig, 'Tenure of the customers who left vs. the ones who stay', 'Rows sum to 100%', name='13_lifetime_bands')
ten = ch.groupby('Group3').agg(Churned=('IorB', 'size'), Median=('LifetimeYears', 'median'), P25=('LifetimeYears', lambda s: s.quantile(.25)), P75=('LifetimeYears', lambda s: s.quantile(.75)),
                               Under1=('LifetimeYears', lambda s: (s < 1).mean() * 100), Over10=('LifetimeYears', lambda s: (s >= 10).mean() * 100), Since=('YearsSinceChurn', 'median')).reindex(GROUP3_ORDER)
ten.loc['All churned'] = [len(ch), ch['LifetimeYears'].median(), ch['LifetimeYears'].quantile(.25), ch['LifetimeYears'].quantile(.75), (ch['LifetimeYears'] < 1).mean() * 100,
                          (ch['LifetimeYears'] >= 10).mean() * 100, ch['YearsSinceChurn'].median()]
display(styled(ten.rename(columns={'Median': 'Median lifetime (y)', 'P25': 'P25 (y)', 'P75': 'P75 (y)', 'Under1': '% left in < 1 y', 'Over10': '% stayed 10+ y', 'Since': 'Median years since leaving'}),
               int_cols=['Churned'], one_dec=['Median lifetime (y)', 'P25 (y)', 'P75 (y)', 'Median years since leaving'], pct_cols=['% left in < 1 y', '% stayed 10+ y'], gradient_cols=['% left in < 1 y'], cmap=CMAP_RED))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, dims in zip(axes, [['IorB'], ['ForD'], ['Select']]):
    order, pal = group_spec(dims)
    kde_by(ax, ch.assign(_grp=make_label(ch, dims)), 'LifetimeYears', '_grp', order, pal, clip=(0, 35)); style(ax, title=f'Churned · lifetime by {" × ".join(dims)}', xlabel='years')
finish(fig, 'How long the leavers stayed', 'Churned customers only · dashed = median', name='13_lifetime_kde')

## 13.6 Who leaves vs. who stays — attrition share by characteristic

Every characteristic the file offers, measured on the base (everyone who ever owned an account). Ages are **at the end of the relationship** (decision-maker age for businesses). Bars above the dotted line lose more than average.

In [ ]:
first_own = acct.dropna(subset=['OwnerCIF']).sort_values('Opened').drop_duplicates('OwnerCIF').set_index('OwnerCIF')
base['MainBranch'] = lump(first_own['Branch'].reindex(base.index).fillna('(missing)'), 8)
base['Families'] = base['FamiliesEver'].astype(str) + ' famil' + np.where(base['FamiliesEver'].eq(1), 'y', 'ies')
base['Cohort'] = pd.cut(base['CohortYear'], [0, 2000, 2005, 2010, 2015, 2020, 2100], right=False, labels=['< 2000', '2000–04', '2005–09', '2010–14', '2015–19', '2020+']).astype(object)
base['HHSize'] = count_band(base['HHMembers'])
behind = oth.assign(_b=is_(oth['OwnerIorB'], 'Business')).groupby('CIF')['_b'].any().reindex(base.index).fillna(False).astype(bool)
base['Behind a business'] = np.where(is_(base['IorB'], 'Individual'), np.where(behind, 'Also behind a business', 'Not behind a business'), 'n/a (business)')
base['KP known'] = np.where(is_(base['IorB'], 'Business'), np.where(base['KP_CIF'].notna(), 'Person behind it identified', 'Nobody identified'), 'n/a (individual)')
base['Churned'] = base['Outcome'].eq('Churned').astype(int)
def rate_by(d, col, order=None, min_n=30):
    t = d.groupby(d[col].fillna('(missing)'))['Churned'].agg(['mean', 'size'])
    t = t[t['size'] >= min_n]
    if order is not None: t = t.reindex([o for o in order if o in t.index] + [o for o in t.index if o not in order])
    return t.assign(rate=t['mean'] * 100)
panels = [('DecisionAgeAtEndBand', 'Individuals · age at end', AGE_BANDS, is_(base['IorB'], 'Individual')),
          ('DecisionAgeAtEndBand', 'Business · key-person age at end', AGE_BANDS, is_(base['IorB'], 'Business')),
          ('BusinessAgeAtEndBand', 'Business · years since incorporation, at end', YEAR_BANDS, is_(base['IorB'], 'Business')),
          ('OwnedEverBand', 'Accounts owned ever', COUNT_BANDS, None),
          ('Families', 'DD / CD / LN families ever held', ['1 family', '2 families', '3 families'], None),
          ('ExitProfile', 'Products ever owned (profile)', EXIT_ORDER, None),
          ('RoleProfile', 'Owner only vs. also in other roles', ORDER['RoleProfile'], None),
          ('Behind a business', 'Individuals: also behind a business account?', ['Also behind a business', 'Not behind a business'], is_(base['IorB'], 'Individual')),
          ('KP known', 'Business: person behind it identified?', ['Person behind it identified', 'Nobody identified'], is_(base['IorB'], 'Business')),
          ('HHType', 'Household type', ORDER['HHType'], None),
          ('HHSize', 'CIFs in the household', COUNT_BANDS, None),
          ('ForeignSplit', 'Domestic / dominant country / other foreign', ORDER['ForeignSplit'], None),
          ('Region', 'Region', REGION_ORDER, None),
          ('MainBranch', 'Branch of the first account (top 8)', None, None),
          ('Cohort', 'Vintage (relationship start)', ['< 2000', '2000–04', '2005–09', '2010–14', '2015–19', '2020+'], None),
          ('SelectProduct', 'Ever owned a Select product', ORDER['Select'], None)]
ncols = 2; nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.7 * nrows))
for ax, (col, ttl, order, mask) in zip(axes.flat, panels):
    d_ = base if mask is None else base[mask]
    t = rate_by(d_, col, order)
    if len(t) == 0: ax.axis('off'); continue
    hbar(ax, [f'{i}  (n={n:,})' for i, n in zip(t.index, t['size'])], t['rate'].values, [RED if r > overall * 1.25 else BLUE if r < overall * 0.8 else GRAY for r in t['rate']], fmt=fmt_pct)
    ax.axvline(overall, color=RED, lw=1, ls=':'); ax.xaxis.set_major_formatter(PCT0); style(ax, title=ttl)
for ax in list(axes.flat)[len(panels):]: ax.axis('off')
finish(fig, 'Attrition share by characteristic', f'Churned ÷ ever-owners · red = > 1.25× the average ({overall:.1f}%), blue = < 0.8×, gray = around it · groups with < 30 CIFs hidden', name='13_drivers')
for col, order in [('OwnedEverBand', COUNT_BANDS), ('Families', ['1 family', '2 families', '3 families']), ('HHType', ORDER['HHType']), ('RoleProfile', ORDER['RoleProfile'])]:
    pv = base.pivot_table(index=col, columns='Group2', values='Churned', aggfunc='mean').reindex(index=[o for o in order if o in set(base[col])], columns=order4) * 100
    nn = base.pivot_table(index=col, columns='Group2', values='Churned', aggfunc='size').reindex_like(pv)
    print(f'Attrition share % by {col} × IorB × ForD (cells with < 30 CIFs blanked):')
    display(pv.where(nn >= 30).style.format(fmt_pct, na_rep='').background_gradient(cmap=CMAP_RED, vmin=0, vmax=100))

## 13.7 Partial attrition inside the active base

Customers who are still here but are shrinking: they closed an owned account in the last 12 months, or they once held a DDA (the transactional anchor) and no longer do while keeping a CD or a loan.

In [ ]:
C = C.copy()
own12 = acct[~acct['IsOpen'] & acct['EndDate'].gt(SNAPSHOT - pd.DateOffset(months=12)) & acct['OwnerCIF'].isin(C.index)]
for f in FAM: C[f'Closed12_{f}'] = own12[own12['Domain'].eq(f)].groupby('OwnerCIF').size().reindex(C.index).fillna(0).gt(0)
C['Closed12_any'] = C[[f'Closed12_{f}' for f in FAM]].any(axis=1)
C['LostDDA'] = C['Had_DD'] & ~C['Has_DD']
pa = C.groupby('Group3').agg(Customers=('IorB', 'size'), Any=('Closed12_any', 'mean'), DD=('Closed12_DD', 'mean'), CD=('Closed12_CD', 'mean'), LN=('Closed12_LN', 'mean'),
                             LostDDA=('LostDDA', 'mean'), Dormant=('OnlyDormant', 'mean')).reindex(GROUP3_ORDER)
for c_ in ['Any', 'DD', 'CD', 'LN', 'LostDDA', 'Dormant']: pa[c_] *= 100
pa = pa.rename(columns={'Any': '% closed any account (12m)', 'DD': '% closed a DDA', 'CD': '% closed a CD', 'LN': '% closed a loan', 'LostDDA': '% lost their DDA, still here', 'Dormant': '% every open account dormant'})
display(styled(pa, int_cols=['Customers'], pct_cols=[c for c in pa.columns if c.startswith('%')], gradient_cols=['% closed any account (12m)', '% lost their DDA, still here'], cmap=CMAP_RED))
labels8 = GROUP3_ORDER; cols8 = [pal4[SEP.join(g_.split(SEP)[:2])] for g_ in labels8]; hatch8 = ['' if g_.endswith(SEP + 'Select') else '///' for g_ in labels8]
fig, axes = plt.subplots(1, 3, figsize=(18, 3.9))
hbar(axes[0], labels8, pa['% closed any account (12m)'].values, cols8, fmt=fmt_pct, hatches=hatch8); style(axes[0], title='% that closed an owned account in the last 12 months'); axes[0].xaxis.set_major_formatter(PCT0)
hbar(axes[1], labels8, pa['% lost their DDA, still here'].values, cols8, fmt=fmt_pct, hatches=hatch8); style(axes[1], title='% that lost their DDA but kept a CD / loan'); axes[1].xaxis.set_major_formatter(PCT0); axes[1].tick_params(labelleft=False)
ax = axes[2]
for g_ in ORDER['IorB']:
    m_ = C.loc[is_(C['IorB'], g_), 'MonthsSinceLastClose'].dropna(); m_ = m_[m_ <= 60]
    if len(m_): sns.histplot(x=m_, bins=30, stat='percent', element='step', fill=True, alpha=0.18, color=COLOR['IorB'][g_], linewidth=1.8, ax=ax, label=f'{g_} (n={len(m_):,})')
ax.axvline(12, color=RED, lw=1, ls=':'); ax.legend(fontsize=8)
style(ax, title='Months since the last owned account closed (last 5 years)', xlabel='months', ylabel='% of those customers')
finish(fig, 'Partial attrition inside active customers', 'Hatched = Non-Select · red line = 12 months', name='13_partial')

## 13.8 Recent leavers — the last 12 and 24 months

The customers whose last account closed recently are the ones still worth a call: the relationship is fresh, the officer is probably still there, and the reason can still be asked.

In [ ]:
rec = ch[ch['MonthsSinceChurn'].notna()]
r12, r24 = rec[rec['MonthsSinceChurn'] <= 12], rec[rec['MonthsSinceChurn'] <= 24]
print(f'Churned in the last 12 months: {len(r12):,} customers · last 24 months: {len(r24):,} (of {len(ch):,} churned)')
rt_ = pd.DataFrame({'Last 12m': r12['Group3'].value_counts(), 'Last 24m': r24['Group3'].value_counts()}).reindex(GROUP3_ORDER).fillna(0).astype(int)
for lab, d_ in [('12m', r12), ('24m', r24)]:
    g_ = d_.groupby('Group3')
    rt_[f'Median lifetime (y) · {lab}'] = g_['LifetimeYears'].median()
    rt_[f'Avg accounts owned · {lab}'] = g_['OwnedEver'].mean()
    rt_[f'% deposit relationship lost · {lab}'] = g_['ExitType'].apply(lambda s: s.eq('Deposit relationship lost').mean() * 100)
    rt_[f'% household still active · {lab}'] = g_['HHStatusNow'].apply(lambda s: s.eq('Active').mean() * 100)
display(styled(rt_, int_cols=['Last 12m', 'Last 24m'], one_dec=[c for c in rt_.columns if c.startswith(('Median', 'Avg'))], pct_cols=[c for c in rt_.columns if c.startswith('%')]))
if len(r24):
    mon = r24.assign(m=r24['ChurnDate'].dt.to_period('M').dt.to_timestamp()).groupby(['m', 'IorB']).size().unstack().reindex(columns=ORDER['IorB']).fillna(0)
    fig, ax = plt.subplots(figsize=(14, 3.4)); bottom = np.zeros(len(mon))
    for g_ in ORDER['IorB']:
        ax.bar(mon.index, mon[g_].values, bottom=bottom, width=20, color=COLOR['IorB'][g_], label=g_, edgecolor=INK['surface'], linewidth=0.4); bottom += mon[g_].values
    ax.legend(fontsize=8); style(ax, title='Customers whose last account closed, per month', ylabel='customers'); ax.tick_params(axis='x', labelrotation=45)
    finish(fig, 'Recent attrition — last 24 months', f'Snapshot {SNAPSHOT.date()} · the last month may be incomplete', name='13_recent')

## 13.9 Churned vs. active — are the differences real?

Mann-Whitney U for direction, **Cliff's δ** for size (A = churned, B = active; δ > 0 means leavers tend to have *more*). Lifetime of active customers is censored, so the tenure row understates how long active relationships will last — read it with the survival curves in §12.3.

In [ ]:
rows = []
measures = [('LifetimeYears', 'Relationship years (to churn / to snapshot)'), ('OwnedEver', 'Accounts owned ever'), ('FamiliesEver', 'DD / CD / LN families ever'),
            ('Ever_DD', 'DDAs ever'), ('Ever_CD', 'CDs ever'), ('Ever_LN', 'Loans ever'), ('RoleAccts', 'Accounts in other roles (ever)'), ('HHMembers', 'CIFs in the household'),
            ('DecisionAgeAtEnd', 'Age at end (decision-maker)'), ('BusinessAgeAtEnd', 'Years since incorporation at end (Business)')]
for g_ in ['All'] + order4:
    d_ = base if g_ == 'All' else base[is_(base['Group2'], g_)]
    a_, b_ = d_[d_['Outcome'].eq('Churned')], d_[d_['Outcome'].eq('Active')]
    for col, lab in measures:
        pval, dlt = cliff(a_[col], b_[col])
        if pd.isna(dlt): continue
        rows.append({'segment': g_, 'measure': lab, 'n churned': int(a_[col].notna().sum()), 'n active': int(b_[col].notna().sum()), 'median churned': a_[col].median(), 'median active': b_[col].median(),
                     'mean churned': a_[col].mean(), 'mean active': b_[col].mean(), 'p (Mann-Whitney)': pval, "Cliff's δ (churned vs active)": dlt, 'effect': effect_label(dlt)})
ctests = pd.DataFrame(rows).set_index(['segment', 'measure']) if rows else pd.DataFrame()
if len(ctests):
    display(ctests.style.format({'median churned': fmt_1, 'median active': fmt_1, 'mean churned': '{:.2f}', 'mean active': '{:.2f}', 'n churned': fmt_int, 'n active': fmt_int,
                                 'p (Mann-Whitney)': fmt_p, "Cliff's δ (churned vs active)": '{:+.2f}'}, na_rep='').background_gradient(cmap=CMAP_DIV, subset=["Cliff's δ (churned vs active)"], vmin=-1, vmax=1))
rows = []
for g_ in ORDER['IorB']:
    d_ = base[is_(base['IorB'], g_) & base['Outcome'].isin(['Churned', 'Active'])]
    for f, col in [('DD', 'Had_DD'), ('CD', 'Had_CD'), ('LN', 'Had_LN'), ('Select product', 'SelectProduct'), ('Real household', 'HHType'), ('Other roles', 'RoleAccts')]:
        x_ = d_[col].eq('Select') if col == 'SelectProduct' else (d_[col].eq('Real household') if col == 'HHType' else (d_[col].gt(0) if col == 'RoleAccts' else d_[col]))
        ct = pd.crosstab(d_['Outcome'], x_)
        pval = chi2_contingency(ct)[1] if ct.shape == (2, 2) else np.nan
        rows.append({'IorB': g_, 'characteristic (ever)': f, '% with it · churned': x_[d_['Outcome'].eq('Churned')].mean() * 100, '% with it · active': x_[d_['Outcome'].eq('Active')].mean() * 100,
                     'p (chi-square)': pval, "Cramér's V": cramers_v(d_['Outcome'], x_)})
display(pd.DataFrame(rows).set_index(['IorB', 'characteristic (ever)']).style.format({'% with it · churned': fmt_pct, '% with it · active': fmt_pct, 'p (chi-square)': fmt_p, "Cramér's V": '{:.2f}'}, na_rep=''))

**Observations §13** *(fill in after the run)*

- **Size of the lost base:** … churned customers, … % of everyone who ever owned an account; … % of households. Data checks: ….
- **Which segments churn:** IorB …; ForD …; Select …; highest cell …, lowest ….
- **Customer vs. household churn:** … % of customers lost inside real households left a household that is still active.
- **What ended:** … % deposit relationships lost, … % loan run-off, … % CD run-off, … % charge-offs.
- **Tenure:** leavers stayed a median … years; … % left within a year; … % after 10+ years.
- **Who leaves vs. who stays:** strongest cuts … (single-account, single-CIF households, no other roles …).
- **Partial attrition:** … % of active customers closed an account in the last 12 months; … % lost their DDA but kept a CD / loan.
- **Recent leavers:** … in the last 12 months, mostly ….

# 14. Relationships & tests — what moves with what

Correlations first (Spearman, so heavy tails do not dominate), then the Select uplift measured **within** segment with confidence intervals, effect sizes for the comparisons that matter (Select, real household, domestic vs. foreign), and the association between every dimension. Active customers only.

In [ ]:
corr_cols = ['Age', 'BusinessAge', 'KP_Age', 'LifetimeYears', 'OwnedOpen', 'ProductFamilies', 'RoleAcctsOpen', 'LinkedBusinesses', 'HHMembers', 'NetworkSize',
             'Deposits', 'Loans', 'DD balance', 'CD balance', 'LN balance']
fig, axes = plt.subplots(1, 2, figsize=(16, 6.2))
for ax, g_ in zip(axes, ORDER['IorB']):
    cc_ = [c_ for c_ in corr_cols if not (g_ == 'Individual' and c_ in ('BusinessAge', 'KP_Age')) and not (g_ == 'Business' and c_ == 'Age')]
    d_ = C.loc[is_(C['IorB'], g_), cc_].astype(float)
    d_ = d_.loc[:, d_.nunique() > 1]
    if d_.shape[1] < 2: no_data(ax); continue
    corr = d_.corr(method='spearman')
    sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool), k=1), annot=True, fmt='.2f', cmap=CMAP_DIV, center=0, vmin=-1, vmax=1, cbar=False, ax=ax,
                annot_kws={'size': 7.2}, linewidths=0.5, linecolor=INK['surface'])
    ax.set_title(g_); ax.tick_params(axis='x', labelrotation=45); ax.grid(False)
finish(fig, 'Spearman correlation between measures (active customers)', 'Red = negative, blue = positive. Balances include zeros (non-holders).', name='14_corr')

orderS, palS = group_spec(['Select'])
medS, loS, hiS = median_table(C, 'Deposits', 'Group2', 'Select', order4, orderS, holders=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.9), gridspec_kw={'width_ratios': [1.5, 1]})
grouped_bars(axes[0], medS, orderS, palS, fmt=money, lo=loS, hi=hiS); style(axes[0], title='Median deposits among holders · Select vs Non-Select within segment (95% CI)', money_y=True)
top_y = np.nanmax(hiS.values) if np.isfinite(np.nanmax(hiS.values)) else 1; axes[0].set_ylim(0, top_y * 1.35)
ratio = medS['Select'] / medS['Non-Select']
for i, g_ in enumerate(order4):
    if np.isfinite(ratio.get(g_, np.nan)): axes[0].text(i, top_y * 1.2, f'×{ratio[g_]:.1f}', ha='center', fontsize=9.5, fontweight='bold', color=INK['primary'])
sel_h = C.loc[is_(C['Select'], 'Select') & (C['Deposits'] > 0), 'Deposits']; non_h = C.loc[is_(C['Select'], 'Non-Select') & (C['Deposits'] > 0), 'Deposits']
agg_ratio = sel_h.median() / non_h.median() if len(sel_h) and len(non_h) else np.nan
hbar(axes[1], ['All customers (mix effect)'] + order4, [agg_ratio] + [ratio.get(g_, np.nan) for g_ in order4], [INK['muted']] + [pal4[g_] for g_ in order4], fmt=lambda v: f'×{v:.1f}')
axes[1].axvline(1, color=RED, lw=1, ls=':'); style(axes[1], title='Select ÷ Non-Select median deposits', xlabel='ratio (red line = no uplift)')
finish(fig, 'The Select uplift, measured honestly', 'The all-customer ratio is inflated by segment mix (Simpson\'s paradox); the within-segment ratios are the real effect', name='14_uplift')

## 14.1 Are the differences real? Effect sizes and association between dimensions

Mann-Whitney U tests whether one group tends to hold more than the other; **Cliff's δ** says how much (−1…+1; |δ| < 0.15 negligible, < 0.33 small, < 0.47 medium, else large). With thousands of customers almost everything is "significant", so the effect size is the number to read.

In [ ]:
rows = []
def add_test(comparison, segment, measure, a, b, holders=False):
    if holders: a = a[a > 0]; b = b[b > 0]
    pval, d = cliff(a, b)
    rows.append({'comparison': comparison, 'segment': segment, 'measure': measure + (' (holders)' if holders else ''), 'n A': int(a.notna().sum()), 'n B': int(b.notna().sum()),
                 'median A': a.median(), 'median B': b.median(), 'p (Mann-Whitney)': pval, "Cliff's δ (A vs B)": d, 'effect': effect_label(d)})
for g_ in order4:
    d_ = C[is_(C['Group2'], g_)]; s1, s0 = is_(d_['Select'], 'Select'), is_(d_['Select'], 'Non-Select')
    add_test('A = Select, B = Non-Select', g_, 'Deposits', d_.loc[s1, 'Deposits'], d_.loc[s0, 'Deposits'], holders=True)
    add_test('A = Select, B = Non-Select', g_, 'Relationship years', d_.loc[s1, 'LifetimeYears'], d_.loc[s0, 'LifetimeYears'])
    add_test('A = Select, B = Non-Select', g_, 'Open accounts owned', d_.loc[s1, 'OwnedOpen'], d_.loc[s0, 'OwnedOpen'])
    r1, r0 = is_(d_['HHType'], 'Real household'), is_(d_['HHType'], 'Single CIF')
    add_test('A = real household, B = single CIF', g_, 'Deposits', d_.loc[r1, 'Deposits'], d_.loc[r0, 'Deposits'], holders=True)
    add_test('A = real household, B = single CIF', g_, 'Product families', d_.loc[r1, 'ProductFamilies'], d_.loc[r0, 'ProductFamilies'])
for g_ in ORDER['IorB']:
    d_ = C[is_(C['IorB'], g_)]; dm_, fr_ = is_(d_['ForD'], 'Domestic'), is_(d_['ForD'], 'Foreign')
    agec = 'Age' if g_ == 'Individual' else 'KP_Age'
    add_test('A = Domestic, B = Foreign', g_, 'Deposits', d_.loc[dm_, 'Deposits'], d_.loc[fr_, 'Deposits'], holders=True)
    add_test('A = Domestic, B = Foreign', g_, LABEL[agec], d_.loc[dm_, agec], d_.loc[fr_, agec])
    add_test('A = Domestic, B = Foreign', g_, 'Relationship years', d_.loc[dm_, 'LifetimeYears'], d_.loc[fr_, 'LifetimeYears'])
tests = pd.DataFrame(rows).set_index(['comparison', 'segment', 'measure'])
def fmt_med(v): return '' if pd.isna(v) else (money(v) if abs(v) >= 1000 else f'{v:,.1f}')
display(tests.style.format({'median A': fmt_med, 'median B': fmt_med, 'p (Mann-Whitney)': fmt_p, "Cliff's δ (A vs B)": '{:+.2f}', 'n A': fmt_int, 'n B': fmt_int}, na_rep='')
        .background_gradient(cmap=CMAP_DIV, subset=["Cliff's δ (A vs B)"], vmin=-1, vmax=1))

C_ = C.assign(SegmentTier=C['Segment_Tier'].astype('Int64').astype(str))
cat_cols = [c_ for c_ in ['IorB', 'ForD', 'Select', 'SelectSegment', 'ForeignSplit', 'Region', 'HHType', 'HHComposition', 'RoleProfile', 'ProductMix', 'TenureBand', 'DecisionAgeBand',
                          'DepositTier', 'SegmentTier'] if c_ in C_.columns]
V = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a_ in cat_cols:
    for b_ in cat_cols:
        V.loc[a_, b_] = 1.0 if a_ == b_ else cramers_v(C_[a_].fillna('(missing)'), C_[b_].fillna('(missing)'))
fig, ax = plt.subplots(figsize=(10, 7))
heat(ax, V, fmt='.2f', vmax=1, size=7.8); ax.tick_params(axis='x', labelrotation=45)
finish(fig, "Association between dimensions (Cramér's V)", '0 = independent, 1 = one determines the other · active customers', name='14_cramers')

**Observations §14** *(fill in after the run)*

- **Strongest relationships:** …
- **Select uplift within segment:** ×… to ×… (the aggregate ×… is a mix effect); Cliff's δ on deposits ….
- **Real household vs. single CIF:** deposits δ …, product families δ ….
- **Domestic vs. foreign:** ….
- **Surprises:** ….

# 15. Segment profiles — who the customer and the household are, final version

The 8 customer segments (IorB × ForD × Select) and the household segments (composition × residence) on the same metrics, plus a "fingerprint" heatmap where every column is scaled 0–1 across segments, so the reader sees at a glance which segment is oldest, longest-tenured, deepest, richest, most connected.

In [ ]:
prof = profile_table(C, ['IorB', 'ForD', 'Select'])
show_profile(prof)
fp_cols = ['% CIFs', 'Median age (Ind.)', 'Median key-person age (Bus.)', 'Median tenure (y)', 'Avg open accts', 'Avg families', '% has DD', '% has CD', '% has LN',
           '% in real household', '% also in other roles', '% Deposits', 'Median deposits (holders)', '% Loans', 'Median loans (holders)']
fp = prof[[c_ for c_ in fp_cols if c_ in prof.columns]].astype(float)
scaled = (fp - fp.min()) / (fp.max() - fp.min()).replace(0, np.nan)
annot = fp.copy().astype(object)
for c_ in fp.columns: annot[c_] = fp[c_].map(money if c_ in MONEY_COLS else (fmt_pct if c_ in PCT_COLS else fmt_1))
fig, ax = plt.subplots(figsize=(17, 0.6 + 0.55 * len(fp)))
heat(ax, scaled.fillna(0), annot=annot, vmax=1, size=8)
ax.set_yticklabels([SEP.join(ix) for ix in fp.index], rotation=0); ax.tick_params(axis='x', labelrotation=30)
finish(fig, 'Customer fingerprint — IorB × ForD × Select', 'Each column scaled 0–1 across the 8 segments (darker = higher). Cells show the actual value.', name='15_fingerprint_cif')
print('\nSegment one-liners (paste into the findings):')
for ix, r in prof.iterrows():
    nm = SEP.join(ix)
    age = f'median age {r["Median age (Ind.)"]:.0f}' if ix[0] == 'Individual' else f'median business age {r["Median business age"]:.0f}y, key person {r["Median key-person age (Bus.)"]:.0f}'
    print(f'• {nm}: {r["CIFs"]:,.0f} customers ({r["% CIFs"]:.1f}%), {age}, relationship {r["Median tenure (y)"]:.1f}y, {r["Avg families"]:.2f} families, '
          f'{r["% in real household"]:.0f}% in a real household, holds {r["% Deposits"]:.1f}% of deposits (median {money(r["Median deposits (holders)"])}) and {r["% Loans"]:.1f}% of loans.')

hprof = hh_profile_table(H, ['Composition', 'HHForD'])
show_profile(hprof)
hfp_cols = ['% HHs', 'Avg members', '% real household', '% with a business', 'Median reference age', 'Median relationship (y)', 'Avg open accts', 'Avg families',
            '% has DD', '% has CD', '% has LN', '% Select', '% Deposits', 'Median deposits (holders)', '% Loans', 'Median loans (holders)']
hfp = hprof[[c_ for c_ in hfp_cols if c_ in hprof.columns]].astype(float)
hscaled = (hfp - hfp.min()) / (hfp.max() - hfp.min()).replace(0, np.nan)
hannot = hfp.copy().astype(object)
for c_ in hfp.columns: hannot[c_] = hfp[c_].map(money if c_ in MONEY_COLS else (fmt_pct if c_ in PCT_COLS else fmt_1))
fig, ax = plt.subplots(figsize=(17, 0.6 + 0.55 * len(hfp)))
heat(ax, hscaled.fillna(0), annot=hannot, vmax=1, size=8)
ax.set_yticklabels([SEP.join(ix) for ix in hfp.index], rotation=0); ax.tick_params(axis='x', labelrotation=30)
finish(fig, 'Household fingerprint — composition × residence', 'Each column scaled 0–1 across household segments. Cells show the actual value.', name='15_fingerprint_hh')

**Observations §15** *(one sentence per segment — paste from the one-liners above and add the interpretation)*

- **Individual · Domestic · Select** — …
- **Individual · Domestic · Non-Select** — …
- **Individual · Foreign · Select** — …
- **Individual · Foreign · Non-Select** — …
- **Business · Domestic · Select** — …
- **Business · Domestic · Non-Select** — …
- **Business · Foreign · Select** — …
- **Business · Foreign · Non-Select** — …
- **Households:** …

# 16. Key findings & exports

Written for someone who will not open the code. The cell below prints the headline numbers; the template after it is where the narrative goes. Then the clean tables are exported for Power BI.

In [ ]:
dep, loans = C['Deposits'], C['Loans']
ind_m, biz_m, fr_m, sel_m = is_(C['IorB'], 'Individual'), is_(C['IorB'], 'Business'), is_(C['ForD'], 'Foreign'), is_(C['Select'], 'Select')
last_full = SNAPSHOT.year - 1
def rate_at(r_, y_): return r_['annual rate %'].get(y_, np.nan) if len(r_) else np.nan
B_ = C[biz_m]
lines = [
    f'Three levels today: {act_h.sum():,} active households ({(act_h & is_(hh["HHType"], "Real household")).sum():,} real groups), {len(C):,} active customers, {acct["IsOpen"].sum():,} open accounts '
    f'(DD {A["Domain"].eq("DD").sum():,} · CD {A["Domain"].eq("CD").sum():,} · LN {A["Domain"].eq("LN").sum():,}); {cif["Status"].eq("Connected (non-owner)").sum():,} connected CIFs own nothing but hold roles.',
    f'Money: {money(dep.sum())} deposits and {money(loans.sum())} loans. Businesses are {biz_m.mean():.0%} of customers and hold {dep[biz_m].sum() / max(dep.sum(), 1):.0%} of deposits and {loans[biz_m].sum() / max(loans.sum(), 1):.0%} of loans; '
    f'foreign customers are {fr_m.mean():.0%} of customers with {dep[fr_m].sum() / max(dep.sum(), 1):.0%} of deposits / {loans[fr_m].sum() / max(loans.sum(), 1):.0%} of loans. Overdrafts (kept apart from deposits): {money(C["Overdraft"].sum())} across {C["Overdraft"].gt(0).sum():,} customers.',
    f'Typical individual: median age {C.loc[ind_m, "Age"].median():.0f}, relationship {C.loc[ind_m, "LifetimeYears"].median():.1f} y, {C.loc[ind_m, "OwnedOpen"].median():.0f} open account(s), median deposits {money(dep[ind_m & (dep > 0)].median())}.',
    f'Typical business: {C.loc[biz_m, "BusinessAge"].median():.0f} years since incorporation, key person aged {B_["KP_Age"].median():.0f} (identified for {B_["KP_CIF"].notna().mean():.0%}), '
    f'median deposits {money(dep[biz_m & (dep > 0)].median())}; {B_["Has_LN"].mean():.0%} borrow.',
    f'Relationships: {A["OtherCIFs"].gt(0).mean():.0%} of open accounts carry someone besides the owner; {A.loc[A["Domain"].eq("LN") & A["Has: Guarantor"], "Loans"].sum() / max(A.loc[A["Domain"].eq("LN"), "Loans"].sum(), 1):.0%} of the loan balance has a guarantor; '
    f'{C["RoleAcctsOpen"].gt(0).mean():.0%} of customers also sign / guarantee / co-own elsewhere.',
    f'Households: {is_(H["HHType"], "Real household").mean():.0%} of active households are real groups holding {H.loc[is_(H["HHType"], "Real household"), "Deposits"].sum() / max(H["Deposits"].sum(), 1):.0%} of deposits; '
    f'{is_(H["Composition"], "Mixed").sum():,} mix individuals and businesses; customers in a network of 2+ CIFs: {C["NetworkSize"].ge(2).mean():.0%}.',
    f'Succession: {B_["KP_Age"].ge(65).sum() / max(B_["KP_Age"].notna().sum(), 1):.0%} of businesses with a known key person depend on someone aged 65+ ({money(B_.loc[B_["KP_Age"].ge(65), "Deposits"].sum())} deposits, '
    f'{money(B_.loc[B_["KP_Age"].ge(65), "Loans"].sum())} loans); {(~B_["KP_Status"].eq("Active customer") & B_["KP_CIF"].notna()).sum():,} businesses have a key person who does not bank with us personally.',
    f'Select: {sel_m.mean():.0%} of customers by {SELECT_BASIS} ({is_(C["SelectProduct"], "Select").mean():.0%} by product, {is_(C["SelectSegment"], "Select").mean():.0%} by segment, agree {(C["SelectProduct"] == C["SelectSegment"]).mean():.0%}); '
    f'median deposits {money(dep[sel_m & (dep > 0)].median())} vs {money(dep[~sel_m & (dep > 0)].median())}.',
    f'Concentration: top 1% of customers hold {top_share(dep, .01):.0f}% of deposits; top 1% of households hold {top_share(H["Deposits"], .01):.0f}%; 80% of deposits sit with {holders_for(dep, 80):.1f}% of holders.',
    f'History: {stock["Active customers"].idxmax()} was the peak year ({stock["Active customers"].max():,} active customers); customer attrition {rate_at(rates_c, last_full):.1f}% in {last_full}, households {rate_at(rates_h, last_full):.1f}%.',
    f'Lost base: {len(ch):,} former customers = {len(ch) / max(len(base), 1):.1%} of everyone who ever owned an account; median stay {ch["LifetimeYears"].median():.1f} y; {(ch["LifetimeYears"] < 1).mean():.0%} left within a year; '
    f'exit types: ' + ', '.join(f'{k} {v:.0%}' for k, v in ch['ExitType'].value_counts(normalize=True).items()) + '.',
    f'Household vs. customer churn: of customers lost inside real households, {ch.loc[in_real, "HHStatusNow"].eq("Active").mean():.0%} left a household that is still active.',
    f'Partial attrition: {C["Closed12_any"].mean():.1%} of active customers closed an owned account in the last 12 months; {C["LostDDA"].mean():.1%} lost their DDA but kept a CD / loan; '
    f'{C["OnlyDormant"].mean():.1%} have every open account dormant.',
    f'Data quality: {(acct["OwnerSource"] != "Primary row").sum():,} accounts with a repaired owner, {acct["EndUnknown"].sum():,} closed accounts without a close date, '
    f'{cif.loc[ind_c, "BirthDate"].isna().mean():.0%} of individuals without a birth date, {acct["OfficerInactive"][acct["IsOpen"]].sum():,} open accounts with a retired / inactive officer.',
]
print('\n'.join(f'{i + 1}. {l}' for i, l in enumerate(lines)))

**Key findings** *(fill in after the run — five numbered points, each with its numbers)*

1. **Who.** …
2. **How relationships are built.** …
3. **Where the money is.** …
4. **History and attrition.** …
5. **Risks and blind spots.** …

In [ ]:
if EXPORT_CLEAN:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    acct_cols = ['AccountNumber', 'Domain', 'Subtype', 'Product', 'AccountSelect', 'StatusCode', 'StatusDesc', 'StatusGroup', 'IsOpen', 'Dormant', 'ChargedOff', 'Opened', 'EndDate', 'EndUnknown',
                 'LifeYears', 'OpenYear', 'CloseYear', 'Balance', 'OpenBalance', 'Deposits', 'Loans', 'BranchCode', 'Branch', 'OfficerCode', 'OfficerInactive', 'OwnerCIF', 'OwnerSource',
                 'IorB', 'ForD', 'Select', 'Group2', 'Group3', 'HouseholdID', 'HHType', 'Composition', 'OtherCIFs', 'Structure'] + [f'Has: {f}' for f in ROLE_FAMILIES[1:]]
    cif_cols = ['HouseholdID', 'HHType', 'HHStatus', 'HHComposition', 'HHMembers', 'IorB', 'ForD', 'Select', 'SelectProduct', 'SelectSegment', 'Group2', 'Group3', 'Segment', 'SegmentDesc',
                'Segment_Tier', 'Country', 'Region', 'ForeignSplit', 'Status', 'RoleProfile', 'MainOtherRole', 'CIFCreated', 'RelationshipStart', 'ChurnDate', 'LifetimeYears', 'TenureBand',
                'CohortYear', 'Age', 'AgeBand', 'AgeAtEnd', 'BusinessAge', 'BusinessAgeBand', 'KP_CIF', 'KP_Role', 'KP_Source', 'KP_Age', 'KP_AgeBand', 'KP_IsCustomer', 'LinkedPersons',
                'DecisionAge', 'DecisionAgeBand', 'OwnedEver', 'OwnedOpen', 'Ever_DD', 'Ever_CD', 'Ever_LN', 'Open_DD', 'Open_CD', 'Open_LN', 'ProductFamilies', 'ProductMix', 'FamiliesEver',
                'MixEver', 'DD balance', 'CD balance', 'LN balance', 'Deposits', 'Loans', 'TotalBalance', 'DepositTier', 'ValueBand', 'RoleAccts', 'RoleAcctsOpen', 'LinkedBusinesses',
                'NetworkID', 'NetworkSize', 'NetworkHouseholds', 'OnlyDormant', 'ExitProfile', 'ExitType', 'LastClosedDomain', 'MonthsSinceChurn']
    hh_cols = ['HHType', 'Status', 'Members', 'Individuals', 'Businesses', 'ActiveMembers', 'ConnectedMembers', 'FormerMembers', 'Composition', 'HHForD', 'HHSelect', 'SizeBand',
               'RelationshipStart', 'HHStartFile', 'EndDate', 'LifetimeYears', 'CohortYear', 'RefAge', 'RefAgeSource', 'RefAgeBand', 'OwnedOpen', 'OwnedEver', 'Open_DD', 'Open_CD', 'Open_LN',
               'ProductFamilies', 'ProductMix', 'Deposits', 'Loans', 'TotalBalance', 'ValueBand', 'LeadCIF', 'LeadIorB', 'LeadForD']
    acct[[c_ for c_ in acct_cols if c_ in acct.columns]].to_csv(os.path.join(EXPORT_DIR, 'accounts.csv'))
    cif[[c_ for c_ in cif_cols if c_ in cif.columns]].to_csv(os.path.join(EXPORT_DIR, 'cifs.csv'))
    hh[[c_ for c_ in hh_cols if c_ in hh.columns]].to_csv(os.path.join(EXPORT_DIR, 'households.csv'))
    year_tbl.join(rt.drop(columns=[c_ for c_ in rt.columns if c_ in year_tbl.columns]), how='left').to_csv(os.path.join(EXPORT_DIR, 'history_by_year.csv'))
    print(f'Exported to {EXPORT_DIR}/: accounts.csv ({len(acct):,}) · cifs.csv ({len(cif):,}) · households.csv ({len(hh):,}) · history_by_year.csv ({len(year_tbl):,})')
    if IN_COLAB:
        print('Download from the Colab file panel, or: from google.colab import files; files.download(f"{EXPORT_DIR}/cifs.csv")')
else:
    print('EXPORT_CLEAN = False — nothing written')

# 17. Open questions, data to request, next steps

**Questions for the data owner (answer before reading the numbers as final)**

1. **Purging.** Does the core keep every closed account forever, or are old closed accounts purged (and from when)? The early years of §12 and the survival curves depend on it.
2. **Status codes.** The 8 account status codes — confirm the mapping to Active / Dormant / Closed / Charged-off in §4.2, and whether *Dormant / Restricted* accounts should count as open customers.
3. **Close dates of open accounts.** Open CDs / loans with a future close date: is it the maturity date? Open accounts with a past close date: re-opened, or stale status?
4. **Household fallbacks.** Which group IDs are temporary or employee groups that were replaced by the CIF? Employee relationships should be excluded from attrition.
5. **Role codes.** Every code mapped to `Other` in §9.1, and whether *Beneficial owner* / *Controlling party* exists as a role for businesses (it drives the key-person rule).
6. **Select.** Which definition does the business use — the product (AccountSegment) or the customer segment? They disagree for … % of customers (§5.4).
7. **Large Relationship.** Its rule and its grain (§3.5).

**Data to request (highest value first)**

1. **Monthly balance history per account** (or 12 monthly averages) — today every year-by-year view counts entities, not money; balances are one overnight snapshot.
2. **Account families not in the extract** — savings, money market, IRAs, cards, safe-deposit, trade finance. Product depth is understated until they arrive.
3. **CIF inactivation date and reason code**, and **closure reason per account** — the churn date here is a proxy (last account closed) and voluntary vs. bank-initiated exits cannot be separated except for charge-offs.
4. **Loan detail:** maturity date, original amount, payoff flag (natural maturity vs. prepayment), risk rating, days past due.
5. **CD detail:** term, maturity, rate, renewal / rollover flag — to separate a matured-and-left CD from a renewed one.
6. **Activity:** last customer-initiated transaction, monthly transaction counts, digital enrollment — "open" by status is not "active" by behaviour.
7. **Firmographics** for businesses (industry, revenue, legal type) and **ownership percentages** for beneficial owners (replaces the key-person priority rule with the real controlling person).
8. **Profitability per account / customer** — the "best relationship" needs money made, not just money held.

**Hypotheses to test next**

- Are customers inside real households (or networks of 2+ CIFs) less likely to leave, once tenure and products are controlled for? (§13.6 says …; a survival model would confirm it.)
- Does losing the DDA predict losing the whole relationship within 12–24 months? (§13.7 + the dates in the file allow a direct test.)
- Do businesses whose key person is 65+ leave more often when the key person leaves? (§10.3 × §13.)
- Which vintages are eroding fastest, and is it a country story or a product story? (§12.2.)

**What feeds the Power BI dashboard.** `exports/accounts.csv`, `exports/cifs.csv`, `exports/households.csv` and `exports/history_by_year.csv` (§16), joined on `OwnerCIF` → `CIF` and `HouseholdID`.